# M08: Cloud Data Pipeline — Part 1 (Build)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/M08_Cloud_Data_Pipeline_Build.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View_on_GitHub-blue?logo=github)](https://github.com/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/M08_Cloud_Data_Pipeline_Build.ipynb)

**Program:** Quintrix Jr. Data Engineer Training
**Author:** Sunil Mogadati
**Module:** 08 of 13 (M00–M13)

> **This is the centerpiece of the program.** Everything before this module built skills — Python, SQL, PySpark, data modeling. This module applies them all on real cloud infrastructure. You will build a full medallion pipeline: Bronze → Silver → Gold.

**Prerequisites:** [M07 (PySpark)](M07_PySpark.ipynb), M05 (Data Modeling), M04 (Advanced SQL)

**Estimated time:** 8-12 hours across multiple sessions. This is the largest module in the program. Don't rush it.

---
## Table of Contents

**Part A — Cloud Platforms**
1. [Why Cloud?](#1-why-cloud)
2. [AWS for Data Engineers](#2-aws-for-data-engineers)
3. [GCP for Data Engineers](#3-gcp-for-data-engineers)
4. [Snowflake](#4-snowflake)

**Part B — Modern Table Formats**
5. [Medallion Architecture — The Blueprint](#5-medallion-architecture)
6. [Delta Lake Deep Dive](#6-delta-lake-deep-dive)
7. [Apache Iceberg](#7-apache-iceberg)

**Part C — Build the Pipeline**
8. [Build: Bronze Layer](#8-build-bronze-layer)
9. [Build: Silver Layer](#9-build-silver-layer)
10. [Build: Gold Layer](#10-build-gold-layer)
11. [Cost Optimization](#11-cost-optimization)

**Appendices**
- [A. AWS CLI (Command Line Interface) Cheat Sheet](#appendix-a-aws-cli-cheat-sheet)
- [B. Glossary](#appendix-b-glossary)

---
## Before We Start: The Problem This Pipeline Solves

Imagine you run a call center. You have **three systems** that don't talk to each other:

| System | What It Stores | Format | Where It Lives |
|--------|---------------|--------|----------------|
| VA Platform (RetellAI) | Call recordings, duration, disposition, customer info | Nested JSON | MongoDB |
| Order System (OLX) | What was sold, SKU, quantity, price | CSV exports | SQL Server |
| Payment Gateway (Chase) | Transaction amount, auth code, card type, approval/decline | XML responses | API (Application Programming Interface) logs |

Your VP of Sales asks a simple question:

> **"Which campaign generated the most revenue last week? And what was our conversion rate by hour of day?"**

Today, answering this takes **a senior developer 2 days** of manual SQL queries across three databases, with timezone bugs, duplicate records, and numbers that don't match between systems.

**By the end of this module, you will have built a pipeline that answers this question automatically, every day, in seconds.**

```
THE PROBLEM:                              WHAT YOU'LL BUILD:

VA Platform (JSON) -----                    calls.json ----\
Order System (CSV) ---+--> (manual SQL) --> (wrong numbers)     orders.csv -----+--> Bronze --> Silver --> Gold --> Dashboard
Payment Gateway (XML) ---                 payments.xml --/     (raw)     (clean)    (marts)

3 systems, 3 formats,                     One pipeline. Automated. Accurate.
manual effort, wrong numbers              VP gets the answer in seconds.
```

**Every section in this module builds toward this.** When we cover S3 (Simple Storage Service), it's because that's where the raw files land. When we cover Delta Lake, it's because that's how we handle the duplicates and late-arriving payments. When we cover Redshift/BigQuery/Snowflake, that's where the VP's dashboard reads from.

The tools serve the business problem. Not the other way around.

---
### A Note Before We Start: Errors Are the Curriculum

In this module, you **will** hit errors. This is not a sign that something is wrong. This IS the learning.

| Error You'll Hit | Why It Happens | What to Do |
|---|---|---|
| `AccessDenied` on S3 | IAM (Identity and Access Management) policy doesn't cover this bucket/action | Read the error. Check the policy. Fix and retry. |
| `InvalidIdentityToken` | AWS credentials expired or misconfigured | Run `aws sts get-caller-identity`. Reconfigure if needed. |
| Region mismatch | Created bucket in us-east-1, EMR (Elastic MapReduce) cluster in us-west-2 | Pick ONE region and stick with it. We use `us-east-1`. |
| `ClusterNotReady` | EMR is still provisioning (5-15 min) | Wait. Check EMR console for status. |
| Billing surprise | Left a cluster running overnight | Set auto-termination. Check Billing Dashboard daily. |
| `ModuleNotFoundError` in PySpark | Missing dependency on EMR | Add to `--packages` or bootstrap action. |

**Real data engineers spend 40% of their time debugging cloud infrastructure.** If you're fighting IAM permissions for 30 minutes, you're not failing the exercise — you're doing the job.

> **Rule for this module:** When you hit an error, read the full message before asking for help. The answer is usually in the error text. This is the single most important skill in cloud engineering.

### Quick Recap: You Already Know How to Build a Pipeline

In [M07 (PySpark)](M07_PySpark.ipynb), you built this locally:

```
calls.json --> PySpark --> clean, deduplicate, join --> write Parquet
```

That's a real pipeline. It works. But it runs on your laptop.

**In this module, we do the exact same thing — on cloud infrastructure.**

| What You Did (M07, Local) | What You'll Do (M08, Cloud) | Why Cloud? |
|---|---|---|
| Read JSON from your laptop | Read JSON from S3/GCS | Data is too big for your laptop (100GB+) |
| PySpark on local Spark | PySpark on EMR/Dataproc | Need 40 cores, not 8 |
| Write Parquet to local disk | Write Delta Lake to S3/GCS | Need ACID, time travel, MERGE |
| Query with local Spark SQL | Query with Redshift/BigQuery/Snowflake | Need dashboards for 50 analysts, not 1 developer |
| Run manually when you remember | Orchestrate with Airflow (M09) | Need it to run at 6 AM whether you're awake or not |

**The concepts are identical. The scale is different.** If you understood M07, you'll understand M08. The new part is the cloud services — not the pipeline logic.

---
## 1. Why Cloud?

This is NOT a general "what is cloud computing?" lecture. You already know what cloud is. This section answers one specific question:

**Why did data engineering move to the cloud — and what changed when it did?**

In [ ]:
# ============================================================
# Quick Check: Can you reach the cloud?
# Run ONE of these to prove your cloud CLI works.
# ============================================================

# GCP (primary):
# !gcloud config get-value project
# You should see: your project ID (e.g., 'de-training')

# AWS (if using AWS):
# !aws sts get-caller-identity
# You should see: your account ID and user ARN

# WHY: Before learning 20 cloud services, make sure
# your CLI is configured. If this fails, fix auth first
# (see Section 3.1a: GCP Setup from Scratch).

print('If you can run gcloud or aws commands, you are ready.')
print('If not, go to Section 3.1a (GCP) or install AWS CLI first.')

### The Architectural Shift: Compute-Storage Separation

In the old world (Hadoop-era, on-prem):
- Storage and compute were **coupled** — your data lived on the same machines that processed it (HDFS)
- Want more storage? Buy more machines (which come with CPU/RAM you might not need)
- Want more compute? Buy more machines (which come with storage you might not need)
- Cluster sits idle at 3 AM but you're still paying for it

In the cloud world:
- Storage and compute are **independent**
- S3/GCS = infinite storage, pay per GB/month (~$0.023/GB)
- EMR/Dataproc = spin up compute when needed, shut down when done
- Redshift/BigQuery/Snowflake = query engines that scale independently of storage

```
OLD WORLD (Hadoop)                    NEW WORLD (Cloud)

┌─────────────────────┐              ┌──────────────┐
│  Machine 1          │              │   S3 / GCS   │ ← Storage (always on,
│  ├── HDFS data      │              │   (data lake) │    pennies per GB)
│  └── Spark executor │              └──────┬───────┘
├─────────────────────┤                     │
│  Machine 2          │              ┌──────┴───────┐
│  ├── HDFS data      │              │  EMR / Dataproc  │ ← Compute (on-demand,
│  └── Spark executor │              │  (PySpark jobs)   │    spin up/down)
├─────────────────────┤              └──────┬───────┘
│  Machine 3          │                     │
│  ├── HDFS data      │              ┌──────┴───────┐
│  └── Spark executor │              │ Redshift / BQ │ ← Query engine
└─────────────────────┘              │ / Snowflake   │    (independent scaling)
                                     └───────────────┘
Always on. Always paying.            Pay for what you use.
Scale = buy more machines.           Scale = adjust each layer.
```

**Why this matters for your career:** Every DE job posting in 2026 assumes cloud. The question isn't "do you know cloud?" — it's "which cloud services have you worked with?"


### Managed Services vs Self-Hosted

| | Self-Hosted (you manage) | Managed Service (cloud manages) |
|---|---|---|
| **Spark** | Install on your own servers, configure YARN, manage upgrades | EMR / Dataproc — click a button, cluster is ready |
| **Airflow** | Install, configure, manage the scheduler/webserver/DB | MWAA / Cloud Composer — fully managed |
| **PostgreSQL** | Install, patch, backup, failover, monitoring | RDS (Relational Database Service) / Cloud SQL — automated backups, failover, patching |
| **Kafka** | Zookeeper, broker tuning, partition rebalancing | MSK / Confluent Cloud — managed clusters |

**The trade-off:**
- **Managed** = less control, less ops work, higher per-unit cost, faster to start
- **Self-hosted** = full control, more ops work, lower per-unit cost (at scale), slower to start

**For junior DEs:** Start with managed services. Learn the concepts. You can always go deeper into self-hosting later. No one expects a junior DE to manage a Kafka cluster.


### The Cloud Landscape for Data Engineers

| Platform | Market Position | DE Strength | Typical Customer |
|----------|----------------|-------------|-----------------|
| **AWS** | Largest market share (~32%). Most services. Most job postings. | Broadest service catalog. S3 is the de facto data lake standard. EMR, Redshift, Glue, Athena. | Enterprises, startups — everywhere |
| **GCP** | ~11% market share. BigQuery is a standout. | BigQuery (serverless warehouse) is best-in-class. Dataproc. Strong AI/ML integration. | Analytics-heavy orgs, Google shops, AI companies |
| **Azure** | ~23% market share. Strong enterprise. | Synapse Analytics, Data Factory, ADLS. Deep Microsoft/Office integration. | Microsoft shops, enterprises with existing Azure |
| **Snowflake** | Cloud-agnostic data platform. Runs ON AWS/GCP/Azure. | Data warehouse + data lake + data sharing. Snowpark (PySpark-like). | Multi-cloud orgs, analytics teams, data sharing use cases |

> **In this module:** We lead with **AWS** (most job postings, broadest services), then show the **GCP equivalent** for each service, then cover **Snowflake** as a cross-cloud platform. You'll build the same pipeline on at least two platforms.


### What We're Building

By the end of this module, you will have built this:

```
                        YOUR PIPELINE
                        =============

  calls.json ──┐
  orders.csv ──┼──→  S3/GCS (Bronze)  ──→  PySpark on EMR/Dataproc (Silver)  ──→  Redshift/BigQuery/Snowflake (Gold)
  payments.xml ┘     raw, append-only       deduplicate, clean, MERGE              marts: daily campaign,
                     Delta Lake tables      Delta Lake tables                       hourly volume, conversion
                                            UTC→EST, flatten JSON,
                                            SCD Type 2, dead letter queue

  Same data. Same logic. Built on real cloud infrastructure.
```

**Before we build, we need to understand the tools.** Sections 2–4 cover the cloud services. Sections 5–7 cover the table formats. Sections 8–10 are the build.


---
## 2. AWS for Data Engineers

AWS has 200+ services. A data engineer uses about 15 of them regularly. This section covers those 15, organized by what they do — not by AWS's marketing categories.

> **How this section works:** For each service, we explain what it is, why a DE cares, and then demo it live. Keep your AWS console open — you'll be running commands alongside.


### 2.1 IAM — Identity and Access Management

Before you touch any AWS service, you need to understand IAM. Every "Access Denied" error you'll ever see traces back to IAM.

**What IAM answers:** "Who can do what to which resources?"

#### Key Concepts

| Term | What It Is | Analogy |
|------|-----------|---------|
| **User** | A person or application with permanent credentials (access key + secret key) | An employee with a badge |
| **Group** | A collection of users (e.g., "data-engineers") | A department — everyone in it gets the same permissions |
| **Role** | A set of permissions that can be **assumed** temporarily (no permanent credentials) | A hat you put on — while wearing it, you have those permissions |
| **Policy** | A JSON document that defines permissions (allow/deny actions on resources) | The rules printed on the badge |
| **ARN** | Amazon Resource Name — the unique ID for any AWS resource | A mailing address — uniquely identifies any resource in AWS |
| **Principal** | The entity (user, role, service) making the request | Whoever is at the door asking for access |

#### How They Fit Together

```
                    ┌─────────────────────┐
                    │       POLICY        │
                    │  "Allow s3:GetObject │
                    │   on bucket XYZ"    │
                    └──────────┬──────────┘
                               │ attached to
                    ┌──────────┴──────────┐
                    │        ROLE          │
                    │  "data-engineer-role" │
                    └──────────┬──────────┘
                               │ assumed by
         ┌─────────────────────┼─────────────────────┐
         │                     │                     │
   ┌─────┴──────┐       ┌─────┴─────┐         ┌─────┴─────┐
   │   GROUP    │       │    EMR    │         │   LAMBDA  │
   │ "data-     │       │ (cluster) │         │ (function)│
   │  engineers"│       └───────────┘         └───────────┘
   └─────┬──────┘
         │ contains
   ┌─────┴─────┐
   │   USER    │
   │  (you)    │
   └───────────┘
```

**Why Roles > Users for DE:**
- EMR clusters, Lambda functions, Glue jobs — they all need permissions
- You don't give them a username/password. You give them a **role**
- The role says "this EMR cluster can read from S3 bucket X and write to Redshift"
- **Never hardcode access keys in your pipeline code.** Use roles.


#### Policy Document — The Language of Permissions

Every IAM policy is a JSON document with this structure:

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowReadCallData",
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                "arn:aws:s3:::de-training-calls",
                "arn:aws:s3:::de-training-calls/*"
            ]
        }
    ]
}
```

| Field | What It Means |
|-------|--------------|
| `Effect` | `Allow` or `Deny` |
| `Action` | What operations (e.g., `s3:GetObject`, `s3:PutObject`, `redshift:GetClusterCredentials`) |
| `Resource` | Which specific resources (ARNs). `*` = everything (dangerous!) |
| `Condition` | Optional — restrict by IP, time, MFA, etc. |

**Two types of policies:**

| Type | Attached To | Example |
|------|------------|---------|
| **Identity Policy** | User, Group, or Role | "This role can read S3 and write to Redshift" |
| **Resource Policy** | The resource itself (S3 bucket, SQS (Simple Queue Service) queue) | "This S3 bucket allows access from account 123456" |

> **Interview tip:** "What's the difference between an identity policy and a resource policy?" — This comes up. Identity policy = what the person/service CAN do. Resource policy = who the resource ALLOWS in.


In [ ]:
# ============================================================
# Demo: Verify your AWS identity
# ============================================================
# Run this to confirm your AWS credentials are configured
# You should see your account ID, user ARN, and user ID

!aws sts get-caller-identity

In [ ]:
# ============================================================
# Demo: List existing IAM roles
# ============================================================
# See what roles already exist in your account
# Look for roles with "EMR", "Glue", "Lambda" in the name --
# AWS often creates service-linked roles automatically

!aws iam list-roles --query 'Roles[*].[RoleName,CreateDate]' --output table | head -30

In [ ]:
# ============================================================
# Demo: Create an IAM policy for our pipeline
# ============================================================
# This policy allows reading from our Bronze S3 bucket
# and writing to our Silver S3 bucket

import json

policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "ReadBronze",
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                "arn:aws:s3:::de-training-bronze",
                "arn:aws:s3:::de-training-bronze/*"
            ]
        },
        {
            "Sid": "WriteSilver",
            "Effect": "Allow",
            "Action": [
                "s3:PutObject",
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                "arn:aws:s3:::de-training-silver",
                "arn:aws:s3:::de-training-silver/*"
            ]
        }
    ]
}

# Save the policy to a file (we'll use it when creating a role)
with open('/tmp/de_pipeline_policy.json', 'w') as f:
    json.dump(policy_document, f, indent=2)

print("Policy document:")
print(json.dumps(policy_document, indent=2))

In [ ]:
# ============================================================
# Demo: Create the IAM policy in AWS
# ============================================================

!aws iam create-policy \
    --policy-name DE-Pipeline-S3-Access \
    --policy-document file:///tmp/de_pipeline_policy.json \
    --description "Allow pipeline to read Bronze and write Silver S3 buckets" 

# **You should see:** A JSON response with the policy ARN (like `arn:aws:iam::123456:policy/DE-Pipeline-S3-Access`). Save this ARN — you'll use it when creating the role. If you see `EntityAlreadyExists`, the policy already exists (fine — skip this step).

In [ ]:
# ============================================================
# Demo: Create a role for our EMR cluster
# ============================================================
# A role needs TWO things:
#   1. Trust policy -- WHO can assume this role (EMR service)
#   2. Permission policy -- WHAT the role can do (S3 access)

import json

# Trust policy: "Allow the EMR service to assume this role"
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "elasticmapreduce.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

with open('/tmp/emr_trust_policy.json', 'w') as f:
    json.dump(trust_policy, f, indent=2)

print("Trust policy (who can assume this role):")
print(json.dumps(trust_policy, indent=2))
print()
print("This says: 'The EMR service can put on this hat.'")
print("The hat (permission policy) says: 'Whoever wears me can read/write S3.'" )

In [ ]:
# ============================================================
# Demo: Create the role and attach the policy
# ============================================================

# Step 1: Create the role with the trust policy
!aws iam create-role \
    --role-name DE-Pipeline-EMR-Role \
    --assume-role-policy-document file:///tmp/emr_trust_policy.json \
    --description "Role for EMR to run DE pipeline jobs"

# Step 2: Attach the permission policy to the role
# (Replace ACCOUNT_ID with your actual account ID)
# !aws iam attach-role-policy \
#     --role-name DE-Pipeline-EMR-Role \
#     --policy-arn arn:aws:iam::ACCOUNT_ID:policy/DE-Pipeline-S3-Access

#### IAM Best Practices for Data Engineers

| Practice | Why | Example |
|----------|-----|---------|
| **Use roles, not access keys** | Keys can leak. Roles are temporary. | EMR cluster assumes a role, not your personal keys |
| **Least privilege** | Only grant what's needed. Start small, add as needed. | Pipeline reads Bronze → only grant `s3:GetObject` on that bucket |
| **Separate roles per service** | If EMR is compromised, it shouldn't access everything | EMR role ≠ Lambda role ≠ Glue role |
| **Never use `*` in production** | `"Resource": "*"` = access to everything in your account | Always scope to specific bucket ARNs |
| **Use AWS managed policies for learning** | Pre-built policies for common use cases | `AmazonS3ReadOnlyAccess`, `AmazonEMRFullAccessPolicy_v2` |
| **Rotate credentials** | If you must use access keys, rotate them regularly | `aws iam create-access-key` → update config → delete old key |

> **Common IAM errors you'll see:**
> - `AccessDenied` — your user/role doesn't have the right policy
> - `UnauthorizedAccess` — the trust policy doesn't allow your entity to assume the role
> - `InvalidIdentityToken` — token expired (re-authenticate)


#### AWS Console Walkthrough: IAM

> **Open the IAM Console:** [console.aws.amazon.com/iam](https://console.aws.amazon.com/iam/)

**Create a Policy (Console):**

1. Left sidebar → **Policies** → **Create policy**
2. Click the **JSON** tab (not the visual editor — DEs write JSON policies)
3. Paste the policy JSON from the demo above (ReadBronze + WriteSilver)
4. Click **Next**
5. Policy name: `DE-Pipeline-S3-Access`
6. Description: "Allow pipeline to read Bronze and write Silver S3 buckets"
7. Click **Create policy**

**Create a Role (Console):**

1. Left sidebar → **Roles** → **Create role**
2. Trusted entity type: **AWS service**
3. Use case: **EMR** (select from dropdown)
   - This sets the trust policy automatically — "EMR service can assume this role"
4. Click **Next**
5. Search for `DE-Pipeline-S3-Access` → check the box
6. Also attach `AmazonEMRFullAccessPolicy_v2` (AWS managed policy for EMR basics)
7. Click **Next**
8. Role name: `DE-Pipeline-EMR-Role`
9. Click **Create role**

**Verify:**

1. Go to **Roles** → search `DE-Pipeline-EMR-Role`
2. Click it → **Permissions** tab → you should see both policies
3. Click **Trust relationships** tab → you should see `elasticmapreduce.amazonaws.com`

> **Teaching point:** Notice how the console auto-generated the trust policy when you selected "EMR" as the use case. The CLI version requires you to write it manually (the `emr_trust_policy.json` we created earlier). Same result, different paths.

### 2.2 S3 — Simple Storage Service (The Data Lake Foundation)

S3 is where your data lives. In a cloud DE pipeline, S3 (or GCS) is the **data lake** — the central store for raw, processed, and archived data.

#### Key Concepts

| Concept | What It Means | Common Mistake |
|---------|--------------|----------------|
| **Bucket** | A container for objects (like a top-level folder). Globally unique name. | Bucket names are global — `my-data` is probably taken |
| **Key** | The full path to an object: `bronze/calls/date=2026-03-19/calls.json` | |
| **Prefix** | A path-like structure: `bronze/calls/date=2026-03-19/` | S3 has NO folders. Prefixes LOOK like folders but aren't |
| **Object** | A file + metadata (up to 5TB per object) | |
| **Region** | Physical location (us-east-1, us-west-2, etc.) | Accessing S3 across regions = slower + costs more |
| **Versioning** | Keep all versions of an object (for recovery/audit) | Versioning = more storage cost. Enable for important buckets. |

#### S3 for Data Lakes — Partitioning Strategy

How you organize files in S3 **directly affects query performance and cost**. Athena, Spark, and Redshift Spectrum all use S3 prefixes to skip irrelevant data (partition pruning).

```
de-training-bronze/
├── calls/
│   ├── date=2026-03-17/
│   │   └── calls_20260317.json        ← one file per day
│   ├── date=2026-03-18/
│   │   └── calls_20260318.json
│   └── date=2026-03-19/
│       └── calls_20260319.json
├── orders/
│   ├── date=2026-03-17/
│   │   └── orders_20260317.csv
│   └── ...
└── payments/
    ├── date=2026-03-17/
    │   └── payments_20260317.xml
    └── ...
```

**Why partition by date?**
- Query: "Give me all calls from March 18" → Spark only reads `date=2026-03-18/` prefix
- Without partitioning: Spark reads ALL files to find March 18 data
- Partition pruning = less data scanned = faster + cheaper

**Partitioning rules of thumb:**

| Partition Column | Good? | Why |
|-----------------|-------|-----|
| `date` | Yes | Most DE queries are time-based. Each partition is ~same size. |
| `customer_id` | No | Too many partitions (high cardinality). Tiny files = slow. |
| `channel` (VA/live) | Maybe | Only 2 values — useful if you always query one channel. |
| `campaign_name` | Maybe | ~50 values — reasonable if queries always filter by campaign. |


In [ ]:
# ============================================================
# Demo: Create an S3 bucket for our data lake
# ============================================================
# Replace 'de-training-YOUR-NAME' with a unique name

BUCKET_NAME = "de-training-demo"  # CHANGE THIS to something unique

!aws s3 mb s3://{BUCKET_NAME} --region us-east-1

print(f"\nBucket created: s3://{BUCKET_NAME}")
print("Verify in the AWS Console: https://s3.console.aws.amazon.com/s3/")

# **You should see:** `make_bucket: de-training-YOUR-NAME`. If you see `BucketAlreadyExists`, try a different name (bucket names are globally unique). Verify in the S3 console — your bucket should appear in the list.

In [ ]:
# ============================================================
# Demo: Create the partition folder structure
# ============================================================
import os

# Create local files to upload
os.makedirs('/tmp/demo_data/calls/date=2026-03-19', exist_ok=True)
os.makedirs('/tmp/demo_data/orders/date=2026-03-19', exist_ok=True)

# Create sample data
with open('/tmp/demo_data/calls/date=2026-03-19/calls.json', 'w') as f:
    f.write('{"call_id": "C001", "dnis": "8005551234", "duration": 120, "disposition": "completed"}\n')
    f.write('{"call_id": "C002", "dnis": "8005551234", "duration": 45, "disposition": "transferred"}\n')

with open('/tmp/demo_data/orders/date=2026-03-19/orders.csv', 'w') as f:
    f.write('order_id,call_id,sku,quantity,total\n')
    f.write('O001,C001,SKU-100,1,49.99\n')

print("Sample data created:")
!find /tmp/demo_data -type f

In [ ]:
# ============================================================
# Demo: Upload to S3 with partition structure
# ============================================================

!aws s3 cp /tmp/demo_data/ s3://{BUCKET_NAME}/bronze/ --recursive

print(f"\nFiles in S3:")
!aws s3 ls s3://{BUCKET_NAME}/bronze/ --recursive

# **You should see:** `upload: ...` lines for each file. After running, `aws s3 ls s3://YOUR-BUCKET/bronze/ --recursive` should show your files with their sizes.

In [ ]:
# ============================================================
# Demo: S3 operations you'll use daily
# ============================================================

print("=== Your buckets ===")
!aws s3 ls

print("\n=== List a specific prefix (like a folder) ===")
!aws s3 ls s3://{BUCKET_NAME}/bronze/calls/

print("\n=== Download a file ===")
!aws s3 cp s3://{BUCKET_NAME}/bronze/calls/date=2026-03-19/calls.json /tmp/downloaded.json
!cat /tmp/downloaded.json

print("\n=== Get file size and metadata ===")
!aws s3 ls s3://{BUCKET_NAME}/bronze/ --recursive --summarize | tail -3

#### AWS Console Walkthrough: S3

> **Open the S3 Console:** [s3.console.aws.amazon.com/s3](https://s3.console.aws.amazon.com/s3/)

**Create a Bucket (Console):**

1. Click **Create bucket**
2. Bucket name: `de-training-YOUR-NAME` (must be globally unique)
3. AWS Region: `US East (N. Virginia) us-east-1`
4. Block Public Access: **leave all checked** (default — data lakes should never be public)
5. Bucket Versioning: **Enable** (for Bronze layer — keeps history of file uploads)
6. Default encryption: **SSE-S3** (default — free server-side encryption)
7. Click **Create bucket**

**Upload Files (Console):**

1. Click into your bucket
2. Click **Create folder** → name it `bronze` → Create
3. Click into `bronze/` → Create folder → `calls` → Create
4. Click into `calls/` → Create folder → `date=2026-03-19` → Create
5. Click into `date=2026-03-19/` → **Upload** → drag your `calls.json` file → Upload

**Explore the S3 Console:**

| Tab/Feature | What It Shows | Why DEs Care |
|---|---|---|
| **Objects** tab | Files and "folders" (prefixes) | Browse your data lake visually |
| **Properties** tab | Versioning, encryption, static website | Check bucket configuration |
| **Permissions** tab | Bucket policy, ACLs, Block Public Access | Debug "Access Denied" errors |
| **Metrics** tab | Request count, bytes downloaded, errors | Monitor pipeline I/O |
| **Management** tab | Lifecycle rules, replication, inventory | Set up archival policies |

**Set a Lifecycle Rule (Console):**

1. Click **Management** tab → **Create lifecycle rule**
2. Rule name: `ArchiveBronzeAfter30Days`
3. Filter: Prefix = `bronze/`
4. Lifecycle rule actions: check **Transition current versions** and **Expire current versions**
5. Transition to Standard-IA after **30 days**
6. Transition to Glacier after **90 days**
7. Expire after **365 days**
8. Click **Create rule**

> **Teaching point:** The console shows you what the CLI/SDK does under the hood. Every button click maps to an API call. The lifecycle rule you just created is the same JSON we built programmatically earlier.

#### S3 Storage Classes and Lifecycle

Not all data needs the same access speed. S3 offers different storage classes at different price points:

| Storage Class | Cost (per GB/month) | Access Time | When to Use |
|--------------|--------------------:|-------------|-------------|
| **Standard** | ~$0.023 | Milliseconds | Active data — Bronze/Silver/Gold |
| **Standard-IA** (Infrequent Access) | ~$0.0125 | Milliseconds | Data accessed < 1x/month but needs quick retrieval |
| **Glacier Instant** | ~$0.004 | Milliseconds | Archive with instant retrieval (compliance, audits) |
| **Glacier Flexible** | ~$0.0036 | Minutes to hours | Archive — rarely accessed (years of historical data) |
| **Glacier Deep Archive** | ~$0.00099 | 12+ hours | Long-term archive — almost never accessed |

**Lifecycle policies** automatically move data between classes:

```
Day 0:   calls.json lands in Standard (Bronze)            $0.023/GB
Day 30:  Pipeline processed it → move to Standard-IA      $0.0125/GB
Day 90:  Compliance requires 1 year retention → Glacier    $0.004/GB
Day 365: Delete (or move to Deep Archive)
```


In [ ]:
# ============================================================
# Demo: Set a lifecycle policy on the Bronze bucket
# ============================================================

import json

lifecycle = {
    "Rules": [
        {
            "ID": "ArchiveBronzeAfter30Days",
            "Filter": {"Prefix": "bronze/"},
            "Status": "Enabled",
            "Transitions": [
                {"Days": 30, "StorageClass": "STANDARD_IA"},
                {"Days": 90, "StorageClass": "GLACIER"}
            ],
            "Expiration": {"Days": 365}
        }
    ]
}

with open('/tmp/lifecycle.json', 'w') as f:
    json.dump(lifecycle, f, indent=2)

print("Lifecycle policy:")
print(json.dumps(lifecycle, indent=2))

# Apply it:
# !aws s3api put-bucket-lifecycle-configuration \
#     --bucket {BUCKET_NAME} \
#     --lifecycle-configuration file:///tmp/lifecycle.json

#### S3 Event Notifications — How File Arrival Triggers Pipelines

S3 can notify other services when something happens (file created, deleted, etc.). This is how event-driven pipelines work:

```
calls.json uploaded to S3
         │
         ▼
S3 Event Notification fires
         │
    ┌────┴────┬──────────┐
    ▼         ▼          ▼
  Lambda     SQS     EventBridge
  (validate  (queue   (route to
   schema)   for      Step Functions,
             batch    multiple targets)
             processing)
```

| Notification Target | When to Use |
|-------------------|-------------|
| **Lambda** | Lightweight action: validate file, move to Bronze, update metadata |
| **SQS** | Buffer events for batch processing: "process all new files every 5 minutes" |
| **SNS (Simple Notification Service)** | Fan-out: notify multiple systems (Lambda + SQS + email) |
| **EventBridge** | Complex routing: different actions based on file prefix, size, etc. |


In [ ]:
# ============================================================
# Demo: S3 Event Notification configuration
# ============================================================

import json

notification_config = {
    "QueueConfigurations": [
        {
            "Id": "NewCallDataNotification",
            "QueueArn": "arn:aws:sqs:us-east-1:ACCOUNT_ID:de-pipeline-trigger",
            "Events": ["s3:ObjectCreated:*"],
            "Filter": {
                "Key": {
                    "FilterRules": [
                        {"Name": "prefix", "Value": "bronze/calls/"}
                    ]
                }
            }
        }
    ]
}

print("S3 Event Notification Config:")
print(json.dumps(notification_config, indent=2))
print()
print("What this does:")
print("  1. A new file lands in s3://bucket/bronze/calls/...")
print("  2. S3 sends a message to the SQS queue")
print("  3. A Lambda or Step Function reads the queue and kicks off the pipeline")

#### AWS Console Walkthrough: S3 Event Notifications

> **In the S3 Console:** Click your bucket → **Properties** tab → scroll to **Event notifications**

**Create an Event Notification (Console):**

1. Click **Create event notification**
2. Event name: `NewCallDataArrival`
3. Prefix: `incoming/calls/` (only trigger for files in this prefix)
4. Suffix: `.json` (only trigger for JSON files)
5. Event types: check **All object create events** (`s3:ObjectCreated:*`)
6. Destination:
   - **SQS queue** → select your queue ARN (must exist first)
   - OR **Lambda function** → select your validation function
7. Click **Save changes**

**What you'll see after setup:**

```
Properties tab → Event notifications section:
┌─────────────────────────────────────────────────┐
│  Name: NewCallDataArrival                       │
│  Events: s3:ObjectCreated:*                     │
│  Prefix: incoming/calls/                        │
│  Suffix: .json                                  │
│  Destination: SQS → arn:aws:sqs:...:de-trigger  │
└─────────────────────────────────────────────────┘
```

**Test it:** Upload a file to `incoming/calls/` → check the SQS console → you should see a message with the S3 event details.

> **Common gotcha:** The SQS queue needs a **resource policy** that allows S3 to send messages to it. If the event notification fails silently, this is usually why. The console will warn you if the policy is missing.

#### S3: Edge Cases, Gotchas, and Interview Questions

**Gotchas:**

| Gotcha | What Happens | Fix |
|---|---|---|
| Two pipelines write to same prefix simultaneously | Last write wins — data loss (plain S3) | Use Delta Lake (ACID) or write to unique file names |
| Upload fails halfway (large file) | Partial file on S3 — corrupted | Use multipart upload (CLI does this automatically for >8MB) |
| Too many small files | Slow reads — each file = 1 S3 API call | Compact with Delta Lake OPTIMIZE or Spark coalesce |
| Forgot to set lifecycle rules | Storage costs grow forever | Set lifecycle on day 1 — Bronze → IA after 30d, Glacier after 90d |
| Bucket name has dots | HTTPS certificate issues with virtual-hosted URLs | Use hyphens, not dots, in bucket names |
| `SELECT *` on Athena/Spectrum | Scans all columns = expensive | Always select specific columns |

**Interview questions you should be able to answer:**

1. "What is partition pruning and why does it matter?" — Query only reads relevant partitions, skipping the rest. Saves time and cost.
2. "S3 is eventually consistent or strongly consistent?" — Strongly consistent since December 2020. Read-after-write consistency for all operations.
3. "How would you handle a pipeline that creates thousands of small files?" — Use Spark coalesce/repartition before writing. Or run Delta Lake OPTIMIZE to compact after.
4. "What happens if you delete a file while a query is reading it?" — S3 serves the object from cache. The read completes. Next read gets 404.
5. "How do you secure data in S3?" — Bucket policies (resource-based), IAM policies (identity-based), encryption (SSE-S3, SSE-KMS), VPC endpoints (no public internet).

### 2.3 Redshift — Cloud Data Warehouse

Redshift is AWS's columnar data warehouse. It's where your **Gold layer** marts live — pre-aggregated, star schema, ready for analysts and dashboards.

#### Architecture

```
┌──────────────────────────────────────────┐
│              REDSHIFT CLUSTER            │
│                                          │
│  ┌──────────┐   ┌──────────┐            │
│  │ Leader   │   │ Compute  │            │
│  │ Node     │──▶│ Node 1   │  Stores data in
│  │          │   │          │  columnar format
│  │ Parses   │   ├──────────┤            │
│  │ queries, │   │ Compute  │            │
│  │ builds   │──▶│ Node 2   │            │
│  │ plan,    │   │          │            │
│  │ returns  │   ├──────────┤            │
│  │ results  │──▶│ Compute  │            │
│  │          │   │ Node N   │            │
│  └──────────┘   └──────────┘            │
│                                          │
│  Spectrum: Query S3 data WITHOUT loading │
│  ──────────────────────────────────────  │
│  CREATE EXTERNAL TABLE → reads Parquet   │
│  on S3 directly (lakehouse pattern)      │
└──────────────────────────────────────────┘
```

#### Key Concepts

| Concept | What It Is | Why It Matters |
|---------|-----------|----------------|
| **Columnar storage** | Data stored by column, not row | Queries that read 3 of 50 columns only scan those 3 |
| **Distribution key** | How rows are spread across nodes | Bad key = data skew = one node does all the work |
| **Sort key** | How data is ordered on disk | Queries with `WHERE date > X` skip sorted blocks |
| **Spectrum** | Query engine for S3 data | Query your data lake without loading into Redshift |
| **Materialized views** | Pre-computed query results | Speed up repeated analytical queries |
| **RA3 nodes** | Managed storage nodes (compute-storage separated) | Scale compute and storage independently |
| **Serverless** | No cluster management | Pay per query, auto-scales — good for variable workloads |


#### AWS Console Walkthrough: Redshift

> **Open the Redshift Console:** [console.aws.amazon.com/redshiftv2](https://console.aws.amazon.com/redshiftv2/)

**Option 1: Redshift Serverless (recommended for training)**

1. Left sidebar → **Serverless dashboard** → **Create workgroup**
2. Workgroup name: `de-training`
3. Namespace: **Create new** → `de-training-ns`
4. Base capacity: **8 RPU** (minimum — cheapest option)
5. Database name: `dev` (default)
6. Admin user: set username and password
7. VPC: default VPC is fine for training
8. Click **Create** (takes 2-3 minutes)

**Option 2: Provisioned Cluster (production-like)**

1. Left sidebar → **Provisioned clusters** → **Create cluster**
2. Cluster identifier: `de-training-cluster`
3. Node type: **dc2.large** (cheapest — $0.25/hr)
4. Number of nodes: **2** (minimum for multi-node)
5. Database name: `dev`, Master user: `admin`, set password
6. Click **Create cluster** (takes 5-10 minutes)

**Connect and Query (Redshift Query Editor v2):**

1. Left sidebar → **Query editor v2** (built into the console — no client needed)
2. Connect to your cluster/workgroup
3. Run:
   ```sql
   CREATE SCHEMA bronze;
   CREATE SCHEMA silver;
   CREATE SCHEMA gold;

   -- Verify
   SELECT schema_name FROM information_schema.schemata;
   ```

**Redshift Console — Key Tabs:**

| Tab | What It Shows | Why DEs Care |
|---|---|---|
| **Query monitoring** | Running queries, queue depth, execution plans | Debug slow queries |
| **Cluster performance** | CPU, disk, network, query throughput | Right-size your cluster |
| **Maintenance** | Snapshots, backups, resize operations | Plan maintenance windows |
| **Properties** | Endpoint URL, port, VPC, IAM roles | Get connection details for your pipeline |

> **Teaching point:** Click the **Properties** tab → **Associated IAM roles** → this is where you attach the `DE-Pipeline-EMR-Role` so Redshift can read S3 via COPY and Spectrum.

#### Redshift Data Loading Commands

| Command | What It Does | When to Use |
|---------|-------------|-------------|
| `COPY` | Bulk load from S3, DynamoDB, EMR, or SSH | **Primary loading method** — fastest way to load. Parallelizes across slices. |
| `UNLOAD` | Export table → S3 (Parquet, CSV, JSON) | Export data for other systems, cross-platform sharing |
| `MERGE INTO` | Upsert (insert + update + delete in one statement) | SCD Type 2, CDC, late-arriving data |
| `INSERT INTO ... SELECT` | Transform and load within Redshift | Silver → Gold transforms when data is already in Redshift |
| `CREATE EXTERNAL TABLE` | Query S3 via Redshift Spectrum (no load) | Lakehouse pattern — query Bronze/Silver directly from Redshift |
| `CREATE EXTERNAL SCHEMA` | Map Glue Catalog to Redshift | Bridge between your data lake metadata and Redshift |
| Streaming Ingestion | Auto-ingest from Kinesis Data Streams | Near-real-time loading into warehouse |
| `ALTER TABLE APPEND` | Move rows from staging → target (no copy, just metadata) | Fast staging table pattern — zero-copy merge |


#### Hello World: Redshift

Before we get into COPY, MERGE, and Spectrum — let's just connect and run one query:


In [ ]:
# ============================================================
# Hello World: Redshift — simplest possible operation
# ============================================================

# Option 1: Use the Redshift Query Editor v2 in the console
# (no setup needed — just open the console and type SQL)

hello_redshift = """
-- Connect to Redshift Query Editor v2, then run:

-- 1. What version am I running?
SELECT version();

-- 2. Create a table with one row
CREATE TABLE test_hello (message VARCHAR(50));
INSERT INTO test_hello VALUES ('Hello from Redshift!');
SELECT * FROM test_hello;

-- 3. Drop it (cleanup)
DROP TABLE test_hello;

-- That's it. You just ran SQL on a cloud data warehouse.
-- Everything after this builds on this: SQL + cloud = Redshift.
"""
print(hello_redshift)

In [ ]:
# ============================================================
# Demo: Redshift COPY command — loading data from S3
# ============================================================

redshift_copy = '''
-- Load calls data from S3 into Redshift
COPY bronze_calls
FROM 's3://de-training-bronze/calls/date=2026-03-19/'
IAM_ROLE 'arn:aws:iam::ACCOUNT_ID:role/DE-Pipeline-Redshift-Role'
FORMAT AS JSON 'auto'
TIMEFORMAT 'auto'
REGION 'us-east-1';

-- Verify the load
SELECT COUNT(*) as total_rows,
       MIN(start_time) as earliest_call,
       MAX(start_time) as latest_call
FROM bronze_calls;
'''
print(redshift_copy)
print("Key points:")
print("  - IAM_ROLE: Redshift assumes this role to read S3 (no keys in SQL!)")
print("  - FORMAT AS JSON 'auto': auto-detect JSON structure")
print("  - COPY parallelizes across nodes -- much faster than INSERT")

#### Why Is COPY So Much Faster Than INSERT? (Under the Hood)

A common beginner mistake: loading data row-by-row with INSERT statements.

```sql
-- SLOW: 100,000 individual INSERT statements
INSERT INTO calls VALUES ('C00001', ...);
INSERT INTO calls VALUES ('C00002', ...);
-- Each INSERT: parse SQL -> find the right node -> write 1 row -> commit
-- 100K inserts = 100K round trips. Takes minutes to hours.

-- FAST: One COPY command
COPY calls FROM 's3://bucket/calls/' IAM_ROLE '...' FORMAT AS JSON;
-- COPY: reads ALL files in parallel across ALL compute nodes simultaneously
-- 100K rows loaded in seconds.
```

**What COPY does under the hood:**

1. Leader node reads the S3 manifest (list of files)
2. Leader distributes files across compute nodes (each node reads different files in parallel)
3. Each node parses, validates, and loads its portion directly into columnar storage
4. Leader commits the transaction (all-or-nothing)

> **Rule of thumb:** Never INSERT row-by-row into Redshift. Always batch-load with COPY. Even if you have just 100 rows, COPY is the right pattern.

In [ ]:
# ============================================================
# Demo: Redshift UNLOAD — exporting to S3
# ============================================================

redshift_unload = '''
-- Export Gold mart to S3 as Parquet (for other tools to consume)
UNLOAD ('SELECT * FROM gold.mart_daily_campaign WHERE report_date >= \'2026-03-01\'')
TO 's3://de-training-gold/exports/mart_daily_campaign/'
IAM_ROLE 'arn:aws:iam::ACCOUNT_ID:role/DE-Pipeline-Redshift-Role'
FORMAT AS PARQUET
PARTITION BY (report_date)
ALLOWOVERWRITE;
'''
print(redshift_unload)
print("Key points:")
print("  - FORMAT AS PARQUET: columnar output — efficient for downstream tools")
print("  - PARTITION BY: creates Hive-style partitions in S3")
print("  - ALLOWOVERWRITE: replace existing files (idempotent)")

In [ ]:
# ============================================================
# Demo: Redshift MERGE — upsert pattern
# ============================================================

redshift_merge = '''
-- MERGE: Update existing records, insert new ones
MERGE INTO silver.calls AS target
USING staging.new_calls AS source
ON target.call_id = source.call_id
WHEN MATCHED THEN UPDATE SET
    disposition = source.disposition,
    payment_status = source.payment_status,
    updated_at = GETDATE()
WHEN NOT MATCHED THEN INSERT (
    call_id, dnis, caller_phone, start_time, duration,
    disposition, payment_status, created_at
)
VALUES (
    source.call_id, source.dnis, source.caller_phone,
    source.start_time, source.duration,
    source.disposition, source.payment_status, GETDATE()
);
'''
print(redshift_merge)
print("Key points:")
print("  - WHEN MATCHED: update existing call records (disposition changed)")
print("  - WHEN NOT MATCHED: insert new calls")
print("  - This is the core pattern for Silver layer updates")

In [ ]:
# ============================================================
# Demo: Redshift Spectrum — query S3 without loading
# ============================================================

redshift_spectrum = '''
-- Step 1: Create an external schema pointing to the Glue Catalog
CREATE EXTERNAL SCHEMA bronze_lake
FROM DATA CATALOG
DATABASE 'de_training'
IAM_ROLE 'arn:aws:iam::ACCOUNT_ID:role/DE-Pipeline-Redshift-Role'
REGION 'us-east-1';

-- Step 2: Query S3 data directly through Redshift
SELECT date, COUNT(*) as call_count, AVG(duration) as avg_duration
FROM bronze_lake.calls
WHERE date >= '2026-03-17'
GROUP BY date
ORDER BY date;

-- Why this matters:
-- Your data stays in S3 (cheap storage)
-- Redshift queries it on demand (pay for compute only when querying)
-- This is the "lakehouse" pattern — warehouse queries on lake storage
'''
print(redshift_spectrum)

#### Redshift: Edge Cases and Interview Questions

**Gotchas:**

| Gotcha | What Happens | Fix |
|---|---|---|
| Wrong distribution key | One node does all the work (data skew) | Choose a key with high cardinality and even distribution |
| COPY fails on row 50K of 100K | Entire COPY fails — no rows loaded | Use MAXERROR to allow some failures, check STL_LOAD_ERRORS |
| Forgot to VACUUM/ANALYZE | Deleted rows still on disk, query plans are stale | Schedule VACUUM and ANALYZE regularly |
| Cluster is full (disk 100%) | Queries fail, can't insert | Monitor disk usage, resize or archive old data |

**Interview questions:**

1. "What's the difference between a distribution key and a sort key?" — Distribution = how rows are spread across nodes. Sort = how data is ordered on disk within a node.
2. "When would you use Redshift Spectrum?" — Query S3 data without loading. Good for historical/cold data. Hot data goes in Redshift tables.
3. "How do you handle slowly growing costs?" — Reserved instances (1yr commit), pause/resume for dev clusters, Spectrum for cold data, archive old data to S3.

### 2.4 Athena — Serverless SQL on S3

> **In plain English:** Athena is a librarian who can read any book on the shelf without checking it out. Your data stays in S3 (the shelf). You ask a question in SQL. Athena reads the files, gives you an answer, and charges $5 per terabyte read. No servers, no setup, no waiting.

Athena lets you run SQL queries directly on files in S3. No server to manage. No data to load. Just point at Parquet/CSV/JSON files and query.

| Aspect | Athena | Redshift |
|--------|--------|----------|
| **Infrastructure** | None — fully serverless | Cluster (or Serverless mode) |
| **Pricing** | $5 per TB scanned | Per-node-hour + storage |
| **Best for** | Ad-hoc queries, exploration, occasional reports | Heavy analytics, concurrent users, complex joins |
| **Setup time** | Minutes | 10-20 minutes (cluster) |
| **Performance** | Good for single queries | Better for repeated/complex workloads |

> **Under the hood:** Athena uses Trino (formerly PrestoDB). Redshift Spectrum uses the Redshift engine. Both query S3 — different engines, different strengths.


In [ ]:
# ============================================================
# Demo: Athena query on S3 data
# ============================================================

athena_query = '''
-- Athena: Query JSON files in S3 directly
CREATE EXTERNAL TABLE IF NOT EXISTS bronze_calls (
    call_id STRING,
    dnis STRING,
    caller_phone STRING,
    start_time TIMESTAMP,
    duration INT,
    disposition STRING
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://de-training-bronze/calls/'
TBLPROPERTIES ('has_encrypted_data'='false');

-- Then query it
SELECT disposition, COUNT(*) as cnt, AVG(duration) as avg_duration
FROM bronze_calls
WHERE date = '2026-03-19'
GROUP BY disposition
ORDER BY cnt DESC;

-- Athena charges $5 per TB scanned
-- Parquet files = columnar = only scan columns you SELECT
-- Partitioning = only scan the dates you WHERE
-- Both save money!
'''
print(athena_query)

### 2.5 Glue — Data Catalog and ETL


> **Key terms:** **ETL** = Extract, Transform, Load (transform data BEFORE loading into the warehouse — clean it first, then store it). **ELT** = Extract, Load, Transform (load raw data first, then transform inside the warehouse — BigQuery/Snowflake are powerful enough to transform after loading). Modern cloud pipelines mostly use ELT because cloud warehouses handle transformation efficiently. Our medallion pipeline (Bronze → Silver → Gold) is an ELT pattern: we load raw data to Bronze (Extract + Load), then transform in Silver/Gold (Transform).


AWS Glue has two main parts:

| Component | What It Does | You'll Use It For |
|-----------|-------------|-------------------|
| **Data Catalog** | Centralized metadata store — table definitions, schemas, partitions | Athena and Spectrum need this to know the schema of S3 files |
| **Crawlers** | Automatically scan S3 and infer schemas → populate the Catalog | Run after uploading new data to Bronze — auto-discovers schema |
| **ETL Jobs** | Serverless PySpark jobs (Glue manages the cluster) | Alternative to EMR for simple transforms |

```
S3 files (Parquet, JSON, CSV)
         │
         ▼ Crawler scans files
         │
    ┌────┴────┐
    │  Glue   │  ← Schema, partitions, column types
    │ Catalog │     stored as metadata
    └────┬────┘
         │
    ┌────┴────┬──────────┐
    ▼         ▼          ▼
  Athena   Spectrum    EMR/Spark
  (queries) (queries)  (reads catalog
                        for schema)
```

> **Think of the Glue Catalog as the "phone book" for your data lake.** Without it, every tool would need to know the schema and location of every file. With it, you define the schema once and every tool can find it.


In [ ]:
# ============================================================
# Demo: Run a Glue Crawler to discover schemas
# ============================================================

glue_commands = '''
# Create a crawler that scans our Bronze bucket
aws glue create-crawler \
    --name de-training-bronze-crawler \
    --role DE-Pipeline-Glue-Role \
    --database-name de_training \
    --targets '{"S3Targets": [{"Path": "s3://de-training-bronze/"}]}'

# Run the crawler
aws glue start-crawler --name de-training-bronze-crawler

# Check status
aws glue get-crawler --name de-training-bronze-crawler \
    --query 'Crawler.State'

# After crawler completes, see discovered tables
aws glue get-tables --database-name de_training \
    --query 'TableList[*].[Name,StorageDescriptor.Columns[*].Name]'
'''
print(glue_commands)
print()
print("What the crawler does:")
print("  1. Scans all files under s3://de-training-bronze/")
print("  2. Infers schema (column names, types) from file content")
print("  3. Detects partitions (date=2026-03-19/)")
print("  4. Creates tables in the Glue Catalog (de_training database)")
print("  5. Now Athena and Spectrum can query these tables by name")

#### AWS Console Walkthrough: Glue

> **Open the Glue Console:** [console.aws.amazon.com/glue](https://console.aws.amazon.com/glue/)

**Create a Database (Console):**

1. Left sidebar → **Databases** (under Data Catalog) → **Add database**
2. Name: `de_training`
3. Click **Create database**

**Create and Run a Crawler (Console):**

1. Left sidebar → **Crawlers** → **Create crawler**
2. Name: `de-training-bronze-crawler`
3. Click **Next**
4. Data source: **Add a data source**
   - Data store: **S3**
   - S3 path: `s3://de-training-YOUR-NAME/bronze/`
   - Crawl all sub-folders: **Yes**
5. Click **Next**
6. IAM role: select or create `DE-Pipeline-Glue-Role`
7. Click **Next**
8. Target database: `de_training`
9. Click **Next** → **Create crawler**
10. Select the crawler → **Run**

**After the crawler completes (1-2 minutes):**

1. Left sidebar → **Tables** (under Data Catalog)
2. You should see tables auto-discovered: `calls`, `orders`, etc.
3. Click a table → see the schema (column names, types) that the crawler inferred
4. Click **Partitions** → see the `date=` partitions it found

**What the Catalog looks like:**

```
de_training (database)
├── calls           ← inferred from bronze/calls/ (JSON)
│   ├── call_id: string
│   ├── dnis: string
│   ├── duration: int
│   ├── customer: struct<first_name:string, ...>
│   └── Partitions: date=2026-03-17, date=2026-03-18, date=2026-03-19
├── orders          ← inferred from bronze/orders/ (CSV)
│   ├── order_id: string
│   ├── call_id: string
│   └── ...
└── products        ← inferred from bronze/products/ (CSV)
```

> **Teaching point:** This catalog is now available to Athena, Spectrum, and EMR Spark. One crawler run = all three query engines can find your tables.

### 2.6 EMR — Elastic MapReduce (Managed Spark)

> **In plain English:** EMR is like hiring temp workers through a staffing agency. You tell AWS: "I need 5 workers with Spark skills for 2 hours." AWS provisions the machines, installs Spark, and your job runs. When it's done, the workers go home and you stop paying. You can even use "Spot" workers — temps who accept 70% less pay but might leave early if someone offers more.

EMR is where your PySpark jobs run in the cloud. It's a managed cluster — you tell AWS what size/type of machines, AWS provisions them, installs Spark, and you submit jobs.

#### Cluster Architecture

```
┌─────────────────────────────────────────────────┐
│                 EMR CLUSTER                      │
│                                                  │
│  ┌──────────────┐                                │
│  │ Master Node  │  Runs YARN ResourceManager,    │
│  │              │  Spark driver, Hive metastore   │
│  └──────┬───────┘                                │
│         │                                        │
│  ┌──────┴───────┐  ┌──────────────┐              │
│  │ Core Node 1  │  │ Core Node 2  │  Store data  │
│  │ (HDFS+Spark) │  │ (HDFS+Spark) │  (optional)  │
│  └──────────────┘  └──────────────┘  + compute   │
│                                                  │
│  ┌──────────────┐  ┌──────────────┐              │
│  │ Task Node 1  │  │ Task Node 2  │  Compute     │
│  │ (Spark only) │  │ (Spark only) │  only -- use │
│  │ SPOT instance│  │ SPOT instance│  spot for    │
│  └──────────────┘  └──────────────┘  cost savings│
└─────────────────────────────────────────────────┘
```

**Cost strategy:**
- **Master node:** On-Demand (must be reliable)
- **Core nodes:** On-Demand or Reserved (data storage)
- **Task nodes:** Spot instances (70-90% discount! If terminated, only lose in-progress tasks)

#### EMR Serverless

New option: **EMR Serverless** — no cluster management at all. You submit a job and AWS allocates compute automatically.

| | EMR on EC2 (clusters) | EMR Serverless |
|---|---|---|
| You manage | Cluster size, instance types, auto-scaling | Nothing — just submit code |
| Startup time | 5-15 min (cluster provision) | ~1 min |
| Cost model | Pay while cluster runs | Pay per vCPU-second while job runs |
| Best for | Long-running clusters, interactive notebooks | Batch jobs, variable workloads |


#### Hello World: EMR

Before cluster configuration, Spot instances, and auto-scaling — let's understand the simplest EMR operation:


In [ ]:
# ============================================================
# Hello World: EMR — the simplest possible Spark job
# ============================================================

hello_emr = """
# EMR Serverless: run a one-liner PySpark job (no cluster to manage)

# Step 1: Create an EMR Serverless application
aws emr-serverless create-application \\
    --name hello-spark \\
    --release-label emr-7.0.0 \\
    --type SPARK

# Step 2: Submit the simplest possible job
# (This PySpark script just counts rows in a CSV)
aws emr-serverless start-job-run \\
    --application-id YOUR_APP_ID \\
    --execution-role-arn arn:aws:iam::ACCOUNT:role/EMR-Serverless-Role \\
    --job-driver '{
        "sparkSubmit": {
            "entryPoint": "s3://de-training-code/hello_spark.py"
        }
    }'

# hello_spark.py (3 lines):
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName('hello').getOrCreate()
# print(f'Row count: {spark.read.csv("s3://de-training/data/orders.csv", header=True).count()}')

# That's it. Spark ran in the cloud, read S3, counted rows, and shut down.
# No cluster to create. No cluster to terminate. Pay only for the seconds it ran.
"""
print(hello_emr)

In [ ]:
# ============================================================
# Demo: Create an EMR cluster and submit a PySpark job
# ============================================================

# Step 1: Create a cluster
emr_create = '''
aws emr create-cluster \\
    --name "DE-Training-Pipeline" \\
    --release-label emr-7.0.0 \\
    --applications Name=Spark Name=Hive \\
    --instance-groups '[
        {"InstanceGroupType":"MASTER","InstanceType":"m5.xlarge","InstanceCount":1},
        {"InstanceGroupType":"CORE","InstanceType":"m5.xlarge","InstanceCount":2},
        {"InstanceGroupType":"TASK","InstanceType":"m5.xlarge","InstanceCount":2,
         "Market":"SPOT","BidPrice":"0.10"}
    ]' \\
    --service-role EMR_DefaultRole \\
    --ec2-attributes InstanceProfile=EMR_EC2_DefaultRole \\
    --auto-terminate
'''

# Step 2: Submit a PySpark job
emr_step = '''
aws emr add-steps \\
    --cluster-id j-XXXXXXXXXXXXX \\
    --steps '[{
        "Type": "Spark",
        "Name": "Silver Transform",
        "ActionOnFailure": "CONTINUE",
        "Args": [
            "spark-submit",
            "--deploy-mode", "cluster",
            "s3://de-training-code/silver_transform.py"
        ]
    }]'
'''

print("Step 1: Create EMR cluster")
print(emr_create)
print("\nStep 2: Submit PySpark job to the cluster")
print(emr_step)
print("\nStep 3: Monitor in EMR Console or CLI")
print("  aws emr describe-cluster --cluster-id j-XXXXXXXXXXXXX")
print("  aws emr list-steps --cluster-id j-XXXXXXXXXXXXX")

#### Why Not Just Run PySpark on Your Laptop? (EMR Under the Hood)

Your laptop can run PySpark on a 100MB file. But your pipeline has 100GB.

| | Your Laptop | EMR Cluster (5 nodes) |
|---|---|---|
| **RAM** | 16 GB | 80 GB (5 x 16 GB) |
| **CPU cores** | 8 | 40 (5 x 8) |
| **Network to S3** | Internet (~100 Mbps) | AWS internal (~10 Gbps) |
| **100GB read time** | ~2 hours | ~80 seconds |
| **Spark executors** | 1 (your machine) | 10+ (across nodes) |

EMR's job is to give you **temporary, scalable compute next to your data**. S3 and EMR are in the same AWS datacenter — the data doesn't travel over the internet.

> **Key insight:** The reason cloud DE pipelines are fast isn't just parallelism — it's **data locality**. EMR nodes and S3 buckets are in the same building. Your laptop is across the country.

#### AWS Console Walkthrough: EMR

> **Open the EMR Console:** [console.aws.amazon.com/emr](https://console.aws.amazon.com/emr/)

**Create a Cluster (Console):**

1. Click **Create cluster**
2. **Name:** `DE-Training-Pipeline`
3. **Release:** `emr-7.0.0` (latest)
4. **Applications:** Select **Spark** and **Hive**
5. **Cluster configuration:**
   - Instance groups:
     - Primary (Master): `m5.xlarge` × 1
     - Core: `m5.xlarge` × 2
     - Task: `m5.xlarge` × 2 → **Purchase option: Spot** (saves 70-90%)
   - Click **Set Spot** → bid price: On-Demand price (let AWS manage)
6. **Networking:** Default VPC, default subnet
7. **Cluster termination:** Set **Auto-termination** → idle timeout: 1 hour (don't forget this!)
8. **Security:** Select IAM roles:
   - Service role: `EMR_DefaultRole`
   - Instance profile: `EMR_EC2_DefaultRole` (or your custom `DE-Pipeline-EMR-Role`)
9. Click **Create cluster** (takes 5-15 minutes to provision)

**Monitor the Cluster:**

| Console View | What It Shows |
|---|---|
| **Summary** tab | Cluster status, DNS name, uptime, instance counts |
| **Steps** tab | Submitted jobs and their status (Pending/Running/Completed/Failed) |
| **Instances** tab | Each node, its type, market (On-Demand/Spot), status |
| **Application UIs** tab | Links to Spark UI, YARN ResourceManager, Ganglia |
| **Events** tab | Timeline of cluster events (provisioning, step execution, termination) |

**Submit a Step (Job) from Console:**

1. Click **Steps** tab → **Add step**
2. Step type: **Spark application**
3. Name: `Silver Transform`
4. Deploy mode: **Cluster**
5. Application location: `s3://de-training-code/silver_transform.py`
6. Click **Add step**

**Access Spark UI:**

1. **Application UIs** tab → click **Spark History Server**
2. This opens the Spark UI where you can see:
   - Jobs, stages, tasks
   - DAG (Directed Acyclic Graph) visualization
   - Executor memory/CPU usage
   - Shuffle read/write sizes

> **Cost alert:** Always set **auto-termination** or manually terminate when done. A 3-node `m5.xlarge` cluster costs ~$0.75/hr. Left running overnight = ~$9. Left running for a week = ~$126.

#### EMR: Edge Cases, Gotchas, and Interview Questions

**Gotchas:**

| Gotcha | What Happens | Fix |
|---|---|---|
| Forgot auto-termination | Cluster runs 24/7 — $$$$ | Always set `--auto-terminate` or max-idle |
| All Spot, no On-Demand | Spot termination kills the master node — job dies | Master = On-Demand. Only task nodes on Spot. |
| Too few partitions in Spark | One executor does all the work | `repartition()` before heavy operations |
| Jar/dependency missing | Job fails with ClassNotFoundException | Include jars in `--packages` or bootstrap action |
| Wrong instance type | OOM errors on data-heavy jobs | Use memory-optimized (r5) for joins, compute-optimized (c5) for transforms |

**Interview questions:**

1. "EMR vs Glue ETL — when would you choose each?" — EMR: complex PySpark, long-running, custom libraries. Glue: simple ETL, auto-scaling, no cluster management.
2. "How do you optimize EMR costs?" — Spot for task nodes (70-90% savings). Auto-terminate when idle. Right-size instances. EMR Serverless for variable workloads.
3. "What happens when a Spot instance is terminated?" — Spark re-schedules the lost tasks on remaining executors. Data on that node's local disk is lost (but S3 data is safe).

### 2.7 Lambda — Serverless Functions

> **In plain English:** Lambda is a motion-sensor light. Nobody manages it. When someone walks by (a file lands in S3), it turns on (runs your code). When they leave (code finishes), it turns off (stops billing). You don't pay rent on the room — only for the seconds the light was on.

Lambda runs code without servers. You upload a function, define a trigger, and AWS runs it when the trigger fires.

**For DE, Lambda is the "glue" (lowercase g) that connects services:**

| Use Case | Trigger | What Lambda Does |
|----------|---------|------------------|
| File validation | S3 event (new file) | Check schema, file size, move to Bronze or quarantine |
| Pipeline trigger | S3 event or CloudWatch schedule | Start EMR step, trigger Glue job, send SQS message |
| Lightweight transform | SQS message | Parse a single record, enrich from API, write to S3 |
| Dead letter processing | SQS DLQ | Log failed records, send alert, attempt retry |
| Alerting | SNS or EventBridge | Send Slack/email notification on pipeline failure |

**Lambda limits to know:**
- Max runtime: 15 minutes
- Max memory: 10 GB
- Max deployment package: 250 MB (unzipped)
- Not for: long-running Spark jobs, heavy data processing (use EMR/Glue)


#### Hello World: Lambda

Before triggers and S3 event processing — let's run the simplest possible function:


In [ ]:
# ============================================================
# Hello World: Lambda — simplest possible function
# ============================================================

hello_lambda = """
# In the Lambda Console -> Create function -> Author from scratch
# Runtime: Python 3.12
# Paste this code:

def lambda_handler(event, context):
    name = event.get('name', 'World')
    message = f'Hello, {name}! This ran on Lambda.'
    print(message)
    return {'statusCode': 200, 'body': message}

# Click Deploy. Click Test. Use this test event:
# {"name": "Data Engineer"}
#
# Result: 'Hello, Data Engineer! This ran on Lambda.'
# Duration: ~1 ms. Cost: ~$0.0000002.
#
# That's it. No server. No deployment pipeline. No Docker.
# Now imagine this function runs automatically when a file lands in S3...
"""
print(hello_lambda)

In [ ]:
# ============================================================
# Demo: Lambda function that validates a new S3 file
# ============================================================

validation_code = '''
import json
import boto3

s3 = boto3.client("s3")

def handler(event, context):
    # Triggered by S3 event when a new file arrives.
    # Validates the file and moves it to the right location.

    bucket = event["Records"][0]["s3"]["bucket"]["name"]
    key = event["Records"][0]["s3"]["object"]["key"]

    print(f"New file: s3://{bucket}/{key}")

    response = s3.get_object(Bucket=bucket, Key=key)
    content = response["Body"].read().decode("utf-8")

    lines = content.strip().split("\\n")
    valid_records = 0
    invalid_records = 0

    for line in lines:
        try:
            record = json.loads(line)
            if all(k in record for k in ["call_id", "dnis", "duration"]):
                valid_records += 1
            else:
                invalid_records += 1
        except json.JSONDecodeError:
            invalid_records += 1

    print(f"Valid: {valid_records}, Invalid: {invalid_records}")

    if invalid_records / max(len(lines), 1) > 0.1:
        dest = key.replace("incoming/", "quarantine/")
        print(f"Too many errors - moving to quarantine: {dest}")
    else:
        dest = key.replace("incoming/", "bronze/")
        print(f"Validation passed - moving to bronze: {dest}")

    s3.copy_object(
        Bucket=bucket,
        CopySource={"Bucket": bucket, "Key": key},
        Key=dest
    )
    s3.delete_object(Bucket=bucket, Key=key)

    return {"valid": valid_records, "invalid": invalid_records, "dest": dest}
'''

print(validation_code)
print("---")
print("Deploy this function, set the trigger to S3 events on the incoming/ prefix,")
print("and every new file gets validated automatically before entering Bronze.")

#### Why Lambda and Not a Script on EC2? (Under the Hood)

You could run this validation script on an EC2 instance with cron. Here's why Lambda is better for this:

| | EC2 + Cron | Lambda |
|---|---|---|
| **When no files arrive** | EC2 running, cron checking, paying ~$0.012/hr | Lambda idle, paying $0 |
| **When 100 files arrive at once** | Script processes them one at a time | Lambda runs 100 instances in parallel |
| **When script crashes** | You find out tomorrow (maybe) | Lambda retries automatically, logs to CloudWatch |
| **Patching the OS** | Your problem | AWS's problem |
| **Monthly cost for occasional use** | ~$8.60 (EC2 t3.micro 24/7) | ~$0.02 (1000 invocations x 200ms) |

> **Key insight:** Lambda is for **event-driven, short-lived** tasks. If your task runs for 15+ minutes or needs persistent state, use EMR or ECS instead. Lambda's sweet spot: validate a file (seconds), trigger a pipeline (seconds), send an alert (milliseconds).

#### AWS Console Walkthrough: Lambda

> **Open the Lambda Console:** [console.aws.amazon.com/lambda](https://console.aws.amazon.com/lambda/)

**Create a Function (Console):**

1. Click **Create function**
2. **Author from scratch**
3. Function name: `de-validate-incoming-file`
4. Runtime: **Python 3.12**
5. Architecture: **x86_64**
6. Execution role: **Create a new role with basic Lambda permissions**
   - (We'll add S3 permissions after)
7. Click **Create function**

**Add the Code:**

1. In the **Code** tab, you'll see an inline editor
2. Replace the default code with the validation function from the demo above
3. Click **Deploy**

**Add S3 Permissions to the Role:**

1. Click **Configuration** tab → **Permissions**
2. Click the role name (link to IAM)
3. **Add permissions** → **Attach policies**
4. Search for `AmazonS3FullAccess` (for training — use `DE-Pipeline-S3-Access` in production)
5. Attach

**Add S3 Trigger:**

1. Click **Add trigger** (in the Function overview diagram)
2. Source: **S3**
3. Bucket: `de-training-YOUR-NAME`
4. Event type: **All object create events**
5. Prefix: `incoming/` (only trigger for files in the incoming folder)
6. Suffix: `.json`
7. Check the acknowledgement box
8. Click **Add**

**Test the Function:**

1. Click **Test** tab
2. Create a test event — use the S3 event template:
   ```json
   {
     "Records": [{
       "s3": {
         "bucket": {"name": "de-training-YOUR-NAME"},
         "object": {"key": "incoming/calls/date=2026-03-19/calls.json"}
       }
     }]
   }
   ```
3. Click **Test** → see execution results, logs, duration, memory used

**Monitor:**

| Console View | What It Shows |
|---|---|
| **Monitor** tab → **Metrics** | Invocation count, errors, duration, throttles |
| **Monitor** tab → **Logs** | CloudWatch log streams — every `print()` shows here |
| **Monitor** tab → **Traces** | X-Ray traces (if enabled) — see latency breakdown |

> **Teaching point:** The console lets you test Lambda instantly without deploying to S3 or setting up triggers. Use this for rapid iteration — write code, test, see logs, fix, repeat.

#### Lambda: Edge Cases, Gotchas, and Interview Questions

**Gotchas:**

| Gotcha | What Happens | Fix |
|---|---|---|
| Cold start | First invocation takes 1-10 seconds (loading your code) | Use Provisioned Concurrency for latency-sensitive functions |
| 15-minute timeout | Long-running job gets killed | Use Step Functions or EMR for long jobs |
| 250 MB deployment package limit | Can't include large ML models or datasets | Use Lambda Layers or container images (up to 10 GB) |
| S3 trigger fires for EVERY file | 1000 files uploaded = 1000 Lambda invocations | Use SQS as buffer between S3 and Lambda (batch processing) |
| Recursive trigger | Lambda writes to S3, triggers itself, infinite loop | Use different input/output buckets or prefixes |

**Interview questions:**

1. "When would you NOT use Lambda?" — Jobs > 15 minutes (use EMR). Jobs needing > 10 GB memory (use ECS/Fargate). Jobs needing persistent connections (use EC2).
2. "How do you handle Lambda failures?" — Configure DLQ (Dead Letter Queue) on the function. Failed events go to SQS/SNS for investigation.
3. "What's a cold start?" — First invocation downloads your code, initializes the runtime. Subsequent invocations reuse the warm container. Cold starts: 1-10s. Warm: <100ms.

### 2.8 Databases — RDS, DynamoDB, DocumentDB

> **In plain English:** These are the *cash registers* in the store. They handle live transactions — "customer called, order placed, payment processed." Your DE pipeline is the *accounting department* that takes the cash register tapes at the end of the day and builds reports. You read FROM these databases. You don't build your pipeline inside them.

These are **source systems**, not your warehouse. As a DE, you read FROM these databases and load INTO your pipeline.

| Service | Type | DE Role | When You'll Use It |
|---------|------|---------|-------------------|
| **RDS** | Managed relational (PostgreSQL, MySQL, SQL Server) | **Source system.** Your app stores data here. You extract it into S3/warehouse. | CDC from app database, initial full loads |
| **DynamoDB** | Key-value / document NoSQL | **Source system.** Low-latency operational data. DynamoDB Streams = built-in CDC. | Real-time event processing, session data, config lookups |
| **DocumentDB** | MongoDB-compatible | **Source system.** The LT production system uses MongoDB for call + customer data. DocumentDB is the AWS managed equivalent. | When source app uses MongoDB |

#### DynamoDB Streams — Built-in CDC

```
App writes to DynamoDB table
         │
         ▼
DynamoDB Streams captures the change
(INSERT, MODIFY, DELETE — with old and new values)
         │
         ▼
Lambda function reads the stream
         │
    ┌────┴────┐
    ▼         ▼
  S3 (Bronze)  SQS (for batch processing)
```

**Why DynamoDB Streams matters:** It's the cleanest CDC pattern in AWS. No polling, no timestamps to track — the stream gives you every change in order.


### 2.9 Messaging — SQS, SNS, Kinesis, EventBridge

> **In plain English:** Without messaging, your pipeline stages are like a relay race where runners must hand off the baton directly — if the next runner isn't ready, you drop it. With messaging (SQS/Pub/Sub), there's a table between runners. Runner A puts the baton on the table. Runner B picks it up whenever they're ready. If Runner B is sick, the baton waits on the table. If Runner B fails, the baton goes to a "lost and found" (dead letter queue).

These services **decouple** your pipeline stages. Instead of "Step A directly calls Step B," you get "Step A sends a message, Step B picks it up when ready."

#### SQS — Simple Queue Service

```
Producer ──→ [ SQS Queue ] ──→ Consumer
                  │
                  ├── Messages wait until consumer is ready
                  ├── If consumer fails → message goes back to queue
                  └── Dead Letter Queue catches messages that fail repeatedly
```

**DE use case:** Buffer between S3 events and pipeline processing. S3 event → SQS → Lambda reads messages in batches → kicks off EMR job.

#### SNS — Simple Notification Service

```
Publisher ──→ [ SNS Topic ] ──→ Subscriber 1 (SQS queue)
                             ──→ Subscriber 2 (Lambda function)
                             ──→ Subscriber 3 (Email)
                             ──→ Subscriber 4 (HTTP endpoint)
```

**DE use case:** Fan-out notifications. Pipeline completes → SNS → notify Slack + trigger downstream pipeline + update dashboard.

#### Kinesis — Real-Time Streaming

| Component | What It Does |
|-----------|-------------|
| **Kinesis Data Streams** | Ingest and buffer real-time data (like Kafka lite) |
| **Kinesis Data Firehose** | Auto-deliver stream data to S3, Redshift, or Elasticsearch |

**DE use case:** Real-time call events → Kinesis → Firehose → S3 (Bronze) or → Lambda for real-time processing.

#### EventBridge — Event Router

EventBridge is the **smart router**. It receives events and routes them based on rules:

```
S3 event (file created) ──┐
DynamoDB stream ───────────┼──→ [ EventBridge ] ──→ Rule 1: If prefix=calls/ → Lambda A
Custom app event ──────────┘                   ──→ Rule 2: If prefix=orders/ → Step Functions
                                               ──→ Rule 3: If size > 1GB → SNS alert
```


In [ ]:
# ============================================================
# Quick Check: Verify your IAM and storage setup
# ============================================================

# GCP:
# !gcloud iam service-accounts list --filter='email:de-pipeline'
# !gcloud storage ls gs://de-training-YOUR-NAME/

# AWS:
# !aws iam list-roles --query 'Roles[?contains(RoleName,`DE-Pipeline`)].RoleName'
# !aws s3 ls s3://de-training-YOUR-NAME/

# BEGINNER NOTE: If these return empty, you haven't set up
# your cloud resources yet. Go to Section 2.1 (AWS IAM)
# or Section 3.1a (GCP Setup) first.
print('Run the gcloud or aws commands above to verify your setup.')

### 2.10 AWS Event Triggers — Summary Table

How pipelines start automatically:

| Trigger | Source | Target | DE Use Case |
|---------|--------|--------|-------------|
| **S3 Event Notification** | File lands in S3 | Lambda, SQS, SNS | New calls.json → kick off Bronze ingestion |
| **S3 → EventBridge** | File lands in S3 | Step Functions, Lambda, ECS | Complex routing: large file → EMR, small file → Lambda |
| **DynamoDB Streams** | Row insert/update/delete | Lambda | CDC from source database — customer record changes |
| **SQS → Lambda** | Message in queue | Lambda | Process failed records from dead letter queue |
| **Kinesis → Lambda** | Stream event | Lambda | Real-time: process call events as they happen |
| **CloudWatch Events (cron)** | Schedule | Lambda, Step Functions, ECS | Daily 6 AM: trigger full pipeline refresh |
| **SNS → SQS (fan-out)** | Notification | Multiple SQS queues | Pipeline complete → notify monitoring + trigger downstream |
| **Redshift Streaming Ingestion** | Kinesis stream | Redshift | Near-real-time loading into warehouse |

> **Interview tip:** "Walk me through how a file landing in S3 triggers your pipeline." — This is a very common DE interview question. The answer is: S3 event → SQS/Lambda → validate → kick off Spark job → write results → notify downstream.


### 2.11 AWS Service Decision Tree

**"Which service do I use?"**

| I need to... | Use this | Not this |
|---|---|---|
| Store raw data files (data lake) | **S3** | Redshift (warehouse, not a lake) |
| Run PySpark at scale | **EMR** (cluster) or **EMR Serverless** | Lambda (15 min limit, no Spark) |
| Run a quick SQL query on S3 data | **Athena** | Redshift (overkill for ad-hoc) |
| Build a data warehouse for analysts | **Redshift** | S3 (no query engine) or Athena (not for heavy workloads) |
| Discover schemas automatically | **Glue Crawler** | Manual DDL |
| React to a new file in S3 | **Lambda** (triggered by S3 event) | Scheduled cron (misses files between runs) |
| Buffer pipeline events | **SQS** | Direct service-to-service calls (brittle) |
| Send alerts | **SNS** | SQS (point-to-point, not fan-out) |
| Stream data in real-time | **Kinesis** | S3 (batch, not streaming) |
| Route events with rules | **EventBridge** | Lambda (wrong layer — Lambda executes, EventBridge routes) |
| Orchestrate a multi-step pipeline | **Step Functions** (AWS-native) or **MWAA** (Airflow) | Lambda chaining (anti-pattern) |


---
## 3. GCP for Data Engineers

> **You know AWS. Now here's the GCP equivalent.** We won't re-explain every concept — we'll focus on where GCP is the same, where it differs, and where it's genuinely better.

### AWS → GCP Service Mapping

| AWS | GCP | Key Difference |
|-----|-----|----------------|
| S3 | **Cloud Storage (GCS)** | Nearly identical; GCS has stronger consistency model |
| Redshift | **BigQuery** | BigQuery is fully serverless (no clusters to manage), slots vs nodes |
| Athena | **BigQuery** | BigQuery serves both warehouse AND ad-hoc query roles |
| EMR | **Dataproc** | Very similar; Dataproc has faster cluster spin-up (~90 sec) |
| Glue Catalog | **Data Catalog** | GCP's is simpler; BigQuery has built-in schema management |
| Glue ETL | **Dataflow** (Apache Beam) | Different programming model (Beam pipelines vs Spark jobs) |
| Lambda | **Cloud Functions** | Nearly identical — same triggers, same limits |
| RDS | **Cloud SQL** | Nearly identical (PostgreSQL, MySQL, SQL Server) |
| DynamoDB | **Firestore / Bigtable** | Firestore = document DB, Bigtable = wide-column (HBase API) |
| DocumentDB | **Firestore** | Firestore covers MongoDB-style document use cases |
| SQS | **Cloud Tasks / Pub/Sub** | Pub/Sub is more versatile than SQS — handles both queue and topic patterns |
| SNS | **Pub/Sub** | Pub/Sub = SNS + SQS combined in one service |
| Kinesis | **Pub/Sub + Dataflow** | Pub/Sub for ingestion, Dataflow for stream processing |
| EventBridge | **Eventarc** | Routes events from GCS, Pub/Sub to Cloud Run/Functions |
| Step Functions | **Workflows** | State machine orchestration |
| MWAA | **Cloud Composer** | Both are managed Airflow |
| Lake Formation | **Dataplex** | Data governance and access control |


### BigQuery — The Standout

> **In plain English:** BigQuery is like a self-driving car. Redshift is a sports car you drive yourself — powerful but you manage the engine, fuel, and parking. BigQuery is fully autonomous: you type the destination (SQL query), it figures out the fastest route (query plan), drives there (executes), and charges by the mile ($5/TB scanned). No keys, no garage, no maintenance.

BigQuery is arguably GCP's best service and the main reason some companies choose GCP for data workloads. Key differences from Redshift:

| Feature | Redshift | BigQuery |
|---------|----------|----------|
| **Infrastructure** | You manage clusters (or use Serverless mode) | Fully serverless — always |
| **Scaling** | Resize cluster, add nodes | Automatic — BigQuery allocates slots |
| **Pricing** | Per node-hour ($0.25–$13.04/hr per node) | Per TB scanned ($5/TB on-demand) OR flat-rate slots |
| **Storage** | On nodes (RA3 = separate managed storage) | Fully separated, $0.02/GB/month |
| **Semi-structured data** | `SUPER` type (JSON) | Native `STRUCT`, `ARRAY` — first-class JSON-like querying |
| **Nested data** | Flatten before loading (usually) | Query nested structures directly: `SELECT customer.address.city` |
| **Partitioning** | Sort keys + distribution keys (you choose) | Time-based or integer-range partitioning (simpler) |
| **Clustering** | Sort keys | Up to 4 clustering columns — auto-reorganized |
| **ML** | Export data → SageMaker | `CREATE MODEL` directly in SQL (BigQuery ML) |

> **The BigQuery "aha" moment:** You can query 1 TB of data, get results in seconds, and it costs $5. No cluster to provision, no nodes to tune, no auto-scaling to configure. That's the value proposition.


### BigQuery Data Loading Commands

| Command | What It Does | When to Use |
|---------|-------------|-------------|
| `bq load` | CLI bulk load from GCS or local file | Batch loading from command line |
| `LOAD DATA` | SQL-based load from GCS → table | Load within a SQL workflow |
| `MERGE INTO` | Upsert (insert + update + delete) | CDC, late-arriving data, SCD Type 2 |
| `CREATE EXTERNAL TABLE` | Query GCS without loading (federated) | Bronze layer access — lakehouse pattern |
| `EXPORT DATA` | Table → GCS (Parquet, CSV, JSON, Avro) | Data lake export |
| BigQuery Data Transfer | Scheduled loads from S3, GCS, SaaS apps | Cross-cloud, recurring loads |
| Pub/Sub → BigQuery Subscription | Stream directly from Pub/Sub → BigQuery | Real-time ingestion |
| `INSERT INTO ... SELECT` | Transform within BigQuery | Silver → Gold transforms |


#### Hello World: BigQuery

Before LOAD DATA, MERGE, and external tables — let's just run one query:


In [ ]:
# ============================================================
# Hello World: BigQuery — query a public dataset (no setup!)
# ============================================================

hello_bq = """
-- In BigQuery Console -> Query Editor, run this:
-- (No dataset setup needed — BigQuery has public datasets)

-- 1. Query a public dataset (free for first 1TB/month)
SELECT
    name,
    SUM(number) as total
FROM `bigquery-public-data.usa_names.usa_1910_current`
WHERE year > 2000
GROUP BY name
ORDER BY total DESC
LIMIT 10;

-- Check the bottom bar: 'X bytes processed' — this is what you pay for.
-- Partition pruning + column pruning = less bytes = less cost.

-- 2. Create your own dataset and table
CREATE SCHEMA IF NOT EXISTS hello_test;
CREATE TABLE hello_test.greetings (message STRING, created_at TIMESTAMP);
INSERT INTO hello_test.greetings VALUES ('Hello from BigQuery!', CURRENT_TIMESTAMP());
SELECT * FROM hello_test.greetings;

-- 3. Cleanup
DROP SCHEMA hello_test CASCADE;
"""
print(hello_bq)

In [ ]:
# ============================================================
# Demo: BigQuery loading and querying
# ============================================================

bq_commands = '''
# Load JSON from GCS into BigQuery
bq load \
    --source_format=NEWLINE_DELIMITED_JSON \
    --autodetect \
    de_training.bronze_calls \
    gs://de-training-bronze/calls/date=2026-03-19/calls.json

# Query with partition pruning
bq query --use_legacy_sql=false '
SELECT
    disposition,
    COUNT(*) as call_count,
    AVG(duration) as avg_duration,
    SUM(CASE WHEN payment_status = "completed" THEN 1 ELSE 0 END) as conversions
FROM de_training.silver_calls
WHERE DATE(start_time) = "2026-03-19"
GROUP BY disposition
ORDER BY call_count DESC
'

# Query nested JSON (BigQuery handles this natively)
bq query --use_legacy_sql=false '
SELECT
    call_id,
    customer.first_name,
    customer.address.city,
    customer.address.state
FROM de_training.bronze_calls
WHERE customer.address.state = "TX"
'

# MERGE (upsert)
bq query --use_legacy_sql=false '
MERGE de_training.silver_calls AS target
USING de_training.staging_calls AS source
ON target.call_id = source.call_id
WHEN MATCHED THEN UPDATE SET
    disposition = source.disposition,
    payment_status = source.payment_status
WHEN NOT MATCHED THEN INSERT ROW
'
'''
print(bq_commands)

#### Console Walkthrough: BigQuery

> **Open the BigQuery Console:** [console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)

**Create a Dataset (Console):**

1. In the Explorer panel (left), click the **three dots** next to your project name
2. Click **Create dataset**
3. Dataset ID: `de_training`
4. Data location: `US`
5. Click **Create dataset**

**Load Data (Console):**

1. Click into `de_training` dataset → click **+** (Create table)
2. Source: **Google Cloud Storage**
3. GCS URI: `gs://de-training-YOUR-NAME/bronze/calls/*.json`
4. File format: **JSON (Newline delimited)**
5. Table name: `bronze_calls`
6. Schema: **Auto detect** (check the box)
7. Partitioning: **Partition by field** → select `date` → **DAY**
8. Click **Create table**

**Query Editor:**

The BigQuery console has a built-in SQL editor — no external tools needed:

1. Click **+ Compose new query** (top)
2. Write SQL directly:
   ```sql
   SELECT disposition, COUNT(*) as cnt
   FROM de_training.bronze_calls
   WHERE date = '2026-03-19'
   GROUP BY disposition
   ```
3. Click **Run** → results appear below
4. Check the bottom bar: **bytes processed** (this is what you pay for)

**Key Console Features:**

| Feature | Where | Why DEs Care |
|---|---|---|
| **Query history** | Left panel → **Job history** | See all queries, cost per query, who ran what |
| **Execution details** | Click a completed query → **Execution details** tab | See stages, slots used, bytes shuffled — equivalent to Spark UI |
| **Schema** tab | Click any table | See columns, types, partitioning, clustering |
| **Preview** tab | Click any table → **Preview** | See sample data without running a query (free) |
| **Details** tab | Click any table → **Details** | Row count, size, last modified, partition info |
| **Scheduled queries** | Top menu → **Scheduled queries** | Set up recurring SQL (simple orchestration) |

> **Teaching point:** BigQuery's console is one of the best in cloud. You can explore data, write SQL, see execution plans, and monitor costs — all without leaving the browser. Redshift requires Query Editor v2 (separate tool). Snowflake has Snowsight (comparable).

#### BigQuery: Edge Cases, Gotchas, and Interview Questions

**Gotchas:**

| Gotcha | What Happens | Fix |
|---|---|---|
| `SELECT *` on a large table | Scans all columns — expensive ($5/TB) | Always select specific columns |
| No partition filter in WHERE clause | Scans entire table | Add `WHERE date = ...` to trigger pruning. Set `require_partition_filter = true` on the table. |
| Streaming inserts into partitioned table | Data in streaming buffer isn't prunable for ~30 min | Use batch loading (LOAD DATA) for cost-sensitive pipelines |
| Query results exceed 128MB (flat response) | Query fails or truncates | Write results to a destination table instead of returning inline |
| Nested STRUCT/ARRAY is confusing | JOINs on nested fields behave differently | Use UNNEST() to flatten before joining |

**Interview questions:**

1. "How does BigQuery pricing work?" — On-demand: $5/TB scanned. Flat-rate: buy slots (compute units) for predictable cost. Storage: $0.02/GB/month (active), $0.01/GB (long-term after 90 days).
2. "What's the difference between partitioning and clustering?" — Partitioning divides the table into segments (by date). Clustering sorts data within each partition (by specified columns). Both reduce bytes scanned.
3. "How do you optimize BigQuery costs?" — Partition + cluster tables. Select only needed columns. Set partition filters as required. Use Preview (free) instead of SELECT * for exploration.
4. "BigQuery vs Redshift?" — BigQuery: fully serverless, pay-per-query, great for ad-hoc. Redshift: provisioned clusters (or Serverless), better for predictable high-concurrency workloads.

### GCP Event Triggers

| Trigger | Source | Target | DE Use Case |
|---------|--------|--------|-------------|
| **GCS → Eventarc** | File lands in GCS | Cloud Functions, Cloud Run | New data file → validate and move to Bronze |
| **GCS → Pub/Sub** | File lands in GCS | Pub/Sub → Dataflow, Cloud Functions | Fan-out: file arrival triggers multiple consumers |
| **Pub/Sub → Cloud Functions** | Message published | Cloud Functions | Process individual records, dead letter handling |
| **Pub/Sub → Dataflow** | Stream message | Dataflow (Apache Beam) | Real-time stream processing |
| **Cloud Scheduler (cron)** | Schedule | Cloud Functions, Pub/Sub, HTTP | Daily pipeline trigger |
| **Firestore triggers** | Document create/update/delete | Cloud Functions | CDC from source database |
| **BigQuery → Pub/Sub** | Query complete, table update | Pub/Sub → downstream | Chain stages: Gold mart refresh → notify |

> **Key takeaway:** GCP uses **Pub/Sub + Eventarc** where AWS uses **SQS + SNS + EventBridge**. GCP is simpler (fewer services) but less flexible in routing. For most DE pipelines, both work equally well.


#### Console Walkthrough: GCP Event Triggers

> **Open Eventarc:** [console.cloud.google.com/eventarc](https://console.cloud.google.com/eventarc)

**Create a GCS → Cloud Function Trigger (Console):**

1. Go to **Cloud Functions** console → Click **Create function**
2. Function name: `validate-incoming-file`
3. Region: `us-central1`
4. Trigger type: **Cloud Storage**
5. Event type: **google.cloud.storage.object.v1.finalized** (file created)
6. Bucket: `de-training-YOUR-NAME`
7. Click **Next** → paste your Python validation code → **Deploy**

**Create a Cloud Scheduler Job (Console):**

1. Go to **Cloud Scheduler** → **Create job**
2. Name: `daily-pipeline-trigger`
3. Frequency: `0 6 * * *` (6 AM daily)
4. Timezone: `America/New_York`
5. Target type: **Pub/Sub**
6. Topic: `de-pipeline-trigger`
7. Message body: `{"action": "run_daily_pipeline"}`
8. Click **Create**

> **Teaching point:** GCP's event model is simpler than AWS — Pub/Sub handles most of what SQS + SNS + EventBridge do in AWS. Fewer services to learn, but also less granular routing control.

### 3.1a GCP Setup from Scratch

> **If this is your first time on GCP, follow these steps exactly.** If you already have a GCP project with billing enabled, skip to Phase 3.

---

#### Phase 1: Create a GCP Account

1. Go to [cloud.google.com](https://cloud.google.com)
2. Click **Get Started for Free**
3. Sign in with your Google account
4. Enter billing details (you get **$300 free credits** — more than enough for this training)
5. Click **Start my free trial**

#### Phase 2: Create a New Project

1. Go to [console.cloud.google.com](https://console.cloud.google.com)
2. Click the **project dropdown** at the top of the page
3. Click **New Project**
4. Enter:
   - Project name: `de-training`
   - Project ID: auto-generated (you can customize — **it's permanent**)
5. Click **Create**
6. Wait a few seconds, then **select the new project** from the dropdown

**Link Billing:**

1. Go to **Billing** in the left menu
2. Click **Link a billing account**
3. Select your billing account → **Set Account**

#### Phase 3: Open Cloud Shell

1. In the GCP Console, look at the **top-right toolbar**
2. Click the **terminal icon** `>_` (next to the search bar)
3. A terminal opens at the bottom of the screen
4. Wait for it to initialize — ready when you see:

```
your-email@cloudshell:~ (de-training)$
```

> **Cloud Shell is free.** It's a Linux VM with `gcloud`, `gcloud storage`, `bq`, Python, and Git pre-installed. You don't need to install anything on your laptop.

#### Phase 4: Authenticate

> **You must authenticate before any `gcloud` command will work.**

In [ ]:
# ============================================================
# GCP Authentication — run these in Cloud Shell
# ============================================================

gcp_auth = """
# Step 1: Login to gcloud
gcloud auth login

# What happens:
# 1. A URL appears in the terminal
# 2. Click it (or copy-paste into browser)
# 3. Sign in with your Google account
# 4. Click 'Allow'
# 5. Copy the verification code
# 6. Paste it back in Cloud Shell, press Enter
#
# You should see: 'You are now logged in as [your-email@gmail.com]'

# Step 2: Set Application Default Credentials
# (needed for services like Cloud Storage, BigQuery, Dataproc)
gcloud auth application-default login
# Same flow as above — click URL, sign in, paste code
"""
print(gcp_auth)

In [ ]:
# ============================================================
# GCP Project & Region Configuration
# ============================================================

gcp_config = """
# Set your project (replace with your actual project ID)
gcloud config set project de-training

# Set region (us-central1 is cheapest for most services)
gcloud config set compute/region us-central1
gcloud config set compute/zone us-central1-a

# Verify everything
gcloud config list

# Expected output:
# [compute]
# region = us-central1
# zone = us-central1-a
# [core]
# account = your-email@gmail.com
# project = de-training
"""
print(gcp_config)

#### Phase 5: Enable Required APIs

> **This is the #1 gotcha for GCP beginners.** Services don't work until you explicitly enable their API. If you skip this step, every command will fail with `API not enabled`.


In [ ]:
# ============================================================
# Enable ALL APIs needed for the DE pipeline
# ============================================================

enable_apis = """
gcloud services enable \\
  storage.googleapis.com \\
  bigquery.googleapis.com \\
  dataproc.googleapis.com \\
  pubsub.googleapis.com \\
  cloudfunctions.googleapis.com \\
  cloudbuild.googleapis.com \\
  cloudresourcemanager.googleapis.com \\
  iam.googleapis.com \\
  composer.googleapis.com

# Wait for all APIs to enable (takes 30-60 seconds)

# Verify
gcloud services list --enabled | grep -E 'bigquery|dataproc|storage|pubsub|functions|composer|iam'

# You should see all 9 services listed as ENABLED.
# If any are missing, re-run the enable command for that specific API.
"""
print(enable_apis)
print()
print('IMPORTANT: If you skip this step, you will see errors like:')
print('  "API [dataproc.googleapis.com] not enabled on project"')
print('  "Permission denied" (even with correct IAM roles)')
print('  "Service not available"')
print()
print('Always enable APIs FIRST, before creating any resources.')

#### Quick Reference: Replace These Placeholders

Throughout the GCP sections, replace these placeholders with your actual values:

| Placeholder | Replace With | Example |
|---|---|---|
| `de-training` or `your-project-id` | Your GCP project ID | `de-training-2026` |
| `de-training-YOUR-NAME` | Your GCS bucket name (globally unique) | `de-training-sunil` |
| `us-central1` | Your chosen region | `us-central1` (recommended) |
| `de-pipeline-sa` | Your service account name | `de-pipeline-sa` |
| `de_training` | Your BigQuery dataset name | `de_training` |

> **Tip:** Run `gcloud config get-value project` anytime to check which project you're in. Wrong project = wrong resources = confusing errors.

#### Phase 6: Create BigQuery Dataset & Verify

> **Do this now — you'll need the dataset ready when you build the Gold layer.**

In [ ]:
# ============================================================
# BigQuery Setup — create dataset and schemas from Cloud Shell
# ============================================================

bq_setup = """
# Create the dataset
bq --location=us-central1 mk \\
  --dataset \\
  --description "DE Training Pipeline" \\
  de-training:de_training

# Verify
bq ls

# Create schemas for Bronze, Silver, Gold
# (BigQuery doesn't have schemas like Redshift — we use datasets or table prefixes)
# Option 1: Separate datasets
bq mk --dataset de-training:bronze
bq mk --dataset de-training:silver
bq mk --dataset de-training:gold

# Verify all datasets
bq ls

# Expected output:
#   datasetId
#  -----------
#   bronze
#   de_training
#   gold
#   silver

# Useful BigQuery shell commands:
bq ls de-training:gold              # list tables in gold dataset
bq show de-training:gold.mart_daily_campaign  # show table schema & row count
bq rm -f -t de-training:gold.mart_daily_campaign  # drop a table
bq rm -r -f -d de-training:gold     # drop entire dataset + all tables

# Query from shell (no console needed)
bq query --use_legacy_sql=false \\
  'SELECT COUNT(*) as total FROM \`de-training.gold.mart_daily_campaign\`'
"""
print(bq_setup)

#### Phase 7: Pub/Sub + Cloud Functions — Event-Driven Pipeline Trigger

> **This sets up the GCP equivalent of S3 Event Notification → Lambda.** When a file lands in GCS, Pub/Sub sends a message, Cloud Function triggers the pipeline.

In [ ]:
# ============================================================
# Pub/Sub + Cloud Functions — complete setup from scratch
# ============================================================

pubsub_setup = """
# Step 1: Create Pub/Sub topic
gcloud pubsub topics create gcs-upload-trigger

# Step 2: Attach topic to GCS bucket (file arrival notifications)
gcloud storage buckets notifications create gs://de-training-YOUR-NAME \\
  --topic=gcs-upload-trigger \\
  --event-types=OBJECT_FINALIZE

# Verify
gcloud pubsub topics list
gcloud storage buckets notifications list gs://de-training-YOUR-NAME
"""

cloud_function_setup = """
# Step 3: Create Cloud Function directory
mkdir -p ~/pipeline-function
cd ~/pipeline-function

# Step 4: Create main.py (file validation + pipeline trigger)
cat > main.py << 'PYEOF'
import functions_framework
from google.cloud import storage
import json

client = storage.Client()

@functions_framework.cloud_event
def validate_and_trigger(cloud_event):
    data = cloud_event.data
    bucket_name = data['bucket']
    file_name = data['name']
    print(f'File uploaded: gs://{bucket_name}/{file_name}')

    # Only process files in incoming/ folder
    if not file_name.startswith('incoming/'):
        print('Not an incoming file, skipping.')
        return

    # Validate the file
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(file_name)
    content = blob.download_as_text()
    lines = content.strip().split('\n')

    valid = sum(1 for l in lines if _validate(l))
    invalid = len(lines) - valid
    print(f'Valid: {valid}, Invalid: {invalid}')

    # Move to bronze/ or quarantine/
    if invalid / max(len(lines), 1) > 0.1:
        dest = file_name.replace('incoming/', 'quarantine/')
    else:
        dest = file_name.replace('incoming/', 'bronze/')

    bucket.copy_blob(blob, bucket, dest)
    blob.delete()
    print(f'Moved to: {dest}')

def _validate(line):
    try:
        record = json.loads(line)
        return all(k in record for k in ['call_id', 'dnis', 'duration'])
    except json.JSONDecodeError:
        return False
PYEOF

# Step 5: Create requirements.txt
cat > requirements.txt << 'EOF'
functions-framework==3.*
google-cloud-storage==2.*
EOF

# Step 6: Deploy the Cloud Function
gcloud functions deploy validate-incoming-file \\
  --gen2 \\
  --runtime=python312 \\
  --region=us-central1 \\
  --source=. \\
  --entry-point=validate_and_trigger \\
  --trigger-topic=gcs-upload-trigger \\
  --service-account=de-pipeline-sa@de-training.iam.gserviceaccount.com

# Wait 1-2 minutes. Verify:
gcloud functions list
# Expected: validate-incoming-file  ACTIVE  pubsub  us-central1
"""

test_trigger = """
# Step 7: Test it!
cd ~

# Upload a test file to incoming/
gcloud storage cp data/calls.json gs://de-training-YOUR-NAME/incoming/calls/date=2026-03-19/

# Check Cloud Function logs (wait 30 seconds)
gcloud functions logs read validate-incoming-file --region=us-central1 --limit=10

# You should see:
#   File uploaded: gs://de-training-YOUR-NAME/incoming/calls/date=2026-03-19/calls.json
#   Valid: 510, Invalid: 0
#   Moved to: bronze/calls/date=2026-03-19/calls.json

# Verify the file moved to bronze/
gcloud storage ls gs://de-training-YOUR-NAME/bronze/calls/
"""

print('--- Pub/Sub Setup ---')
print(pubsub_setup)
print('--- Cloud Function Setup ---')
print(cloud_function_setup)
print('--- Test the Trigger ---')
print(test_trigger)

#### Phase 8: GCP Monitoring Commands

Keep these handy — you'll use them throughout the labs:

```bash
# --- Cloud Functions ---
gcloud functions logs read FUNCTION_NAME --region=us-central1 --limit=50
gcloud functions describe FUNCTION_NAME --region=us-central1

# --- Dataproc ---
gcloud dataproc jobs list --region=us-central1
gcloud dataproc jobs describe JOB_ID --region=us-central1
gcloud dataproc clusters list --region=us-central1

# --- Pub/Sub ---
gcloud pubsub topics list
gcloud pubsub subscriptions list
gcloud pubsub subscriptions pull SUB_NAME --auto-ack --limit=5

# --- BigQuery ---
bq ls                                    # list datasets
bq ls PROJECT:DATASET                    # list tables
bq show PROJECT:DATASET.TABLE            # table schema + row count
bq query --use_legacy_sql=false 'SQL'    # run a query
bq rm -f -t PROJECT:DATASET.TABLE        # drop a table
bq rm -r -f -d PROJECT:DATASET           # drop dataset + all tables

# --- GCS ---
gcloud storage ls gs://BUCKET/           # list contents
gcloud storage ls -r gs://BUCKET/        # recursive list
gcloud storage cp LOCAL gs://BUCKET/PATH # upload
gcloud storage cp gs://BUCKET/PATH LOCAL # download
gcloud storage rm gs://BUCKET/PATH       # delete

# --- Billing (check costs!) ---
gcloud billing accounts list
gcloud billing projects describe PROJECT_ID
```

> **Bookmark this cell.** You'll come back to these commands every day.

### 3.2 GCP IAM — Identity and Access Management

> **Console:** [console.cloud.google.com/iam-admin](https://console.cloud.google.com/iam-admin)

GCP IAM uses the same concepts as AWS but with different terminology:

| AWS | GCP | Key Difference |
|-----|-----|----------------|
| User | Principal (user, service account, group) | GCP uses Google accounts or service accounts |
| Role (assume temporarily) | Role (bind to principal) | GCP roles are bound, not assumed |
| Policy (JSON document) | IAM binding (role + principal + resource) | GCP binds at the resource level |
| Access keys | Service account key (JSON) | Same idea — avoid both in production |
| Instance profile (EC2/EMR) | Service account (attached to VM/cluster) | Same pattern — service identity |

**Console Walkthrough:**

1. **Create a Service Account** (equivalent of AWS IAM Role for services):
   - IAM Console → **Service Accounts** → **Create Service Account**
   - Name: `de-pipeline-sa`
   - Description: "Service account for DE pipeline (Dataproc, Cloud Functions)"
   - Click **Create and Continue**
   - Grant roles:
     - `Storage Object Viewer` (read Bronze bucket)
     - `Storage Object Creator` (write Silver bucket)
     - `BigQuery Data Editor` (write to Gold marts)
     - `Dataproc Worker` (run Spark jobs)
   - Click **Done**

2. **Verify:** IAM Console → **Service Accounts** → click `de-pipeline-sa` → **Permissions** tab

In [ ]:
# ============================================================
# Demo: GCP IAM — Create service account via gcloud CLI
# ============================================================

gcp_iam = """
# Create a service account (equivalent of AWS IAM Role)
gcloud iam service-accounts create de-pipeline-sa \\
    --display-name="DE Pipeline Service Account" \\
    --description="Service account for Bronze/Silver/Gold pipeline"

# Grant roles to the service account
PROJECT_ID=$(gcloud config get-value project)

# Read Bronze bucket
gcloud projects add-iam-policy-binding $PROJECT_ID \\
    --member="serviceAccount:de-pipeline-sa@$PROJECT_ID.iam.gserviceaccount.com" \\
    --role="roles/storage.objectViewer"

# Write Silver bucket
gcloud projects add-iam-policy-binding $PROJECT_ID \\
    --member="serviceAccount:de-pipeline-sa@$PROJECT_ID.iam.gserviceaccount.com" \\
    --role="roles/storage.objectCreator"

# BigQuery access for Gold marts
gcloud projects add-iam-policy-binding $PROJECT_ID \\
    --member="serviceAccount:de-pipeline-sa@$PROJECT_ID.iam.gserviceaccount.com" \\
    --role="roles/bigquery.dataEditor"

# Verify
gcloud iam service-accounts list --filter="email:de-pipeline-sa"
"""
print(gcp_iam)

# **You should see:** `Created service account [de-pipeline-sa]`. Verify with `gcloud iam service-accounts list` — your service account should appear. If you see `Permission denied`, check that your user has the `iam.serviceAccounts.create` permission.

### 3.3 Cloud Storage (GCS) — Equivalent of S3

> **Console:** [console.cloud.google.com/storage](https://console.cloud.google.com/storage)

**Console Walkthrough:**

1. Click **Create bucket**
2. Name: `de-training-YOUR-NAME` (globally unique)
3. Location: `us-central1` (single region — cheapest)
4. Storage class: **Standard**
5. Access control: **Uniform** (recommended)
6. Click **Create**

**Upload with partition structure:**

1. Click into your bucket
2. **Create folder** → `bronze` → `calls` → `date=2026-03-19`
3. Click into the folder → **Upload files** → select `calls.json`

**Set Lifecycle Rule (Console):**

1. Click bucket → **Lifecycle** tab → **Add a rule**
2. Action: **Set storage class to Nearline**
3. Condition: Age > **30 days**
4. Add another rule: **Delete** when age > **365 days**
5. Click **Save**

In [ ]:
# ============================================================
# Demo: GCS — Create bucket, upload data, list contents
# ============================================================

gcs_demo = """
# Create a bucket
gcloud storage buckets create -l us-central1 gs://de-training-YOUR-NAME

# Upload with partition structure
gcloud storage cp data/calls.json gs://de-training-YOUR-NAME/bronze/calls/date=2026-03-19/
gcloud storage cp data/orders.csv gs://de-training-YOUR-NAME/bronze/orders/date=2026-03-19/
gcloud storage cp data/products.csv gs://de-training-YOUR-NAME/bronze/products/
gcloud storage cp data/campaigns.csv gs://de-training-YOUR-NAME/bronze/campaigns/

# List contents
gcloud storage ls gs://de-training-YOUR-NAME/bronze/

# List recursively with sizes
gcloud storage ls -l -r gs://de-training-YOUR-NAME/bronze/

# Set lifecycle policy (JSON file)
gcloud storage buckets update --lifecycle-file= lifecycle.json gs://de-training-YOUR-NAME

# Set up GCS event notification -> Pub/Sub
gcloud storage buckets notifications create -t de-pipeline-trigger -f json \\
    -e OBJECT_FINALIZE gs://de-training-YOUR-NAME
"""
print(gcs_demo)

# **You should see:** `Creating gs://de-training-YOUR-NAME/...`. Verify in the GCS console — your bucket should appear. If `ServiceException: 409`, the bucket name is already taken globally.

### 3.4 Dataproc — Managed Spark (Equivalent of EMR)

> **Console:** [console.cloud.google.com/dataproc](https://console.cloud.google.com/dataproc)

**Console Walkthrough — Create a Cluster:**

1. Click **Create cluster** → **Cluster on Compute Engine**
2. Name: `de-training-pipeline`
3. Region: `us-central1`
4. Cluster type: **Standard** (1 master, N workers)
5. **Master node:** `n1-standard-4` (4 vCPU, 15 GB)
6. **Worker nodes:** `n1-standard-4` x 2
7. **Preemptible workers** (equivalent of Spot): add 2 → **70-80% cheaper**
8. **Autoscaling:** Enable → min 2, max 6 workers
9. **Scheduled deletion:** Enable → delete after **1 hour idle** (don't forget!)
10. Click **Create** (takes ~90 seconds — faster than EMR)

**Submit a Job (Console):**

1. Go to **Jobs** tab → **Submit job**
2. Cluster: `de-training-pipeline`
3. Job type: **PySpark**
4. Main Python file: `gs://de-training-code/silver_transform.py`
5. Click **Submit**
6. Monitor: click the job → see output logs in real-time

**Dataproc vs EMR:**

| Feature | EMR | Dataproc |
|---|---|---|
| Spin-up time | 5-15 min | ~90 seconds |
| Spot/Preemptible | Spot instances (bid price) | Preemptible VMs (fixed 60-91% discount) |
| Auto-terminate | Set idle timeout | Scheduled deletion |
| Notebook | EMR Notebooks (JupyterHub) | Built-in Jupyter (optional component) |
| Pricing | Per-instance-hour | Per-instance-hour (similar rates) |

In [ ]:
# ============================================================
# Demo: Dataproc — Create cluster and submit PySpark job
# ============================================================

dataproc_demo = """
# Create a Dataproc cluster
gcloud dataproc clusters create de-training-pipeline \\
    --region=us-central1 \\
    --master-machine-type=n1-standard-4 \\
    --num-workers=2 \\
    --worker-machine-type=n1-standard-4 \\
    --num-secondary-workers=2 \\
    --secondary-worker-type=preemptible \\
    --max-idle=1h \\
    --optional-components=JUPYTER

# Submit a PySpark job
gcloud dataproc jobs submit pyspark \\
    gs://de-training-code/silver_transform.py \\
    --cluster=de-training-pipeline \\
    --region=us-central1

# Check job status
gcloud dataproc jobs list --region=us-central1 --cluster=de-training-pipeline

# Delete cluster when done
gcloud dataproc clusters delete de-training-pipeline --region=us-central1 --quiet
"""
print(dataproc_demo)
print()
print('Key difference from EMR:')
print('  - Dataproc spins up in ~90 seconds (EMR: 5-15 min)')
print('  - --max-idle=1h auto-deletes (EMR: separate auto-termination)')
print('  - Preemptible VMs have fixed discount (Spot: variable bidding)')

# **You should see:** Cluster creation progress, then `Created [... de-training-pipeline]` after ~90 seconds. Verify in the Dataproc console — status should be `Running`. If it fails with quota errors, check that your project has Compute Engine API enabled and sufficient quota.

### 3.5 Cloud Functions — Equivalent of Lambda

> **Console:** [console.cloud.google.com/functions](https://console.cloud.google.com/functions)

**Console Walkthrough:**

1. Click **Create function**
2. Function name: `validate-incoming-file`
3. Region: `us-central1`
4. Trigger type: **Cloud Storage**
5. Event type: `google.cloud.storage.object.v1.finalized` (file created)
6. Bucket: `de-training-YOUR-NAME`
7. Click **Next**
8. Runtime: **Python 3.12**
9. Entry point: `handler`
10. Paste validation code (same logic as AWS Lambda demo)
11. Click **Deploy**

**Monitor:**

1. Click the function → **Logs** tab → see every invocation
2. **Metrics** tab → invocation count, latency, errors
3. **Testing** tab → send a test event directly from console

In [ ]:
# ============================================================
# Demo: Cloud Functions — Deploy via gcloud CLI
# ============================================================

cloud_func_demo = """
# Deploy a Cloud Function triggered by GCS file upload
gcloud functions deploy validate-incoming-file \\
    --runtime=python312 \\
    --trigger-resource=de-training-YOUR-NAME \\
    --trigger-event=google.storage.object.finalize \\
    --source=./cloud_functions/validate/ \\
    --entry-point=handler \\
    --region=us-central1 \\
    --service-account=de-pipeline-sa@PROJECT_ID.iam.gserviceaccount.com

# Test by uploading a file
gcloud storage cp test_calls.json gs://de-training-YOUR-NAME/incoming/calls/

# Check logs
gcloud functions logs read validate-incoming-file --region=us-central1 --limit=10

# View function details
gcloud functions describe validate-incoming-file --region=us-central1
"""
print(cloud_func_demo)

### 3.6 Pub/Sub — Equivalent of SQS + SNS

> **Console:** [console.cloud.google.com/cloudpubsub](https://console.cloud.google.com/cloudpubsub)

Pub/Sub replaces **both** SQS (message queue) and SNS (pub/sub notifications) in one service.

| AWS | GCP Pub/Sub |
|-----|-------------|
| SNS Topic → multiple subscribers | Pub/Sub Topic → multiple Subscriptions |
| SQS Queue (pull) | Pub/Sub Subscription (pull or push) |
| SQS Dead Letter Queue | Pub/Sub Dead Letter Topic |
| SNS → SQS fan-out | One Topic → multiple Subscriptions (built-in) |

**Console Walkthrough:**

1. Click **Create topic** → Topic ID: `de-pipeline-events` → Create
2. Click into the topic → **Create subscription**
3. Subscription ID: `de-pipeline-worker`
4. Delivery type: **Pull**
5. Message retention: **7 days**
6. Dead letter topic: create `de-pipeline-dlq`
7. Click **Create**

**Test:** Click the topic → **Publish message** → add body → **Publish** → check subscription

In [ ]:
# ============================================================
# Demo: Pub/Sub — Create topic, subscribe, publish, pull
# ============================================================

pubsub_demo = """
# Create a topic
gcloud pubsub topics create de-pipeline-events

# Create a subscription
gcloud pubsub subscriptions create de-pipeline-worker \\
    --topic=de-pipeline-events \\
    --ack-deadline=60 \\
    --message-retention-duration=7d

# Create a dead letter topic
gcloud pubsub topics create de-pipeline-dlq

# Publish a message
gcloud pubsub topics publish de-pipeline-events \\
    --message='{"action": "process_new_calls", "date": "2026-03-19"}'

# Pull messages
gcloud pubsub subscriptions pull de-pipeline-worker --auto-ack --limit=5

# Set up GCS -> Pub/Sub notification
gcloud storage buckets notifications create -t de-pipeline-events -f json \\
    -e OBJECT_FINALIZE gs://de-training-YOUR-NAME
"""
print(pubsub_demo)
print()
print('GCS file arrival -> Pub/Sub message -> your pipeline reacts')
print('This is the GCP equivalent of: S3 event -> SQS -> Lambda')

### 3.7 Cloud SQL & Firestore — Equivalent of RDS & DynamoDB

These are **source systems** your pipeline reads from:

| AWS | GCP | Console |
|-----|-----|--------|
| RDS (PostgreSQL, MySQL) | **Cloud SQL** | [console.cloud.google.com/sql](https://console.cloud.google.com/sql) |
| DynamoDB (key-value/document) | **Firestore** | [console.cloud.google.com/firestore](https://console.cloud.google.com/firestore) |
| DynamoDB Streams (CDC) | **Firestore triggers** → Cloud Functions | Built-in |
| DocumentDB (MongoDB) | **Firestore** (document mode) | Same console |

**Console Walkthrough — Cloud SQL:**

1. Click **Create instance** → **PostgreSQL**
2. Instance ID: `de-training-source-db`
3. Password: set admin password
4. Region: `us-central1`
5. Machine type: **Lightweight** (1 vCPU, 3.75 GB — cheapest)
6. Storage: 10 GB SSD
7. Click **Create instance** (takes 3-5 minutes)
8. Click the instance → **Databases** → **Create database** → `call_center`

> **Teaching point:** Cloud SQL is where your app stores operational data. As a DE, you extract FROM Cloud SQL INTO your pipeline (GCS → Bronze → Silver → Gold).

### 3.8 Dataflow — Streaming + Batch ETL (Apache Beam)

> **In plain English:** Dataproc is "rent a Spark cluster." Dataflow is "give me your code, I'll figure out the infrastructure." You never see a cluster. You never configure workers. You submit a pipeline definition (Apache Beam) and Dataflow auto-scales workers to match the data volume.

#### When Dataflow vs Dataproc

| Factor | Dataproc (Spark) | Dataflow (Beam) |
|---|---|---|
| **You already know** | PySpark | Need to learn Beam API |
| **Best for** | Batch ETL, complex joins, Delta Lake/Iceberg | Streaming pipelines, simple batch transforms |
| **Infrastructure** | Create/manage clusters (or Serverless) | Fully managed — zero cluster config |
| **Streaming** | Spark Structured Streaming (OK) | Native streaming — Beam was built for this |
| **Auto-scaling** | You configure auto-scale rules | Automatic — Dataflow decides worker count |
| **Cost model** | Per-cluster-hour | Per-worker-hour (auto-scaled) |
| **Ecosystem** | Spark ecosystem (Delta Lake, MLlib, etc.) | Beam ecosystem (Beam SQL, Beam ML) |
| **GCP-native?** | Hadoop-era, works on any cloud | Built by Google, deeply GCP-integrated |

> **For this cohort:** We use Dataproc because you already know PySpark. But many GCP-native shops use Dataflow. Know both — interviewers will ask.

#### Dataflow Architecture

```
You write a Beam Pipeline (Python or Java)
     |
     v
Submit to Dataflow
     |
     v
Dataflow auto-provisions workers
     |
     v
Workers read from source (GCS, Pub/Sub, BigQuery)
     |
     v
Workers transform data (your Beam code)
     |
     v
Workers write to sink (BigQuery, GCS, Bigtable)
     |
     v
Workers auto-scale up/down as data volume changes
     |
     v
Job completes → workers shut down → you stop paying
```

#### Console Walkthrough: Dataflow

> **Console:** [console.cloud.google.com/dataflow](https://console.cloud.google.com/dataflow)

1. Click **Create job from template** (easiest way to start)
2. Choose template: **Text Files on Cloud Storage to BigQuery**
3. Fill in:
   - Input: `gs://de-training-YOUR-NAME/bronze/calls/*.json`
   - BigQuery table: `de-training:bronze.raw_calls`
   - Schema file: `gs://de-training-YOUR-NAME/schema.json`
   - Temp location: `gs://de-training-YOUR-NAME/temp/`
4. Click **Run job**
5. Watch the DAG visualization — Dataflow shows your pipeline as a graph with throughput metrics

In [ ]:
# ============================================================
# Demo: Dataflow — submit a batch job via gcloud
# ============================================================

dataflow_demo = """
# Option 1: Use a pre-built template (no code)
gcloud dataflow jobs run load-calls-to-bq \\
    --gcs-location=gs://dataflow-templates/latest/GCS_Text_to_BigQuery \\
    --region=us-central1 \\
    --parameters \\
      inputFilePattern=gs://de-training-YOUR-NAME/bronze/calls/*.json,\\
      outputTable=de-training:bronze.raw_calls,\\
      JSONPath=gs://de-training-YOUR-NAME/schema.json,\\
      bigQueryLoadingTemporaryDirectory=gs://de-training-YOUR-NAME/temp/ \\
    --service-account-email=de-pipeline-sa@de-training.iam.gserviceaccount.com

# Check job status
gcloud dataflow jobs list --region=us-central1

# Option 2: Write a custom Beam pipeline (Python)
# pip install apache-beam[gcp]
#
# import apache_beam as beam
# from apache_beam.options.pipeline_options import PipelineOptions
#
# options = PipelineOptions([
#     '--runner=DataflowRunner',
#     '--project=de-training',
#     '--region=us-central1',
#     '--temp_location=gs://de-training-YOUR-NAME/temp/',
#     '--service_account_email=de-pipeline-sa@de-training.iam.gserviceaccount.com',
# ])
#
# with beam.Pipeline(options=options) as p:
#     (p
#      | 'Read' >> beam.io.ReadFromText('gs://bucket/calls/*.json')
#      | 'Parse' >> beam.Map(json.loads)
#      | 'Filter' >> beam.Filter(lambda x: x['duration'] > 0)
#      | 'Write' >> beam.io.WriteToBigQuery('de-training:silver.calls'))
"""
print(dataflow_demo)

#### Hands-On: Silver Transform with Dataflow (Beam Alternative)

> **Same transform you built in PySpark on Dataproc — now in Apache Beam on Dataflow.** This shows you the alternative. Companies choose one or the other.

In [ ]:
# ============================================================
# Lab: Silver transform using Apache Beam on Dataflow
# Same logic as PySpark version: read Bronze, clean, write Silver
# ============================================================

beam_transform = '''
# File: silver_transform_beam.py
# Run: python silver_transform_beam.py --runner=DataflowRunner

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
import json
from datetime import datetime, timedelta

class ParseCallRecord(beam.DoFn):
    """Parse JSON, flatten nested customer, convert UTC to EST."""
    def process(self, line):
        record = json.loads(line)

        # Flatten customer
        customer = record.get("customer", {})
        address = customer.get("address", {})

        # UTC to EST (subtract 5 hours)
        utc_time = datetime.strptime(record["start_time"], "%Y-%m-%dT%H:%M:%SZ")
        est_time = utc_time - timedelta(hours=5)

        yield {
            "call_id": record["call_id"],
            "dnis": record["dnis"],
            "caller_phone": record.get("caller_phone") or "UNKNOWN",
            "start_time_utc": record["start_time"],
            "start_time_est": est_time.isoformat(),
            "call_date": est_time.strftime("%Y-%m-%d"),
            "duration": record["duration"],
            "disposition": record["disposition"],
            "customer_first_name": customer.get("first_name"),
            "customer_last_name": customer.get("last_name"),
            "customer_city": address.get("city"),
            "customer_state": address.get("state"),
        }

class DeduplicateByCallId(beam.DoFn):
    """Keep only the first record for each call_id."""
    def process(self, element):
        call_id, records = element
        yield list(records)[0]  # Keep first occurrence

def run():
    options = PipelineOptions([
        "--runner=DataflowRunner",
        "--project=de-training",
        "--region=us-central1",
        "--temp_location=gs://de-training-YOUR-NAME/temp/",
        "--staging_location=gs://de-training-YOUR-NAME/staging/",
        "--service_account_email=de-pipeline-sa@de-training.iam.gserviceaccount.com",
    ])

    with beam.Pipeline(options=options) as p:
        # Read Bronze
        raw = (p
            | "Read Bronze" >> beam.io.ReadFromText(
                "gs://de-training-YOUR-NAME/bronze/calls/date=*/*.json")
            | "Parse JSON" >> beam.ParDo(ParseCallRecord())
            | "Filter zero duration" >> beam.Filter(lambda x: x["duration"] > 0)
        )

        # Deduplicate
        deduped = (raw
            | "Key by call_id" >> beam.Map(lambda x: (x["call_id"], x))
            | "Group by call_id" >> beam.GroupByKey()
            | "Take first" >> beam.ParDo(DeduplicateByCallId())
        )

        # Write to BigQuery Silver table
        deduped | "Write Silver" >> beam.io.WriteToBigQuery(
            "de-training:silver.calls",
            write_disposition=beam.io.BigQueryDisposition.WRITE_TRUNCATE,
            create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED,
        )

if __name__ == "__main__":
    run()
'''

deploy_beam = """
# Install Beam
pip install apache-beam[gcp]

# Save the script
# (paste the code above into silver_transform_beam.py)

# Run on Dataflow
python silver_transform_beam.py

# Monitor the job
gcloud dataflow jobs list --region=us-central1

# Watch in console: console.cloud.google.com/dataflow/jobs
# You'll see a DAG visualization with throughput per step
"""

print(beam_transform)
print('--- Deploy ---')
print(deploy_beam)
print()
print('Same result as PySpark on Dataproc. Different API. Different engine.')
print('PySpark: DataFrames. Beam: PCollections + DoFn.')
print('Your choice depends on what the company already uses.')

### 3.9 Datastream — Real-Time CDC from Databases

> **In plain English:** Datastream watches your source database (MySQL, PostgreSQL, Oracle) and copies every insert, update, and delete to BigQuery or GCS — in real time, continuously. Like a security camera for your database: it records everything that changes.

| Feature | Datastream | Manual CDC (polling with SQL) |
|---|---|---|
| **Latency** | Seconds (real-time) | Minutes to hours (scheduled queries) |
| **Completeness** | Captures ALL changes (insert, update, delete) | Might miss deletes, overwrites |
| **Load on source DB** | Reads the database log (low impact) | Runs queries against the DB (high impact) |
| **Schema changes** | Auto-detects new columns | You have to update your queries |

**When to use:** Your app database is the source of truth, and you need changes reflected in BigQuery within seconds — not next morning.

#### Console Walkthrough: Datastream

> **Console:** [console.cloud.google.com/datastream](https://console.cloud.google.com/datastream)

1. Click **Create stream**
2. Source: **MySQL** (or PostgreSQL, Oracle)
3. Enter connection details (host, port, user, password)
4. Select databases and tables to replicate
5. Destination: **BigQuery** (or GCS)
6. Click **Create & start**
7. Watch the stream dashboard — see rows flowing in real-time

In [ ]:
# ============================================================
# Demo: Datastream — create a CDC stream via gcloud
# ============================================================

datastream_demo = """
# Step 1: Create a connection profile for the source database
gcloud datastream connection-profiles create source-mysql \\
    --location=us-central1 \\
    --type=mysql \\
    --mysql-hostname=10.0.0.5 \\
    --mysql-port=3306 \\
    --mysql-username=replication_user \\
    --mysql-password=PROMPT

# Step 2: Create a connection profile for BigQuery destination
gcloud datastream connection-profiles create dest-bigquery \\
    --location=us-central1 \\
    --type=bigquery

# Step 3: Create the stream
gcloud datastream streams create mysql-to-bq \\
    --location=us-central1 \\
    --source=source-mysql \\
    --destination=dest-bigquery \\
    --source-config='{"mysql_source_config": {"include_objects": {"mysql_databases": [{"database": "call_center"}]}}}' \\
    --destination-config='{"bigquery_destination_config": {"data_freshness": "300s"}}'

# Step 4: Start the stream
gcloud datastream streams update mysql-to-bq \\
    --location=us-central1 \\
    --state=RUNNING

# Monitor
gcloud datastream streams describe mysql-to-bq --location=us-central1
"""
print(datastream_demo)

### 3.10 Cloud Data Fusion — Visual ETL (No Code)

> **In plain English:** Data Fusion is a drag-and-drop pipeline builder. You connect boxes on a canvas instead of writing PySpark code. Think of it as the "PowerPoint of data pipelines" — accessible to analysts and non-coders.

| Factor | Dataproc / Dataflow | Cloud Data Fusion |
|---|---|---|
| **Interface** | Code (Python/Java) | Visual canvas (drag-and-drop) |
| **Users** | Data engineers | Data engineers + analysts |
| **Flexibility** | Unlimited — write any logic | Limited to available plugins |
| **Best for** | Complex transforms, custom logic | Standard ETL patterns, rapid prototyping |
| **Under the hood** | Spark / Beam | Runs on Dataproc (it generates Spark code for you) |

**Console:** [console.cloud.google.com/data-fusion](https://console.cloud.google.com/data-fusion)

> **For this cohort:** You're learning to write code. Data Fusion is useful to know exists — you may encounter it at companies where analysts build simple pipelines.

### 3.11 Dataplex — Data Governance & Quality

> **In plain English:** Dataplex is the librarian for your data lake. It knows where every dataset lives, who owns it, what it contains, whether it's fresh, and whether it passes quality checks. Without Dataplex, you have data scattered across GCS and BigQuery with no catalog.

**What Dataplex does:**

| Feature | What It Means | Why DEs Care |
|---|---|---|
| **Discovery** | Auto-scans GCS + BigQuery, catalogs all datasets | "What data do we have?" — answered automatically |
| **Data Quality** | Define rules (no nulls, valid ranges, freshness), run checks automatically | Catch bad data before it reaches Gold layer |
| **Lineage** | Track where data came from and where it goes | "Where did this number come from?" — traceable |
| **Data Profiling** | Auto-generate statistics (min, max, null %, distinct count) | Understand a new dataset in seconds |
| **Access Control** | Manage who can read/write which datasets | Column-level security for PII |

**Console:** [console.cloud.google.com/dataplex](https://console.cloud.google.com/dataplex)

> **AWS equivalent:** Lake Formation + Glue Data Quality

In [ ]:
# ============================================================
# Demo: Dataplex — create a lake and run data quality checks
# ============================================================

dataplex_demo = """
# Create a Dataplex lake (logical grouping of your data)
gcloud dataplex lakes create de-training-lake \\
    --location=us-central1 \\
    --display-name="DE Training Data Lake"

# Create a zone (Bronze, Silver, Gold)
gcloud dataplex zones create bronze-zone \\
    --lake=de-training-lake \\
    --location=us-central1 \\
    --type=RAW \\
    --resource-location-type=SINGLE_REGION \\
    --display-name="Bronze Zone"

# Attach a GCS bucket as an asset
gcloud dataplex assets create bronze-calls \\
    --lake=de-training-lake \\
    --zone=bronze-zone \\
    --location=us-central1 \\
    --resource-type=STORAGE_BUCKET \\
    --resource-name=projects/de-training/buckets/de-training-YOUR-NAME \\
    --display-name="Bronze Calls Data"

# Dataplex will auto-discover and catalog the data
# Check discovery results:
gcloud dataplex assets describe bronze-calls \\
    --lake=de-training-lake \\
    --zone=bronze-zone \\
    --location=us-central1
"""
print(dataplex_demo)

#### Hands-On: Run Data Quality Checks with Dataplex

> **After your Silver transform runs, verify the data meets quality standards before it reaches Gold. This is the GCP-native alternative to writing custom quality checks in PySpark.**

In [ ]:
# ============================================================
# Lab: Dataplex data quality scan on Silver tables
# ============================================================

dataplex_lab = """
# Step 1: Create a data quality scan on your Silver calls table in BigQuery
# (Dataplex quality scans work on BigQuery tables)

# First, load your Silver data into BigQuery if not already there:
bq load --source_format=PARQUET --replace \\
    silver.calls \\
    gs://de-training-YOUR-NAME/silver/calls/*.parquet

# Step 2: Create a data quality spec file
cat > /tmp/quality_rules.yaml << 'EOF'
rules:
  - column: call_id
    nonNullExpectation: {}
    description: "call_id must never be null"
  - column: call_id
    uniquenessExpectation: {}
    description: "call_id must be unique (no duplicates after dedup)"
  - column: dnis
    nonNullExpectation: {}
    description: "dnis must never be null"
  - column: duration
    rangeExpectation:
      minValue: "1"
      maxValue: "3600"
    description: "duration must be between 1 and 3600 seconds"
  - column: disposition
    setExpectation:
      values: ["completed", "transferred", "dropped", "no_answer", "voicemail"]
    description: "disposition must be a known value"
  - rowConditionExpectation:
      sqlExpression: "start_time_est IS NOT NULL"
    description: "UTC to EST conversion must have succeeded"
EOF

# Step 3: Create and run the data quality scan
gcloud dataplex datascans create data-quality silver-calls-quality \\
    --location=us-central1 \\
    --data-source-resource='//bigquery.googleapis.com/projects/de-training/datasets/silver/tables/calls' \\
    --data-quality-spec-file=/tmp/quality_rules.yaml

# Run the scan
gcloud dataplex datascans run silver-calls-quality --location=us-central1

# Check results
gcloud dataplex datascans jobs list --datascan=silver-calls-quality --location=us-central1

# Get detailed results
gcloud dataplex datascans jobs describe LATEST \\
    --datascan=silver-calls-quality \\
    --location=us-central1 \\
    --format='table(dataQualityResult.rules.rule.column,dataQualityResult.rules.passed)'
"""
print(dataplex_lab)
print()
print('Expected results after Silver transform:')
print('  call_id NOT NULL:  PASSED (all rows have call_id)')
print('  call_id UNIQUE:    PASSED (dedup removed duplicates)')
print('  dnis NOT NULL:     PASSED')
print('  duration 1-3600:   PASSED (filtered zero-duration in Silver)')
print('  disposition valid:  PASSED')
print('  UTC->EST done:     PASSED (start_time_est populated)')
print()
print('If ANY rule fails: investigate Silver transform, fix, re-run.')
print('In production: integrate this scan into your Airflow DAG (M09).')

### 3.12 Looker Studio — Dashboards (The Consumer Layer)

> **In plain English:** Looker Studio is where your pipeline's output becomes visible to the business. The VP of Sales doesn't open BigQuery — they open a dashboard. Looker Studio connects directly to BigQuery and auto-refreshes.

**This is the "so what" of your entire pipeline:**

```
calls.json --> Bronze --> Silver --> Gold --> BigQuery --> Looker Studio Dashboard
                                                          |
                                                          v
                                              VP sees: 'Spring Health Promo
                                              converted 18% this week,
                                              up from 12% last week.
                                              Peak hours: 10am-2pm.'
```

**Console:** [lookerstudio.google.com](https://lookerstudio.google.com)

**Quick Setup (5 minutes):**

1. Go to Looker Studio → **Create** → **Report**
2. Data source: **BigQuery**
3. Select project → `gold` dataset → `mart_daily_campaign` table
4. Click **Add** → **Add to report**
5. Drag fields to build charts:
   - Bar chart: `campaign_name` (dimension) × `total_calls` (metric)
   - Line chart: `call_date` (dimension) × `conversion_rate` (metric)
   - Scorecard: `SUM(total_revenue)`
6. Click **View** to see the live dashboard

The dashboard auto-refreshes from BigQuery. When your pipeline updates the Gold marts, the dashboard shows fresh numbers.

> **AWS equivalent:** QuickSight. **Other options:** Tableau, Metabase, Superset.

#### Hands-On: Build a Dashboard from Your Gold Marts (10 minutes)

> **This is the payoff. The VP finally sees the data.** You'll connect Looker Studio directly to the BigQuery Gold marts you built.

**Prerequisites:** Gold marts loaded in BigQuery (`gold.mart_daily_campaign`, `gold.mart_hourly_volume`)

**Step-by-step:**

1. Go to [lookerstudio.google.com](https://lookerstudio.google.com)
2. Click **Blank Report** (or **Create** -> **Report**)
3. **Add data** -> **BigQuery** -> select your project -> `gold` dataset -> `mart_daily_campaign`
4. Click **Add** -> **Add to report**

**Build these 4 charts:**

| Chart | Type | Dimension | Metric | What It Shows |
|---|---|---|---|---|
| Campaign performance | Bar chart | `campaign_name` | `total_calls`, `conversions` | Which campaign gets the most calls and converts best |
| Daily trend | Line chart | `call_date` | `total_revenue` | Is revenue trending up or down? |
| Conversion rate | Scorecard | — | `AVG(conversion_rate)` | Overall conversion rate (one big number) |
| Revenue total | Scorecard | — | `SUM(total_revenue)` | Total revenue (one big number) |

**Add a second data source** (for hourly volume):

5. Click **Add data** -> BigQuery -> `gold.mart_hourly_volume`
6. Add a **heatmap or bar chart**: `call_hour` (dimension) x `total_calls` (metric)
   - This shows peak hours — when should the call center staff more agents?

**Add a date filter:**

7. Click **Add a control** -> **Date range control**
8. Drag it to the top of the report
9. Now the VP can filter by date range

**Share it:**

10. Click **Share** -> add the VP's email -> they get a live link
11. The dashboard auto-refreshes from BigQuery. When your pipeline updates Gold, the VP sees fresh numbers.

```
What the VP sees:

┌──────────────────────────────────────────────────────────────┐
│  Campaign Performance Dashboard           [Mar 17-19, 2026] │
│                                                             │
│  Total Revenue: $12,450    Conversion Rate: 14.7%           │
│                                                             │
│  ████████████  Spring Health Promo (52 calls, 8 conv)       │
│  █████████     Electronics Clearance (41 calls, 6 conv)     │
│  ███████       Skin Care Launch (35 calls, 5 conv)          │
│  ██████        Fitness New Year (30 calls, 4 conv)          │
│                                                             │
│  Peak Hours: 10am-2pm (staff more agents here)              │
└──────────────────────────────────────────────────────────────┘
```

> **This is why you built the pipeline.** Not for the Bronze layer. Not for the MERGE. For this dashboard. Every technical decision maps to this moment.

### 3.13 Cloud Run — Containerized Serverless

> **In plain English:** Cloud Functions has a 15-minute limit and limited memory. Cloud Run runs Docker containers serverlessly — same "no server management" but can run for hours, use more memory, and handle any language/framework.

| Factor | Cloud Functions | Cloud Run |
|---|---|---|
| **Runtime limit** | 15 minutes | No limit (configurable timeout) |
| **Memory** | Up to 32 GB | Up to 32 GB |
| **Packaging** | Single Python file + requirements.txt | Docker container (any language) |
| **Best for** | Quick event handlers (file validation, alerts) | Longer ETL jobs, API servers, complex transforms |
| **Trigger** | Pub/Sub, HTTP, GCS events | HTTP, Pub/Sub, Cloud Scheduler, Eventarc |

**When to use Cloud Run instead of Cloud Functions:**
- Your transform takes > 15 minutes
- You need custom dependencies (C libraries, ML models)
- You want to run the same container locally and in the cloud

**Console:** [console.cloud.google.com/run](https://console.cloud.google.com/run)

### 3.14 Secret Manager — Credential Management

> **In plain English:** Never hardcode passwords, API keys, or connection strings in your pipeline code. Secret Manager stores them encrypted and your code fetches them at runtime.

**Without Secret Manager:**
```python
# BAD — password in code (gets committed to Git, visible in logs)
connection = psycopg2.connect(host='10.0.0.5', password='P@ssw0rd123')
```

**With Secret Manager:**
```python
# GOOD — password fetched at runtime from encrypted store
from google.cloud import secretmanager
client = secretmanager.SecretManagerServiceClient()
secret = client.access_secret_version(name='projects/de-training/secrets/db-password/versions/latest')
password = secret.payload.data.decode('utf-8')
connection = psycopg2.connect(host='10.0.0.5', password=password)
```

**Console:** [console.cloud.google.com/security/secret-manager](https://console.cloud.google.com/security/secret-manager)

In [ ]:
# ============================================================
# Demo: Secret Manager — store and retrieve secrets
# ============================================================

secret_demo = """
# Create a secret
echo -n 'P@ssw0rd123' | gcloud secrets create db-password \\
    --data-file=- \\
    --replication-policy=automatic

# Read the secret
gcloud secrets versions access latest --secret=db-password
# Output: P@ssw0rd123

# Grant your service account access to the secret
gcloud secrets add-iam-policy-binding db-password \\
    --member='serviceAccount:de-pipeline-sa@de-training.iam.gserviceaccount.com' \\
    --role='roles/secretmanager.secretAccessor'

# Update a secret (add new version)
echo -n 'NewP@ssw0rd456' | gcloud secrets versions add db-password --data-file=-

# List all secrets
gcloud secrets list
"""
print(secret_demo)
print()
print('AWS equivalent: aws secretsmanager create-secret / get-secret-value')

#### Hands-On: Use Secret Manager in Your Pipeline

> **Instead of hardcoding credentials in your PySpark scripts, fetch them from Secret Manager at runtime.**

In [ ]:
# ============================================================
# Lab: Store and use credentials via Secret Manager
# Run these in Cloud Shell
# ============================================================

setup_secrets = """
# Step 1: Store your BigQuery project ID as a secret
echo -n 'de-training' | gcloud secrets create bq-project-id --data-file=-

# Step 2: Store your Snowflake password
echo -n 'YourSnowflakePassword' | gcloud secrets create snowflake-password --data-file=-

# Step 3: Grant your service account access
for SECRET in bq-project-id snowflake-password; do
    gcloud secrets add-iam-policy-binding $SECRET \\
        --member='serviceAccount:de-pipeline-sa@de-training.iam.gserviceaccount.com' \\
        --role='roles/secretmanager.secretAccessor'
done

# Step 4: Verify secrets exist
gcloud secrets list
"""

pipeline_code = '''
# In your PySpark pipeline (silver_transform.py), fetch secrets:

from google.cloud import secretmanager

def get_secret(secret_id):
    """Fetch a secret from GCP Secret Manager."""
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/de-training/secrets/{secret_id}/versions/latest"
    response = client.access_secret_version(name=name)
    return response.payload.data.decode("utf-8")

# Use in your pipeline
PROJECT_ID = get_secret("bq-project-id")
SNOWFLAKE_PW = get_secret("snowflake-password")

# Now use these variables in your connections
# No passwords in code. No passwords in Git. No passwords in logs.
print(f"Connected to project: {PROJECT_ID}")
'''

print('--- Setup (run in Cloud Shell) ---')
print(setup_secrets)
print('--- Use in Pipeline Code ---')
print(pipeline_code)
print()
print('You should see: gcloud secrets list shows bq-project-id and snowflake-password')
print('Your pipeline code never contains credentials — they are fetched at runtime.')

### 3.15 Cloud Scheduler + Workflows — Lightweight Orchestration

> **In plain English:** Not every pipeline needs full Airflow. Cloud Scheduler is a cron job in the cloud. Workflows is a simple state machine. Together they handle pipelines that have 3-5 steps without the overhead of Cloud Composer.

| Tool | What It Does | When to Use | AWS Equivalent |
|---|---|---|---|
| **Cloud Scheduler** | Triggers something on a cron schedule | Daily BigQuery refresh, hourly file check | EventBridge Scheduler |
| **Workflows** | Chains API calls with conditions and retries | Simple multi-step pipeline (3-5 steps) | Step Functions |
| **Cloud Composer** | Full Airflow (DAGs, operators, XCom (Cross-Communication), backfill) | Complex pipelines (10+ steps, dependencies) | MWAA |

**Rule of thumb:**
- 1 step → Cloud Scheduler + Cloud Function
- 3-5 steps → Workflows
- 10+ steps with complex dependencies → Cloud Composer (Airflow)

In [ ]:
# ============================================================
# Demo: Cloud Scheduler — trigger a pipeline daily
# ============================================================

scheduler_demo = """
# Create a cron job that triggers a Pub/Sub message daily at 6 AM EST
gcloud scheduler jobs create pubsub daily-pipeline-trigger \\
    --schedule='0 6 * * *' \\
    --time-zone='America/New_York' \\
    --topic=de-pipeline-events \\
    --message-body='{"action": "run_daily_pipeline", "date": "today"}' \\
    --location=us-central1

# List scheduled jobs
gcloud scheduler jobs list --location=us-central1

# Test it (run now without waiting for schedule)
gcloud scheduler jobs run daily-pipeline-trigger --location=us-central1

# Pause/Resume
gcloud scheduler jobs pause daily-pipeline-trigger --location=us-central1
gcloud scheduler jobs resume daily-pipeline-trigger --location=us-central1
"""
print(scheduler_demo)

#### Hands-On: Schedule Your Pipeline with Cloud Scheduler

> **This triggers your Cloud Function (which triggers Dataproc) on a daily schedule. A simpler alternative to Cloud Composer for pipelines with few steps.**

In [ ]:
# ============================================================
# Lab: Set up daily pipeline trigger with Cloud Scheduler
# ============================================================

scheduler_lab = """
# Step 1: Create a Pub/Sub topic for scheduled triggers
gcloud pubsub topics create daily-pipeline-trigger

# Step 2: Create a Cloud Scheduler job
# This fires every day at 6:00 AM Eastern Time
gcloud scheduler jobs create pubsub daily-etl-run \\
    --schedule='0 6 * * *' \\
    --time-zone='America/New_York' \\
    --topic=daily-pipeline-trigger \\
    --message-body='{"pipeline": "va_analytics", "mode": "incremental", "date": "today"}' \\
    --location=us-central1

# Step 3: Create a Cloud Function that listens to this topic
# and submits a Dataproc job
# (Reuse the function from Phase 7, but trigger from this topic
#  instead of GCS events)

# Step 4: Verify the schedule
gcloud scheduler jobs list --location=us-central1

# Step 5: Test it NOW (don't wait until 6 AM)
gcloud scheduler jobs run daily-etl-run --location=us-central1

# Step 6: Check that it triggered
gcloud functions logs read validate-incoming-file --region=us-central1 --limit=5

# You should see: the function was invoked by the scheduled message

# Useful commands:
gcloud scheduler jobs pause daily-etl-run --location=us-central1   # stop scheduling
gcloud scheduler jobs resume daily-etl-run --location=us-central1  # restart
gcloud scheduler jobs delete daily-etl-run --location=us-central1  # remove
"""
print(scheduler_lab)
print()
print('Flow: Cloud Scheduler (6 AM) --> Pub/Sub message --> Cloud Function --> Dataproc job')
print('This is the lightweight alternative to Cloud Composer (Airflow).')
print('Use this when your pipeline has 1-3 steps. Use Composer for 10+ steps.')

### 3.16 Transfer Service + BigQuery Data Transfer

**Storage Transfer Service** — move data from other clouds or on-prem to GCS:

| Source | Destination | Use Case |
|---|---|---|
| AWS S3 | GCS | Migrating data lake from AWS to GCP |
| Azure Blob | GCS | Cross-cloud replication |
| On-prem HDFS | GCS | Hadoop migration to cloud |
| Another GCS bucket | GCS | Cross-region backup |

**BigQuery Data Transfer Service** — scheduled loading from SaaS into BigQuery:

| Source | What It Loads |
|---|---|
| Google Ads | Campaign performance, clicks, impressions, cost |
| YouTube | Video analytics, channel stats |
| Amazon S3 | Recurring loads from S3 into BigQuery |
| Teradata / Redshift | Migration from other warehouses |

**Console:** [console.cloud.google.com/bigquery/transfers](https://console.cloud.google.com/bigquery/transfers)

In [ ]:
# ============================================================
# Demo: Transfer Service — copy data from S3 to GCS
# ============================================================

transfer_demo = """
# Transfer data from AWS S3 to GCS (one-time or recurring)
gcloud transfer jobs create \\
    --source-agent-pool='' \\
    --source-creds-file=s3-credentials.json \\
    s3://source-bucket/data/ \\
    gs://de-training-YOUR-NAME/imported-from-s3/ \\
    --name=s3-to-gcs-migration

# Check transfer status
gcloud transfer jobs list
gcloud transfer jobs describe s3-to-gcs-migration

# BigQuery Data Transfer — schedule daily load from GCS
bq mk --transfer_config \\
    --data_source=google_cloud_storage \\
    --target_dataset=bronze \\
    --display_name='Daily Call Data Load' \\
    --schedule='every 24 hours' \\
    --params='{"data_path_template": "gs://de-training-YOUR-NAME/bronze/calls/*.json", "destination_table_name_template": "raw_calls", "file_format": "JSON"}'
"""
print(transfer_demo)

### 3.17 Vertex AI — When Your Pipeline Feeds ML

> **In plain English:** Your pipeline builds clean, reliable data in BigQuery. Vertex AI is what happens next — when data scientists use that data to train ML models. As a DE, you build the pipeline. The ML team consumes your Gold marts.

**How DE pipelines connect to ML:**

```
Your pipeline:  Bronze --> Silver --> Gold (BigQuery)
                                       |
                                       v
ML team:                        Vertex AI reads Gold marts
                                       |
                                       v
                                Train model (churn prediction,
                                fraud detection, demand forecast)
                                       |
                                       v
                                Deploy model as API endpoint
```

**BigQuery ML** — train models directly in SQL (no Python, no Vertex AI):

```sql
-- Train a model to predict call conversion
CREATE MODEL gold.conversion_predictor
OPTIONS (model_type='logistic_reg') AS
SELECT
    duration, hour_of_day, campaign_name, channel,
    CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END AS converted
FROM gold.mart_calls_detail;

-- Predict on new data
SELECT * FROM ML.PREDICT(MODEL gold.conversion_predictor,
    (SELECT * FROM silver.calls WHERE call_date = CURRENT_DATE()));
```

**Console:** [console.cloud.google.com/vertex-ai](https://console.cloud.google.com/vertex-ai)

> **For this cohort:** Focus on building the pipeline. Know that Vertex AI and BigQuery ML exist — ML teams depend on the quality of YOUR pipeline. If your data is wrong, the model is wrong. "Garbage in, garbage out" starts at Bronze.

### 3.18 Cloud Logging & Monitoring — Observability

> **In plain English:** If your pipeline fails at 3 AM, how do you know? Cloud Logging captures every log from every service. Cloud Monitoring triggers alerts when something looks wrong (error rate spikes, job takes too long, no data for 2 hours).

**Console:**
- Logging: [console.cloud.google.com/logs](https://console.cloud.google.com/logs)
- Monitoring: [console.cloud.google.com/monitoring](https://console.cloud.google.com/monitoring)

**Key queries for pipeline monitoring:**

```bash
# View Dataproc job logs
gcloud logging read 'resource.type="cloud_dataproc_job"' --limit=20 --format=json

# View Cloud Function logs
gcloud logging read 'resource.type="cloud_function" AND resource.labels.function_name="validate-incoming-file"' --limit=20

# View BigQuery job errors
gcloud logging read 'resource.type="bigquery_resource" AND severity>=ERROR' --limit=10

# Create an alert policy (email when Dataproc job fails)
# Console: Monitoring -> Alerting -> Create Policy
# Condition: resource.type = "cloud_dataproc_job" AND metric.type = "dataproc.googleapis.com/job/failed_count" > 0
# Notification: Email, Slack webhook, PagerDuty
```

> **AWS equivalent:** CloudWatch Logs + CloudWatch Alarms

#### Hands-On: Monitor Your Pipeline with Cloud Logging

> **Your pipeline ran. Did it work? How long did it take? Were there errors?** These commands answer those questions.

In [ ]:
# ============================================================
# Lab: Query logs from your actual pipeline components
# Run these in Cloud Shell after running the pipeline
# ============================================================

logging_lab = """
# 1. Cloud Function logs — did file validation work?
gcloud logging read '
    resource.type="cloud_function"
    resource.labels.function_name="validate-incoming-file"
    ' --limit=20 --format='table(timestamp,textPayload)'

# You should see:
#   File uploaded: gs://bucket/incoming/calls/...
#   Valid: 510, Invalid: 0
#   Moved to: bronze/calls/...

# 2. Dataproc job logs — did Silver transform succeed?
gcloud logging read '
    resource.type="cloud_dataproc_job"
    severity>=INFO
    ' --limit=20 --format='table(timestamp,textPayload)'

# 3. BigQuery job logs — did Gold mart refresh work?
gcloud logging read '
    resource.type="bigquery_resource"
    protoPayload.methodName="jobservice.jobcompleted"
    ' --limit=10 --format='table(timestamp,protoPayload.serviceData.jobCompletedEvent.job.jobName.jobId,protoPayload.serviceData.jobCompletedEvent.job.jobStatus.state)'

# 4. Find ERRORS only (across all services)
gcloud logging read 'severity>=ERROR' --limit=20 \\
    --format='table(timestamp,resource.type,textPayload)'

# 5. How long did the Dataproc job take?
gcloud dataproc jobs list --region=us-central1 \\
    --format='table(jobId,status.state,statusHistory[0].stateStartTime,placement.clusterName)'

# 6. Set up a log-based alert (get notified when pipeline fails)
# Console: Monitoring -> Alerting -> Create Policy
# Or via gcloud:
gcloud alpha monitoring policies create \\
    --notification-channels=YOUR_CHANNEL_ID \\
    --display-name='Pipeline Failure Alert' \\
    --condition-display-name='Dataproc Job Failed' \\
    --condition-filter='resource.type=\"cloud_dataproc_job\" AND severity>=ERROR'
"""
print(logging_lab)
print()
print('Console alternative: console.cloud.google.com/logs')
print('  - Use the query builder to filter by resource type, severity, time range')
print('  - Click any log entry to see full details')
print('  - Pin important queries as saved queries')

### 3.19 Cloud KMS — Encryption Key Management

> **In plain English:** By default, GCP encrypts all data at rest with Google-managed keys. Cloud KMS lets you manage your OWN encryption keys — required for compliance (HIPAA, PCI, SOC 2).

**When you need KMS:**
- PII/PHI data in your pipeline (customer phone, email, health records)
- Compliance requirements mandate customer-managed encryption keys (CMEK)
- Regulatory audit requires you to prove who has access to decryption keys

```bash
# Create a key ring and key
gcloud kms keyrings create de-training-keyring --location=us-central1
gcloud kms keys create pipeline-key \
    --keyring=de-training-keyring \
    --location=us-central1 \
    --purpose=encryption

# Use this key when creating BigQuery datasets
bq mk --dataset \
    --default_kms_key=projects/de-training/locations/us-central1/keyRings/de-training-keyring/cryptoKeys/pipeline-key \
    de-training:gold_encrypted
```

**Console:** [console.cloud.google.com/security/kms](https://console.cloud.google.com/security/kms)

> **AWS equivalent:** AWS KMS (same concept, same pattern)

### 3.20 Complete GCP Data Engineering Service Map

Here's every GCP service a data engineer uses, organized by pipeline stage:

```
INGEST              STORE              PROCESS            SERVE              CONSUME
-------             -----              -------            -----              -------
Pub/Sub             GCS                Dataproc           BigQuery           Looker Studio
Datastream          Cloud SQL          Dataflow           Cloud SQL          Vertex AI
Transfer Service    Firestore          Cloud Functions    Bigtable           BigQuery ML
Cloud Functions     Bigtable           Cloud Run          Memorystore        API endpoints
IoT Core            BigQuery           Cloud Data Fusion                     Notebooks

ORCHESTRATE         GOVERN             SECURE             MONITOR
-----------         ------             ------             -------
Cloud Composer      Dataplex           IAM                Cloud Logging
Cloud Scheduler     Data Catalog       Secret Manager     Cloud Monitoring
Workflows           DLP API            Cloud KMS          Error Reporting
Snowflake Tasks     Lineage            VPC/Private SA     Alerting
```

**In this module, you used:** GCS, Dataproc, BigQuery, Cloud Functions, Pub/Sub, IAM, Secret Manager

**In M09, you'll add:** Cloud Composer (orchestration), Cloud Monitoring (alerting)

**Know they exist:** Dataflow, Datastream, Data Fusion, Dataplex, Looker Studio, Cloud Run, Workflows, Transfer Service, Vertex AI, Cloud KMS

---
## 4. Snowflake

> **In plain English:** Think of a food court in a mall. Each restaurant (virtual warehouse) has its own kitchen and cooks, but they all share the same walk-in refrigerator (storage). The pizza place being slammed at lunch doesn't slow down the sushi place — separate kitchens. When a restaurant closes for the night, the kitchen shuts off (auto-suspend) and stops costing money. But the refrigerator stays on — the food (data) is always there.

Snowflake is a cloud data platform that runs **on top of** AWS, GCP, or Azure. It's not a cloud provider — it's a cloud-agnostic data warehouse, data lake, and data sharing platform.

### Why Snowflake Exists

> "If Redshift is great on AWS and BigQuery is great on GCP, why do I need Snowflake?"

**Answer:** Multi-cloud and simplicity.

| Scenario | Choose Snowflake When |
|----------|----------------------|
| **Multi-cloud** | Your company uses AWS for apps and GCP for analytics. Snowflake runs on both. |
| **Data sharing** | You need to share data with partners/customers without copying files. Snowflake's data sharing is zero-copy, cross-account. |
| **Simplicity** | Redshift requires cluster management. BigQuery has its own SQL dialect. Snowflake is standard SQL with simple scaling. |
| **Mixed workloads** | You need warehouse + data lake + data engineering + data science in one platform. |

### Architecture

```
┌──────────────────────────────────────────────────────┐
│                    SNOWFLAKE                          │
│                                                      │
│  ┌─────────────────────────────────────────────────┐ │
│  │  Cloud Services Layer                           │ │
│  │  Metadata, query optimization, security,        │ │
│  │  access control, infrastructure management      │ │
│  └─────────────────────────────────────────────────┘ │
│                                                      │
│  ┌───────────┐  ┌───────────┐  ┌───────────┐       │
│  │ Warehouse │  │ Warehouse │  │ Warehouse │       │
│  │   XS      │  │    M      │  │    L      │       │
│  │ (dev/test)│  │  (ETL)    │  │ (analysts)│       │
│  └───────────┘  └───────────┘  └───────────┘       │
│  Each warehouse = independent compute               │
│  Can run simultaneously, no contention              │
│  Auto-suspend when idle (stop paying!)              │
│                                                      │
│  ┌─────────────────────────────────────────────────┐ │
│  │  Storage Layer                                  │ │
│  │  Compressed columnar on S3/GCS/Azure Blob       │ │
│  │  Managed by Snowflake — you don't touch it      │ │
│  └─────────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────┘
```

**The key insight:** Each virtual warehouse is **independent compute**. Your ETL warehouse can run a heavy transform at 3 AM without affecting the analyst warehouse running morning reports. They share the same data but not the same compute.


### Snowflake Data Loading & Processing Commands

| Command | What It Does | When to Use |
|---------|-------------|-------------|
| `PUT` | Upload local file → internal stage | Dev/testing, small files from your laptop |
| `GET` | Download from stage → local | Export, debugging |
| `COPY INTO <table>` | Bulk load from stage → table | **Primary batch loading** — fast, parallel |
| `COPY INTO <location>` | Unload table → stage or S3/GCS | Export to data lake |
| `Snowpipe` | Auto-ingest on file arrival (uses SQS/GCS notifications) | **Continuous loading** — file lands in S3 → loaded in minutes |
| `MERGE INTO` | Upsert (insert + update + delete) | SCD Type 2, late-arriving data, CDC |
| `INSERT INTO ... SELECT` | Transform and load within Snowflake | Silver → Gold transforms |
| `CREATE EXTERNAL TABLE` | Query S3/GCS data without loading | Lakehouse pattern, Bronze layer access |
| `STREAM` | CDC capture — tracks inserts/updates/deletes on a table | **Incremental processing** — only see what changed |
| `TASK` | Scheduled SQL or Snowpark execution | Orchestration within Snowflake |


#### Hello World: Snowflake

Before stages, Snowpipe, and Streams — let's just run one query:


In [ ]:
# ============================================================
# Hello World: Snowflake — simplest possible operation
# ============================================================

hello_snowflake = """
-- In Snowsight worksheet (or SnowSQL CLI):

-- 1. What account am I on?
SELECT CURRENT_ACCOUNT(), CURRENT_ROLE(), CURRENT_WAREHOUSE();

-- 2. Create a database and table with one row
CREATE DATABASE IF NOT EXISTS hello_test;
USE hello_test;
CREATE TABLE greetings (message VARCHAR);
INSERT INTO greetings VALUES ('Hello from Snowflake!');
SELECT * FROM greetings;

-- 3. Try the VARIANT type (Snowflake's superpower for JSON)
CREATE TABLE json_test (data VARIANT);
INSERT INTO json_test SELECT PARSE_JSON('{"name": "Alice", "score": 95}');
SELECT data:name::STRING, data:score::INT FROM json_test;

-- 4. Cleanup
DROP DATABASE hello_test;

-- That's it. SQL + JSON support + instant scaling = Snowflake.
"""
print(hello_snowflake)

In [ ]:
# ============================================================
# Demo: Snowflake — Stages and COPY INTO
# ============================================================

snowflake_demo = '''
-- Step 1: Create a database and schema
CREATE DATABASE IF NOT EXISTS de_training;
USE DATABASE de_training;
CREATE SCHEMA IF NOT EXISTS bronze;

-- Step 2: Create an external stage pointing to S3
CREATE OR REPLACE STAGE bronze.s3_calls
    URL = 's3://de-training-bronze/calls/'
    -- STORAGE_INTEGRATION = de_training_s3_integration  (preferred -- no keys)
    FILE_FORMAT = (TYPE = 'JSON');

-- Step 3: See what files are in the stage
LIST @bronze.s3_calls;

-- Step 4: Create the target table
CREATE OR REPLACE TABLE bronze.raw_calls (
    raw_data VARIANT  -- VARIANT = Snowflake semi-structured type (like JSON)
);

-- Step 5: COPY INTO — load the data
COPY INTO bronze.raw_calls
FROM @bronze.s3_calls
FILE_FORMAT = (TYPE = 'JSON')
ON_ERROR = 'CONTINUE';

-- Step 6: Query the loaded data (VARIANT allows dot notation)
SELECT
    raw_data:call_id::STRING AS call_id,
    raw_data:dnis::STRING AS dnis,
    raw_data:duration::INT AS duration,
    raw_data:disposition::STRING AS disposition,
    raw_data:customer.first_name::STRING AS first_name,
    raw_data:customer.address.city::STRING AS city
FROM bronze.raw_calls
LIMIT 10;
'''
print(snowflake_demo)

# **You should see:** After running the Snowflake COPY INTO, run `SELECT COUNT(*) FROM bronze.raw_calls` — it should return the number of records in your calls.json file. If 0 rows, check: (1) stage path is correct, (2) file format matches, (3) `ON_ERROR='CONTINUE'` is set.

In [ ]:
# ============================================================
# Demo: Snowpipe — auto-ingest on file arrival
# ============================================================

snowpipe_demo = '''
-- Snowpipe = continuous loading triggered by S3 events

-- Step 1: Create the pipe
CREATE OR REPLACE PIPE bronze.calls_pipe
    AUTO_INGEST = TRUE
AS
COPY INTO bronze.raw_calls
FROM @bronze.s3_calls
FILE_FORMAT = (TYPE = 'JSON')
ON_ERROR = 'CONTINUE';

-- Step 2: Get the SQS ARN (configure in S3 event notifications)
SHOW PIPES LIKE 'calls_pipe';
-- The 'notification_channel' column = the SQS queue ARN

-- Step 3: Monitor pipe status
SELECT SYSTEM$PIPE_STATUS('bronze.calls_pipe');
'''
print(snowpipe_demo)

#### Why Did That Work? (Snowpipe Under the Hood)

Here's the full chain of events when a file lands in S3:

```
1. File uploaded to S3          (your source system or pipeline)
       │
       ▼
2. S3 Event Notification fires  (configured when you created the pipe)
       │
       ▼
3. SQS queue receives message   (the 'notification_channel' from SHOW PIPES)
       │
       ▼
4. Snowpipe service polls SQS   (Snowflake manages this — you don't)
       │
       ▼
5. COPY INTO runs serverlessly   (uses Snowflake's shared compute, not YOUR warehouse)
       │
       ▼
6. Data available in table       (typically 1-3 minutes end-to-end)
```

> **The key insight:** Snowpipe doesn't use your virtual warehouse. It uses Snowflake's own serverless compute, billed per-file (very cheap). This is why auto-ingest is cost-effective even for frequent small files — you're not spinning up a warehouse for each file.

In [ ]:
# ============================================================
# Demo: Snowflake MERGE INTO — upsert pattern
# ============================================================

snowflake_merge = '''
MERGE INTO silver.calls AS target
USING staging.new_calls AS source
ON target.call_id = source.call_id

WHEN MATCHED AND source.updated_at > target.updated_at THEN
    UPDATE SET
        disposition = source.disposition,
        payment_status = source.payment_status,
        amount = source.amount,
        updated_at = source.updated_at

WHEN NOT MATCHED THEN
    INSERT (call_id, dnis, caller_phone, start_time, duration,
            disposition, payment_status, amount, created_at, updated_at)
    VALUES (source.call_id, source.dnis, source.caller_phone,
            source.start_time, source.duration,
            source.disposition, source.payment_status,
            source.amount, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());
'''
print(snowflake_merge)

In [ ]:
# ============================================================
# Demo: Snowflake Streams and Tasks — CDC + Orchestration
# ============================================================

streams_tasks = '''
-- STREAM: Track changes on a table (CDC)
CREATE OR REPLACE STREAM bronze.calls_changes
ON TABLE bronze.raw_calls;

-- Check what is in the stream
SELECT * FROM bronze.calls_changes LIMIT 10;
-- METADATA$ACTION = 'INSERT', 'DELETE'
-- METADATA$ISUPDATE = TRUE/FALSE

-- TASK: Scheduled execution (like a cron job inside Snowflake)
CREATE OR REPLACE TASK silver.process_new_calls
    WAREHOUSE = 'DE_ETL  # ETL = Extract, Transform, Load_WH'
    SCHEDULE = '5 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('bronze.calls_changes')
AS
MERGE INTO silver.calls AS target
USING (
    SELECT
        raw_data:call_id::STRING AS call_id,
        raw_data:dnis::STRING AS dnis,
        raw_data:duration::INT AS duration,
        raw_data:disposition::STRING AS disposition
    FROM bronze.calls_changes
    WHERE METADATA$ACTION = 'INSERT'
) AS source
ON target.call_id = source.call_id
WHEN MATCHED THEN UPDATE SET disposition = source.disposition
WHEN NOT MATCHED THEN INSERT (call_id, dnis, duration, disposition)
    VALUES (source.call_id, source.dnis, source.duration, source.disposition);

ALTER TASK silver.process_new_calls RESUME;

-- Task tree (chaining tasks like a mini-DAG):
CREATE OR REPLACE TASK gold.refresh_daily_campaign
    WAREHOUSE = 'DE_ETL_WH'
    AFTER silver.process_new_calls
AS
INSERT OVERWRITE INTO gold.mart_daily_campaign
SELECT
    DATE(start_time) AS report_date,
    campaign_name,
    COUNT(*) AS total_calls,
    SUM(CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END) AS conversions
FROM silver.calls c
JOIN silver.campaigns camp ON c.dnis = camp.dnis
GROUP BY 1, 2;

ALTER TASK gold.refresh_daily_campaign RESUME;
'''
print(streams_tasks)

### Snowpark — PySpark-Like API on Snowflake

Snowpark lets you write DataFrame transformations that execute **inside Snowflake** — on Snowflake compute, not your laptop or an EMR cluster.

| PySpark | Snowpark | Same? |
|---------|----------|-------|
| `spark.read.json("s3://...")` | `session.table("bronze.raw_calls")` | Different source, same concept |
| `df.filter(col("duration") > 60)` | `df.filter(col("duration") > 60)` | Identical syntax |
| `df.groupBy("dnis").agg(avg("duration"))` | `df.group_by("dnis").agg(avg("duration"))` | Nearly identical (`groupBy` vs `group_by`) |
| `df.write.parquet("s3://...")` | `df.write.save_as_table("silver.calls")` | Different destination, same concept |
| Runs on EMR/Dataproc cluster | Runs on Snowflake virtual warehouse | Different compute |

> **When to use Snowpark vs PySpark:** If your data is already in Snowflake and your output stays in Snowflake → Snowpark. If you're processing data across multiple systems (S3, databases, APIs) → PySpark on EMR/Dataproc.


In [ ]:
# ============================================================
# Demo: Snowpark DataFrame operations
# ============================================================

snowpark_demo = '''
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, avg, count, sum as sum_, when

connection_params = {
    "account": "your_account",
    "user": "your_user",
    "password": "your_password",  # Use env vars in production!
    "warehouse": "DE_ETL_WH",
    "database": "DE_TRAINING",
    "schema": "SILVER"
}
session = Session.builder.configs(connection_params).create()

calls_df = session.table("bronze.raw_calls")

silver_calls = (
    calls_df
    .select(
        col("raw_data")["call_id"].cast("string").alias("call_id"),
        col("raw_data")["dnis"].cast("string").alias("dnis"),
        col("raw_data")["duration"].cast("int").alias("duration"),
        col("raw_data")["disposition"].cast("string").alias("disposition"),
    )
    .filter(col("duration") > 0)
    .drop_duplicates(["call_id"])
)

campaign_stats = (
    silver_calls
    .group_by("dnis")
    .agg(
        count("call_id").alias("total_calls"),
        avg("duration").alias("avg_duration"),
        sum_(when(col("disposition") == "completed", 1).otherwise(0)).alias("conversions")
    )
    .sort(col("total_calls").desc())
)

campaign_stats.write.save_as_table("gold.mart_campaign_summary", mode="overwrite")
campaign_stats.show()
session.close()
'''
print(snowpark_demo)

### Snowflake vs Redshift vs BigQuery — When to Use Which

| Factor | Redshift | BigQuery | Snowflake |
|--------|----------|----------|-----------|
| **Provisioning** | Clusters (RA3 nodes) or Serverless mode | Fully serverless — always | Virtual warehouses (auto-suspend) |
| **Pricing** | Per-node-hour + storage | Per-TB-scanned (on-demand) or per-slot (flat-rate) | Per-credit (compute) + storage |
| **Best for** | AWS-heavy shops, predictable workloads | GCP-heavy shops, ad-hoc analytics, ML in SQL | Multi-cloud, data sharing, mixed workloads |
| **Scaling** | Resize cluster or add concurrency scaling | Auto-scales (add slots) | Spin up independent warehouses |
| **Semi-structured** | `SUPER` type (JSON) | Native STRUCT/ARRAY | `VARIANT` type (JSON, Avro, Parquet) |
| **Data sharing** | Limited (Redshift data sharing) | Analytics Hub | Native zero-copy data sharing (cross-account, cross-cloud) |
| **Streaming** | Kinesis → Redshift Streaming Ingestion | Pub/Sub → BigQuery Subscription | Snowpipe (near real-time) |
| **ML** | Export to SageMaker | BigQuery ML (CREATE MODEL in SQL) | Snowpark ML |
| **Lock-in** | AWS only | GCP only | Runs on AWS, GCP, Azure |


#### Snowflake: Edge Cases, Gotchas, and Interview Questions

**Gotchas:**

| Gotcha | What Happens | Fix |
|---|---|---|
| Warehouse left running | Credits burn even with no queries (if auto-suspend is off) | Always set AUTO_SUSPEND = 300 (5 min) |
| Used XL warehouse for a 10-row query | Paid 16 credits/hr for a 1-second query | Match warehouse size to workload. XS for dev. |
| COPY INTO without ON_ERROR | One bad row fails the entire load | Use `ON_ERROR = 'CONTINUE'` or `'SKIP_FILE'` |
| Snowpipe loads duplicate files | Same file loaded twice = duplicate data | Snowpipe tracks loaded files for 14 days. Don't rename and re-upload. |
| VARIANT column with no extraction | Queries on raw VARIANT are slow | Extract into typed columns in Silver layer |

**Interview questions:**

1. "How does Snowflake differ from Redshift?" — Snowflake: cloud-agnostic, true compute-storage separation (independent warehouses), zero-copy data sharing, VARIANT for semi-structured. Redshift: AWS-only, tighter S3 integration, Spectrum for lakehouse.
2. "What are virtual warehouses?" — Independent compute clusters that auto-suspend and auto-resume. ETL warehouse can run without affecting analyst warehouse. You pay per-second when running.
3. "How does Snowpipe work?" — S3 event → SQS → Snowpipe service (serverless) → COPY INTO table. Billed per-file, not per-warehouse-hour.
4. "What are Streams and Tasks?" — Stream = CDC capture (tracks changes on a table). Task = scheduled execution (cron inside Snowflake). Together they enable incremental pipelines without Airflow.

#### Console Walkthrough: Snowflake (Snowsight)

> **Open Snowsight:** Log in at `https://YOUR_ACCOUNT.snowflakecomputing.com`

**First Login Setup:**

1. Navigate to your Snowflake account URL
2. Log in with admin credentials
3. You land in **Snowsight** — Snowflake's web UI

**Create Database and Warehouse (Console):**

1. Left sidebar → **Data** → **+ Database** → name: `DE_TRAINING` → Create
2. Left sidebar → **Admin** → **Warehouses** → **+ Warehouse**
   - Name: `DE_ETL_WH`
   - Size: **X-Small** (cheapest — 1 credit/hr)
   - Auto-suspend: **5 minutes** (stop paying when idle)
   - Auto-resume: **Yes**
   - Click **Create Warehouse**

**SQL Worksheets (Query Editor):**

1. Left sidebar → **Worksheets** → **+** (new worksheet)
2. Select warehouse: `DE_ETL_WH`
3. Select database: `DE_TRAINING`
4. Write and run SQL:
   ```sql
   CREATE SCHEMA bronze;
   CREATE SCHEMA silver;
   CREATE SCHEMA gold;
   ```

**Key Snowsight Features:**

| Feature | Where | Why DEs Care |
|---|---|---|
| **Worksheets** | Left sidebar | SQL editor with auto-complete, results, charts |
| **Databases** | Left sidebar → **Data** | Browse schemas, tables, columns, preview data |
| **Query History** | Left sidebar → **Activity** → **Query History** | Every query: duration, warehouse, bytes, user |
| **Warehouses** | Admin → Warehouses | Monitor credit usage, resize, auto-suspend settings |
| **Task History** | Monitoring → **Task History** | See Snowflake Task runs, success/failure, duration |
| **Copy History** | Monitoring → **Copy History** | Track COPY INTO loads — files loaded, rows, errors |
| **Stages** | Data → your database → Stages | Browse external stages (S3/GCS), list files, preview |

**Upload a File Directly (Console):**

1. Go to **Data** → `DE_TRAINING` → `BRONZE` schema
2. Click **Create** → **Table from File**
3. Drag and drop `products.csv`
4. Snowflake auto-detects schema
5. Click **Load** — table created and populated

> **Teaching point:** Snowsight's "Create Table from File" is the fastest way to load reference data during development. For production, always use Stages + COPY INTO or Snowpipe.

---
## 5. Medallion Architecture — The Blueprint

> **In plain English:** Think of a water treatment plant. **Bronze** is the raw water intake — everything flows in, including dirt and debris. You never throw raw water away because you might need to reprocess it. **Silver** is the filtration stage — remove contaminants, standardize pH, test quality. **Gold** is the tap water — clean, safe, ready for consumers. A restaurant (analyst) doesn't need to know how the water was treated. They just turn on the tap.

You saw this in M00 and M01. Now we build it.

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│   BRONZE LAYER  │    │   SILVER LAYER  │    │    GOLD LAYER   │
│                 │    │                 │    │                 │
│ Raw data as-is  │──▶│ Cleaned &       │──▶│ Business-ready  │
│ Append-only     │    │ conformed       │    │ marts           │
│ Schema enforced │    │ Deduplicated    │    │ Pre-aggregated  │
│                 │    │ UTC → EST       │    │ Star schema     │
│ S3/GCS          │    │ Delta Lake      │    │ Redshift/BQ/SF  │
│ (Parquet/JSON)  │    │ MERGE/upsert    │    │                 │
└─────────────────┘    └─────────────────┘    └─────────────────┘
   calls.json            silver_calls           mart_daily_campaign
   orders.csv            silver_orders          mart_hourly_volume
   payments.xml          silver_customers       mart_conversion
                         (SCD Type 2)           mart_product_mix
```

### Why Three Layers?

| Principle | Without Medallion | With Medallion |
|-----------|------------------|----------------|
| **Raw preservation** | Transform corrupts original → can't reprocess | Bronze is append-only → always go back to raw |
| **Single responsibility** | One script does ingest + clean + aggregate | Each layer does one thing well |
| **Reprocessability** | Bug in transform? Re-run from... where? | Bug? Re-run Silver from Bronze. Gold from Silver. |
| **Data quality** | Bad data reaches dashboards | Quality gates between Silver and Gold |
| **Debugging** | "Where did this number come from?" — no idea | Trace: Gold mart → Silver table → Bronze file → source |


### Our Pipeline Blueprint

| Layer | Technology | Call Center Data | What Happens Here |
|-------|-----------|-----------------|-------------------|
| **Bronze** | S3/GCS + Delta Lake (append) | `raw_calls`, `raw_orders`, `raw_payments` | Land raw files, enforce schema, partition by date. Nothing is modified or deleted. |
| **Silver** | S3/GCS + Delta Lake (MERGE) | `silver_calls`, `silver_orders`, `silver_customers` | Deduplicate, flatten JSON, UTC→EST, join calls↔orders, SCD Type 2 on customers, quarantine bad records |
| **Gold** | Redshift / BigQuery / Snowflake | `mart_daily_campaign`, `mart_hourly_volume`, `mart_conversion`, `mart_product_mix` | Pre-aggregated, star schema, optimized for analyst queries |

**Now let's build it.** But first, we need to understand the table format that makes Bronze and Silver work: **Delta Lake**.


---
## 6. Delta Lake Deep Dive

### What Is Delta Lake?

> **In plain English:** Plain Parquet files on S3 are like paper documents in a filing cabinet. Anyone can shove papers in, rearrange them, or tear pages out — there's no log of who did what. If two people file documents at the same time, they might overwrite each other.
>
> Delta Lake adds a **sign-in sheet** (transaction log) to the filing cabinet. Every change is recorded: who, when, what was added or removed. You can see what the cabinet looked like last Tuesday (time travel). If someone tries to file a document that doesn't match the folder's format, it gets rejected (schema enforcement). And two people can file simultaneously without conflicts (ACID transactions).

Delta Lake is an **open-source storage layer** that adds reliability to data lakes. It sits on top of S3/GCS (Parquet files) and adds:

- **ACID transactions** — no more partial writes or corrupted tables
- **Schema enforcement** — reject writes that don't match the schema
- **Schema evolution** — safely add new columns without rewriting data
- **MERGE / Upsert** — update existing rows (impossible with plain Parquet)
- **Time travel** — query any previous version of the table
- **OPTIMIZE / VACUUM** — compact small files, clean up old versions

#### Delta Lake vs Plain Parquet

| Feature | Plain Parquet on S3 | Delta Lake on S3 |
|---------|--------------------|--------------------|
| ACID transactions | No — partial writes can corrupt the table | Yes — atomic commits via transaction log |
| Schema enforcement | No — write anything, break everything | Yes — reject mismatched schemas |
| Update a row | Impossible — Parquet is immutable | `MERGE INTO` — update, insert, or delete |
| Time travel | No — data is overwritten | Yes — query any version: `VERSION AS OF 5` |
| Small files | Accumulate forever | `OPTIMIZE` compacts them |
| Concurrent writes | Last writer wins (data loss) | Optimistic concurrency (conflict detection) |

> **Under the hood:** Delta Lake stores data as Parquet files + a `_delta_log/` directory that records every transaction. The log is the magic — it tracks which files are valid, which are obsolete, and what schema is expected.


#### Hello World: Delta Lake

Before schema enforcement, MERGE, and time travel — let's write and read one Delta table:


In [ ]:
# ============================================================
# Hello World: Delta Lake — write 3 rows, read them back
# ============================================================

hello_delta = """
# Create a simple DataFrame
data = [(1, 'Alice'), (2, 'Bob'), (3, 'Charlie')]
df = spark.createDataFrame(data, ['id', 'name'])

# Write as Delta (not Parquet — just change the format string)
df.write.format('delta').save('/tmp/hello_delta')

# Read it back
spark.read.format('delta').load('/tmp/hello_delta').show()

# +---+-------+
# | id|   name|
# +---+-------+
# |  1|  Alice|
# |  2|    Bob|
# |  3|Charlie|
# +---+-------+

# That's it. Same as Parquet, but now you get ACID, time travel, and MERGE.
# The magic is in the _delta_log/ folder that was created alongside your data.
"""
print(hello_delta)

In [ ]:
# ============================================================
# Demo: Create a Delta Lake table
# ============================================================

delta_create = '''
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

spark = (SparkSession.builder
    .appName("DE-Training-DeltaLake")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())

calls_schema = StructType([
    StructField("call_id", StringType(), False),
    StructField("dnis", StringType(), True),
    StructField("caller_phone", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("duration", IntegerType(), True),
    StructField("disposition", StringType(), True),
])

data = [
    ("C001", "8005551234", "5125559876", "2026-03-19 10:30:00", 120, "completed"),
    ("C002", "8005551234", "5125551111", "2026-03-19 10:45:00", 45, "transferred"),
    ("C003", "8005555678", None, "2026-03-19 11:00:00", 200, "completed"),
]
df = spark.createDataFrame(data, schema=calls_schema)

df.write.format("delta").mode("overwrite").save("s3://de-training-bronze/delta/calls")

print(f"Wrote {df.count()} records to Delta Lake table")
df.show()
'''
print(delta_create)

In [ ]:
# ============================================================
# Demo: Schema enforcement — reject bad writes
# ============================================================

schema_enforce = '''
bad_data = [("C004", "8005551234", "extra_field_that_shouldnt_exist")]
bad_df = spark.createDataFrame(bad_data, ["call_id", "dnis", "oops_wrong_column"])

try:
    bad_df.write.format("delta").mode("append").save("s3://de-training-bronze/delta/calls")
    print("Write succeeded (unexpected!)")
except Exception as e:
    print(f"Write rejected! Error: {e}")
    print("\nDelta Lake enforced the schema — bad data never entered the table.")
    print("This is why Bronze layer uses Delta Lake: schema is a contract.")
'''
print(schema_enforce)

In [ ]:
# ============================================================
# Demo: Schema evolution — safely add columns
# ============================================================

schema_evolution = '''
from pyspark.sql.functions import lit

new_data = [("C004", "8005551234", "5125552222", "2026-03-19 12:00:00", 90, "completed")]
new_df = spark.createDataFrame(new_data, schema=calls_schema)
new_df = new_df.withColumn("payment_status", lit("paid"))

new_df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("s3://de-training-bronze/delta/calls")

updated = spark.read.format("delta").load("s3://de-training-bronze/delta/calls")
updated.show()
# Old records have NULL for payment_status, new records have "paid"
'''
print(schema_evolution)

In [ ]:
# ============================================================
# Demo: MERGE / Upsert — the killer feature
# ============================================================

delta_merge = '''
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

target = DeltaTable.forPath(spark, "s3://de-training-silver/delta/calls")

updates = spark.createDataFrame([
    ("C002", "8005551234", "5125551111", "2026-03-19 10:45:00", 45, "completed"),
    ("C005", "8005555678", "5125553333", "2026-03-19 13:00:00", 180, "completed"),
], schema=calls_schema)

(target.alias("t")
    .merge(updates.alias("s"), "t.call_id = s.call_id")
    .whenMatchedUpdate(set={
        "disposition": "s.disposition",
        "payment_status": "s.payment_status"
    })
    .whenNotMatchedInsertAll()
    .execute())

target.toDF().orderBy("call_id").show()
'''
print(delta_merge)
print()
print("This is the core pattern for Silver layer updates:")
print("  - Call disposition changes after initial recording")
print("  - Payment confirmation arrives 10 minutes after call ends")
print("  - Customer address update for SCD Type 2")

#### GCP Note: Delta Lake MERGE on GCS

The MERGE code above works identically on GCS. Delta Lake's transaction log (`_delta_log/`) works the same on S3 and GCS — it's just Parquet files + JSON metadata.

```python
# GCS version — only the path changes
target = DeltaTable.forPath(spark, "gs://de-training-YOUR-NAME/silver/calls")
# ... rest of MERGE code is identical
```

In [ ]:
# ============================================================
# Demo: Time travel — query previous versions
# ============================================================

time_travel = '''
# Read the table as of version 0 (before any updates)
v0 = spark.read.format("delta").option("versionAsOf", 0) \
    .load("s3://de-training-silver/delta/calls")
print("Version 0 (original):")
v0.show()

current = spark.read.format("delta").load("s3://de-training-silver/delta/calls")
print("Current version:")
current.show()

yesterday = spark.read.format("delta") \
    .option("timestampAsOf", "2026-03-18 23:59:59") \
    .load("s3://de-training-silver/delta/calls")
print("As of yesterday:")
yesterday.show()

from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, "s3://de-training-silver/delta/calls")
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)
'''
print(time_travel)
print()
print("Use cases for time travel:")
print("  - Audit: 'What did yesterday\'s report show vs today?'")
print("  - Debug: 'When did this record change? What was the old value?'")
print("  - Recovery: 'Restore the table to before that bad deploy'")

In [ ]:
# ============================================================
# Demo: OPTIMIZE and VACUUM — table maintenance
# ============================================================

maintenance = '''
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, "s3://de-training-silver/delta/calls")

# OPTIMIZE: compact small files into larger ones
dt.optimize().executeCompaction()

# Z-ORDER: co-locate related data for faster queries
dt.optimize().executeZOrderBy("call_date", "dnis")

# VACUUM: delete old file versions (save storage)
dt.vacuum(retentionHours=168)  # 168 hours = 7 days

print("After OPTIMIZE: fewer, larger files = faster reads")
print("After VACUUM: old versions deleted = less storage cost")
print("WARNING: After VACUUM, time travel only works within retention period!")
'''
print(maintenance)

#### Console Walkthrough: Delta Lake (via AWS/GCP Consoles)

Delta Lake doesn't have its own console — it runs on Spark. But you can see your Delta tables through other consoles:

**View Delta Tables in Glue Catalog (AWS Console):**

1. Run a Glue Crawler on your `silver/` S3 prefix (same steps as Section 2.5)
2. Go to Glue Console → **Tables** → you'll see your Delta tables
3. Click a table → **Schema** shows columns, **Partitions** shows date partitions
4. Now Athena can query your Delta tables:
   ```sql
   -- In Athena Query Editor:
   SELECT * FROM de_training.silver_calls WHERE call_date = '2026-03-19' LIMIT 10;
   ```

**View Delta Transaction Log in S3 Console:**

1. Go to S3 Console → navigate to `silver/calls/`
2. You'll see:
   ```
   silver/calls/
   ├── _delta_log/                    ← Transaction log (the magic)
   │   ├── 00000000000000000000.json  ← Version 0 (initial write)
   │   ├── 00000000000000000001.json  ← Version 1 (first MERGE)
   │   └── 00000000000000000002.json  ← Version 2 (second MERGE)
   ├── call_date=2026-03-17/
   │   └── part-00000-...parquet      ← Actual data files
   ├── call_date=2026-03-18/
   │   └── part-00000-...parquet
   └── call_date=2026-03-19/
       └── part-00000-...parquet
   ```
3. Click a `_delta_log/*.json` file → **Download** → open it to see:
   - Which Parquet files were added/removed in this commit
   - Schema at that version
   - Operation (WRITE, MERGE, OPTIMIZE)

**View Delta Tables in Spark UI (EMR Console):**

1. EMR Console → your cluster → **Application UIs** → **Spark History Server**
2. Click your job → **SQL** tab → see the logical plan for your Delta operations
3. **Stages** tab → see how the MERGE was executed (scan + join + write)

> **Teaching point:** Delta Lake is invisible in the query console — Athena/Spectrum just see "Parquet files in S3." The Delta transaction log is what adds ACID, time travel, and MERGE. Click through the `_delta_log/` files in S3 to see what Delta is actually doing under the hood.

#### Delta Lake: Edge Cases, Gotchas, and Interview Questions

**Gotchas:**

| Gotcha | What Happens | Fix |
|---|---|---|
| MERGE with duplicate keys in source | Multiple matches per target row — error or nondeterministic | Deduplicate source before MERGE |
| Schema evolution changes column type (string → int) | Fails — Delta enforces type compatibility | Cast explicitly, or add new column + deprecate old |
| VACUUM with too-short retention | Deletes files needed for time travel or running queries | Never VACUUM < 7 days. Default is 168 hours. |
| Too many small files after streaming inserts | Slow reads | Run OPTIMIZE regularly (or set auto-optimize) |
| Concurrent MERGE on same partition | Conflict exception | Retry with backoff, or partition to reduce contention |

**Interview questions:**

1. "How does Delta Lake achieve ACID on S3?" — Transaction log (`_delta_log/`) records every change atomically. Optimistic concurrency detects conflicts.
2. "Delta Lake vs plain Parquet — why not just use Parquet?" — Parquet has no transactions, no MERGE, no schema enforcement, no time travel. Delta adds all of these.
3. "What is time travel and when would you use it?" — Query previous versions by number or timestamp. Use cases: audit, debug, restore after bad deploy.
4. "How do you handle late-arriving data?" — MERGE: match on business key, update if newer, insert if new.
5. "What is the small file problem and how do you fix it?" — Many small files = slow reads (too many S3 API calls). Fix: OPTIMIZE compacts them. Z-ORDER co-locates related data.

---
## 7. Apache Iceberg

> **You know Delta Lake. Here's how Iceberg differs.**

Iceberg is another open table format — same goals as Delta Lake (ACID, schema evolution, time travel) but different architecture and trade-offs.

### Delta Lake vs Iceberg — Side by Side

| Feature | Delta Lake | Iceberg |
|---------|-----------|---------|
| **Transaction log** | `_delta_log/` directory with JSON files | Metadata files + manifest lists in a catalog |
| **Partitioning** | Hive-style (`date=2026-03-19/`) — visible in file paths | **Hidden partitioning** — partition by any expression, not visible in paths |
| **Schema evolution** | Add/rename/drop columns (merge schema on write) | Add/rename/drop/reorder columns (broader support) |
| **Time travel** | Version number or timestamp | Snapshot ID or timestamp |
| **Engine support** | Best with Spark (created by Databricks) | Engine-agnostic (Spark, Trino, Flink, Dremio, Snowflake) |
| **Catalog** | Hive metastore or Unity Catalog | Pluggable: Hive, AWS Glue, REST, Nessie |
| **Community** | Databricks-driven | Apache Foundation, broad contributor base (Apple, Netflix, etc.) |
| **Industry momentum** | Dominant in Databricks ecosystem | Growing fast — adopted by Snowflake, AWS, Dremio, Netflix |

### Hidden Partitioning — The "Aha" Moment

In Delta Lake (Hive-style partitioning), you write:
```sql
-- You must know the partition column and format
SELECT * FROM calls WHERE date = '2026-03-19'
-- Under the hood: reads s3://bucket/calls/date=2026-03-19/
```

In Iceberg (hidden partitioning):
```sql
-- You query normally — Iceberg handles partition pruning automatically
SELECT * FROM calls WHERE start_time > '2026-03-19 00:00:00'
-- Iceberg knows start_time is partitioned by day
-- It prunes to the right files automatically — no partition column needed
```

**Why this matters:** Users don't need to know HOW data is partitioned. They write normal SQL, and Iceberg optimizes. You can even change the partitioning strategy without rewriting data.


#### Iceberg Demo Code

```python
from pyspark.sql import SparkSession

# Spark session with Iceberg support
spark = (SparkSession.builder
    .appName("DE-Training-Iceberg")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "hadoop")
    .config("spark.sql.catalog.iceberg.warehouse", "s3://de-training-iceberg/")
    .getOrCreate())

# Create table with hidden partitioning
spark.sql("""
    CREATE TABLE iceberg.bronze.calls (
        call_id STRING, dnis STRING, start_time TIMESTAMP,
        duration INT, disposition STRING
    ) USING iceberg
    PARTITIONED BY (days(start_time))  -- hidden partition by day!
""")

# Insert data
spark.sql("""
    INSERT INTO iceberg.bronze.calls VALUES
    ('C001', '8005551234', TIMESTAMP '2026-03-19 10:30:00', 120, 'completed'),
    ('C002', '8005551234', TIMESTAMP '2026-03-19 10:45:00', 45, 'transferred'),
    ('C003', '8005555678', TIMESTAMP '2026-03-20 09:00:00', 200, 'completed')
""")

# Query -- Iceberg auto-prunes partitions
result = spark.sql("""
    SELECT * FROM iceberg.bronze.calls
    WHERE start_time >= TIMESTAMP '2026-03-20 00:00:00'
""")
result.show()

# Snapshots (time travel)
spark.sql("SELECT * FROM iceberg.bronze.calls.snapshots").show(truncate=False)

# Schema evolution
spark.sql("ALTER TABLE iceberg.bronze.calls ADD COLUMN payment_status STRING")
```

> Run the above code in a PySpark session with Iceberg catalog configured (EMR or Dataproc with Iceberg JARs).


### Decision Framework: Delta Lake vs Iceberg

| Choose Delta Lake When | Choose Iceberg When |
|----------------------|---------------------|
| Using Databricks | Using Snowflake, Trino, Dremio, or multi-engine |
| Spark is your only engine | Multiple query engines need to access the same tables |
| You want Databricks Unity Catalog | You need hidden partitioning (cleaner user experience) |
| Ecosystem maturity matters most | Cross-platform compatibility matters most |

> **For this training:** We use **Delta Lake** as the primary format (more common in DE job postings, simpler to set up) and show Iceberg for comparison. In production, your company's existing ecosystem usually decides which one.


#### Console Walkthrough: Apache Iceberg (via AWS/GCP Consoles)

Like Delta Lake, Iceberg doesn't have a standalone console. You interact with Iceberg tables through query engines:

**View Iceberg Tables in Athena (AWS Console):**

1. Open Athena Console → Query Editor
2. If using Iceberg with Glue Catalog:
   ```sql
   -- Query Iceberg table (same SQL as regular tables)
   SELECT * FROM de_training.iceberg_calls 
   WHERE start_time >= TIMESTAMP '2026-03-19 00:00:00'
   LIMIT 10;

   -- View snapshots (Iceberg-specific)
   SELECT * FROM de_training."iceberg_calls$snapshots";

   -- View partition info
   SELECT * FROM de_training."iceberg_calls$partitions";
   ```

**View Iceberg Metadata in S3 Console:**

1. Navigate to your Iceberg table location in S3:
   ```
   s3://de-training-iceberg/bronze/calls/
   ├── data/                          ← Parquet data files
   │   ├── 00000-0-...parquet
   │   └── 00001-0-...parquet
   └── metadata/                      ← Iceberg metadata (like Delta's _delta_log/)
       ├── v1.metadata.json           ← Snapshot 1
       ├── v2.metadata.json           ← Snapshot 2
       ├── snap-...avro               ← Manifest lists
       └── ...avro                    ← Manifest files
   ```
2. Download `v2.metadata.json` → see current schema, partitioning, snapshot history

**View Iceberg in Snowflake Console (Snowsight):**

1. Snowflake supports Iceberg tables natively:
   ```sql
   -- Create Iceberg table backed by external S3 storage
   CREATE ICEBERG TABLE bronze.calls_iceberg
     CATALOG = 'SNOWFLAKE'
     EXTERNAL_VOLUME = 'de_s3_volume'
     BASE_LOCATION = 'bronze/calls/';
   
   -- Query normally — Snowflake reads Iceberg metadata
   SELECT * FROM bronze.calls_iceberg LIMIT 10;
   ```
2. In Snowsight → Data → the table appears like any other table

> **Teaching point:** Iceberg's "hidden partitioning" means you never see partition folders in S3 (unlike Delta Lake's `date=2026-03-19/` Hive-style partitions). The partition scheme lives in the metadata files. This is cleaner but means you can't "browse partitions" in S3 — you have to query the `$partitions` metadata table.

---
## 8. Build: Bronze Layer

> **Now we build.** This section is hands-on. You will ingest the call center dataset into a Bronze layer on GCS with Delta Lake tables, using Dataproc for processing.

### What Goes in Bronze

| Rule | Why |
|------|-----|
| **Raw data as-is** — no transforms, no cleaning | You can always reprocess from raw |
| **Append-only** — never update or delete | History is preserved forever |
| **Schema enforced** — reject data that doesn't match | Catch bad data at the door |
| **Partitioned by date** — for efficient querying | Only scan the dates you need |

> **Platform note:** This build uses GCS + Dataproc + BigQuery (GCP). AWS equivalent commands (S3 + EMR + Redshift) are shown in sidebars.

In [ ]:
# ============================================================
# Lab: Set up the Bronze layer on GCS
# Run these in Cloud Shell
# ============================================================

gcs_setup = """
# Create the bucket (if not already created)
gcloud storage buckets create gs://de-training-YOUR-NAME --location=us-central1

# Upload the synthetic dataset
gcloud storage cp data/calls.json gs://de-training-YOUR-NAME/incoming/calls/date=2026-03-19/
gcloud storage cp data/orders.csv gs://de-training-YOUR-NAME/incoming/orders/date=2026-03-19/
gcloud storage cp data/payments.xml gs://de-training-YOUR-NAME/incoming/payments/date=2026-03-19/
gcloud storage cp data/products.csv gs://de-training-YOUR-NAME/reference/products/
gcloud storage cp data/campaigns.csv gs://de-training-YOUR-NAME/reference/campaigns/

# Verify
gcloud storage ls -r gs://de-training-YOUR-NAME/
"""
print(gcs_setup)

# You should see: all 5 files uploaded with partition paths
# Verify in GCS Console: console.cloud.google.com/storage -> your bucket

#### AWS Path: Bronze Setup on S3

> **If your cohort uses AWS, follow these steps instead of the GCS steps above. Same pipeline, same data, different cloud.**

In [ ]:
# ============================================================
# Lab (AWS): Set up Bronze layer on S3
# Run these in your terminal with AWS CLI configured
# ============================================================

aws_bronze_setup = """
# Step 1: Create S3 bucket
aws s3 mb s3://de-training-YOUR-NAME --region us-east-1

# Step 2: Upload the synthetic dataset
aws s3 cp data/calls.json s3://de-training-YOUR-NAME/incoming/calls/date=2026-03-19/
aws s3 cp data/orders.csv s3://de-training-YOUR-NAME/incoming/orders/date=2026-03-19/
aws s3 cp data/payments.xml s3://de-training-YOUR-NAME/incoming/payments/date=2026-03-19/
aws s3 cp data/products.csv s3://de-training-YOUR-NAME/reference/products/
aws s3 cp data/campaigns.csv s3://de-training-YOUR-NAME/reference/campaigns/

# Step 3: Verify
aws s3 ls s3://de-training-YOUR-NAME/ --recursive --summarize

# You should see: 5 files, total size ~225 KB
"""
print(aws_bronze_setup)

In [ ]:
# ============================================================
# Lab (AWS): Create EMR cluster for pipeline processing
# ============================================================

emr_setup = """
# Step 1: Create an EMR cluster with Spark + Delta Lake
aws emr create-cluster \\
    --name "DE-Training-Pipeline" \\
    --release-label emr-7.0.0 \\
    --applications Name=Spark Name=Hive \\
    --instance-groups '[
        {"InstanceGroupType":"MASTER","InstanceType":"m5.xlarge","InstanceCount":1},
        {"InstanceGroupType":"CORE","InstanceType":"m5.xlarge","InstanceCount":2},
        {"InstanceGroupType":"TASK","InstanceType":"m5.xlarge","InstanceCount":2,
         "Market":"SPOT","BidPrice":"OnDemandPrice"}
    ]' \\
    --service-role EMR_DefaultRole \\
    --ec2-attributes InstanceProfile=EMR_EC2_DefaultRole \\
    --auto-termination-policy '{"IdleTimeout": 3600}' \\
    --region us-east-1

# Save the cluster ID from the output: j-XXXXXXXXXXXXX
# Check status:
aws emr describe-cluster --cluster-id j-XXXXXXXXXXXXX \\
    --query 'Cluster.Status.State'

# Wait for status: WAITING (cluster is ready)
# Takes 5-15 minutes
"""
print(emr_setup)
print()
print('COST ALERT: This cluster costs ~$0.75/hr.')
print('Auto-terminates after 1 hour idle.')
print('Always verify it shut down: aws emr list-clusters --active')

In [ ]:
# ============================================================
# Lab (AWS): Submit Bronze ingest to EMR
# ============================================================

emr_bronze = """
# Step 1: Upload the PySpark script to S3
aws s3 cp bronze_ingest.py s3://de-training-YOUR-NAME/code/

# In bronze_ingest.py, change the BUCKET line:
# BUCKET = "s3://de-training-YOUR-NAME"   # <-- S3 path
# (rest of code is identical to GCP version)

# Step 2: Submit as an EMR step
aws emr add-steps \\
    --cluster-id j-XXXXXXXXXXXXX \\
    --steps '[{
        "Type": "Spark",
        "Name": "Bronze Ingest",
        "ActionOnFailure": "CONTINUE",
        "Args": [
            "spark-submit",
            "--deploy-mode", "cluster",
            "--packages", "io.delta:delta-spark_2.12:3.1.0",
            "s3://de-training-YOUR-NAME/code/bronze_ingest.py"
        ]
    }]'

# Step 3: Monitor the step
aws emr list-steps --cluster-id j-XXXXXXXXXXXXX \\
    --query 'Steps[*].[Name,Status.State]' --output table

# Wait for: COMPLETED

# Step 4: Verify output in S3
aws s3 ls s3://de-training-YOUR-NAME/bronze/calls/ --recursive | head -10
# Should see: _delta_log/ directory + .parquet files in date= partitions
"""
print(emr_bronze)

In [ ]:
# ============================================================
# Lab: Create Bronze Delta Lake tables on GCS via Dataproc
# ============================================================

bronze_ingest = '''
# File: bronze_ingest.py
# Run on Dataproc with Delta Lake JARs

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name, lit

spark = (SparkSession.builder
    .appName("Bronze-Ingestion")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())

BUCKET = "gs://de-training-YOUR-NAME"   # <-- GCS path
# AWS: BUCKET = "s3://de-training-YOUR-NAME"

# --- Bronze: Calls ---
raw_calls = (spark.read
    .json(f"{BUCKET}/incoming/calls/")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
    .withColumn("_batch_id", lit("batch-2026-03-19-001"))
)

(raw_calls.write
    .format("delta")
    .mode("append")
    .partitionBy("date")
    .save(f"{BUCKET}/bronze/calls"))

print(f"Bronze calls: {raw_calls.count()} records ingested")

# --- Bronze: Orders ---
raw_orders = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BUCKET}/incoming/orders/")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

(raw_orders.write
    .format("delta")
    .mode("append")
    .partitionBy("date")
    .save(f"{BUCKET}/bronze/orders"))

# --- Bronze: Reference data ---
products = spark.read.option("header", "true").csv(f"{BUCKET}/reference/products/")
products.write.format("delta").mode("overwrite").save(f"{BUCKET}/bronze/products")

campaigns = spark.read.option("header", "true").csv(f"{BUCKET}/reference/campaigns/")
campaigns.write.format("delta").mode("overwrite").save(f"{BUCKET}/bronze/campaigns")

print("Bronze layer complete!")
print(f"  calls:     {BUCKET}/bronze/calls")
print(f"  orders:    {BUCKET}/bronze/orders")
print(f"  products:  {BUCKET}/bronze/products")
print(f"  campaigns: {BUCKET}/bronze/campaigns")
'''
print(bronze_ingest)

In [ ]:
# ============================================================
# Lab: Submit Bronze ingest to Dataproc
# ============================================================

submit_bronze = """
# Step 1: Upload the script to GCS
gcloud storage cp bronze_ingest.py gs://de-training-YOUR-NAME/code/

# Step 2: Create Dataproc cluster (if not already running)
gcloud dataproc clusters create de-training-pipeline \\
    --region=us-central1 \\
    --master-machine-type=n1-standard-4 \\
    --num-workers=2 \\
    --worker-machine-type=n1-standard-4 \\
    --max-idle=1h \\
    --optional-components=JUPYTER

# Step 3: Submit the job
gcloud dataproc jobs submit pyspark \\
    gs://de-training-YOUR-NAME/code/bronze_ingest.py \\
    --cluster=de-training-pipeline \\
    --region=us-central1 \\
    --properties=spark.jars.packages=io.delta:delta-spark_2.12:3.1.0

# Step 4: Check job status
gcloud dataproc jobs list --region=us-central1 --cluster=de-training-pipeline

# Step 5: Verify output in GCS
gcloud storage ls gs://de-training-YOUR-NAME/bronze/calls/
gcloud storage ls gs://de-training-YOUR-NAME/bronze/orders/
"""
print(submit_bronze)

# AWS Equivalent:
# aws emr add-steps --cluster-id j-XXX --steps '[{"Type":"Spark","Name":"Bronze Ingest",
#   "Args":["spark-submit","s3://bucket/code/bronze_ingest.py"]}]'

# You should see: Job completed successfully
# GCS should have: bronze/calls/_delta_log/, bronze/calls/date=2026-03-19/*.parquet

In [ ]:
# ============================================================
# Lab: Verify Bronze — query and inspect
# ============================================================

verify_bronze = '''
# Run this on Dataproc (submit as another PySpark job or use Jupyter on the cluster)

BUCKET = "gs://de-training-YOUR-NAME"

calls = spark.read.format("delta").load(f"{BUCKET}/bronze/calls")
orders = spark.read.format("delta").load(f"{BUCKET}/bronze/orders")

print("=== Bronze Calls ===")
print(f"Row count: {calls.count()}")
print(f"Partitions: {calls.select('date').distinct().count()} dates")
calls.printSchema()
calls.show(5, truncate=False)

print("\n=== Bronze Orders ===")
print(f"Row count: {orders.count()}")
orders.show(5, truncate=False)

# Check data quality issues (we will fix these in Silver)
from pyspark.sql.functions import col, isnull

null_phones = calls.filter(isnull("caller_phone")).count()
print(f"\nNull phone numbers: {null_phones} (~5% expected)")

dup_calls = calls.groupBy("call_id").count().filter(col("count") > 1).count()
print(f"Duplicate call_ids: {dup_calls} (~10 expected)")

print("\nTimestamps are UTC -- conversion to EST happens in Silver")
calls.select("start_time").show(5)
'''
print(verify_bronze)

# EXPECTED OUTPUT:
# Bronze calls: ~510 rows (500 + 10 duplicates)
# Null phones: ~25 (4.9%)
# Duplicate call_ids: ~10
# Partitions: 3 dates (2026-03-17, 18, 19)

#### Exercise: Explain Like I'm CEO (Bronze Layer)

**Pause. Close the notebook. Answer these out loud (2 minutes):**

Your VP asks: *"What did you build today and why should I care?"*

> "We set up a system where every call record, order, and payment automatically lands in cloud storage the moment it's generated. Nothing is modified or deleted — it's a complete, tamper-proof record. Think of it as the company's filing cabinet in the cloud. Right now it's raw data. Next step: we clean it up so you can actually query it."

**If you can't explain it without jargon, you don't understand it yet.**

---
## 9. Build: Silver Layer

> **Silver is where the data engineering happens.** Deduplication, cleaning, type conversion, joins, SCD Type 2, and MERGE — all in this layer.

### Silver Layer Responsibilities

| Responsibility | What We Do | Why |
|---------------|-----------|-----|
| **Deduplicate** | Remove ~2% duplicate call_ids | Same call recorded twice (system bug) |
| **Flatten JSON** | Extract nested customer fields to flat columns | Nested JSON is hard to query at scale |
| **UTC to EST** | Convert all timestamps to Eastern Time | Reports are in EST; source data is UTC |
| **Null handling** | Handle ~5% null phone numbers | Decide: keep as null, fill with UNKNOWN, or quarantine |
| **MERGE** | Handle late-arriving data (disposition changes, payment updates) | Call records update after initial creation |
| **Dead letter queue** | Quarantine ~1% orphaned orders | Orders with no matching call_id — investigate, don't discard |

In [ ]:
# ============================================================
# Lab: Silver transform — deduplicate, clean, flatten
# File: silver_transform.py (submit to Dataproc)
# ============================================================

silver_transform = '''
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, from_utc_timestamp, when, lit, row_number,
    current_timestamp
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("Silver-Transform").getOrCreate()

BUCKET = "gs://de-training-YOUR-NAME"   # <-- GCS
# AWS: BUCKET = "s3://de-training-YOUR-NAME"

# --- Read Bronze ---
bronze_calls = spark.read.format("delta").load(f"{BUCKET}/bronze/calls")
bronze_orders = spark.read.format("delta").load(f"{BUCKET}/bronze/orders")

# --- Step 1: Deduplicate ---
window = Window.partitionBy("call_id").orderBy(col("_ingested_at").desc())
deduped = (bronze_calls
    .withColumn("_row_num", row_number().over(window))
    .filter(col("_row_num") == 1)
    .drop("_row_num"))

before = bronze_calls.count()
after = deduped.count()
print(f"Deduplication: {before} -> {after} ({before - after} duplicates removed)")

# --- Step 2: Flatten nested customer JSON ---
flat_calls = (deduped
    .withColumn("customer_first_name", col("customer.first_name"))
    .withColumn("customer_last_name", col("customer.last_name"))
    .withColumn("customer_phone", col("caller_phone"))
    .withColumn("customer_city", col("customer.address.city"))
    .withColumn("customer_state", col("customer.address.state"))
    .withColumn("customer_zip", col("customer.address.zip"))
    .drop("customer"))

# --- Step 3: UTC to EST ---
silver_calls = (flat_calls
    .withColumn("start_time_utc", col("start_time"))
    .withColumn("start_time_est", from_utc_timestamp(col("start_time"), "America/New_York"))
    .withColumn("end_time_est", from_utc_timestamp(col("end_time"), "America/New_York"))
    .withColumn("call_date", col("start_time_est").cast("date")))

# --- Step 4: Handle null phones ---
silver_calls = silver_calls.withColumn(
    "customer_phone",
    when(col("customer_phone").isNull(), lit("UNKNOWN"))
    .otherwise(col("customer_phone")))

# --- Write Silver ---
silver_calls.write.format("delta").mode("overwrite") \\
    .partitionBy("call_date") \\
    .save(f"{BUCKET}/silver/calls")

print(f"Silver calls written: {silver_calls.count()} rows")
silver_calls.printSchema()
'''
print(silver_transform)

# EXPECTED OUTPUT:
# Deduplication: 510 -> 500 (10 duplicates removed)
# Silver calls written: 500 rows

In [ ]:
# ============================================================
# Lab: Submit Silver transform to Dataproc
# ============================================================

submit_silver = """
# Upload script
gcloud storage cp silver_transform.py gs://de-training-YOUR-NAME/code/

# Submit to Dataproc
gcloud dataproc jobs submit pyspark \\
    gs://de-training-YOUR-NAME/code/silver_transform.py \\
    --cluster=de-training-pipeline \\
    --region=us-central1 \\
    --properties=spark.jars.packages=io.delta:delta-spark_2.12:3.1.0

# Verify output
gcloud storage ls gs://de-training-YOUR-NAME/silver/calls/
# Should see: _delta_log/ and call_date=2026-03-17/, 18/, 19/ partitions
"""
print(submit_silver)

# AWS Equivalent:
# aws emr add-steps --cluster-id j-XXX --steps '[{"Type":"Spark",
#   "Args":["spark-submit","s3://bucket/code/silver_transform.py"]}]'

In [ ]:
# ============================================================
# Lab: Silver — Dead letter queue (quarantine orphaned orders)
# ============================================================

dead_letter = '''
BUCKET = "gs://de-training-YOUR-NAME"

silver_orders = spark.read.format("delta").load(f"{BUCKET}/bronze/orders")
calls_df = spark.read.format("delta").load(f"{BUCKET}/silver/calls")

# Find orphaned orders (no matching call)
orphaned = silver_orders.join(calls_df, "call_id", "left_anti")
orphan_count = orphaned.count()
total_orders = silver_orders.count()
print(f"Orphaned orders: {orphan_count} / {total_orders}")

# Write orphans to quarantine
orphaned.write.format("delta").mode("append") \\
    .save(f"{BUCKET}/quarantine/orphaned_orders")

# Write valid orders to Silver
valid_orders = silver_orders.join(calls_df.select("call_id"), "call_id", "inner")
valid_orders.write.format("delta").mode("overwrite") \\
    .save(f"{BUCKET}/silver/orders")

print(f"Valid orders: {valid_orders.count()}")
print(f"Quarantined: {orphan_count}")
'''
print(dead_letter)

#### Why Quarantine Instead of Delete?

| Possibility | What Happened | Right Action |
|---|---|---|
| **Timing issue** | Order arrived before its call was logged | Wait and retry |
| **ID mismatch** | Order system uses different call_id format | Fix join logic, reprocess |
| **Manual order** | Placed outside the call system | Create "no-call" category |
| **Bad data** | Source system bug | Log it, alert source team |

> **Rule:** Never silently drop data. Quarantine it, log it, alert on it.

#### Exercise: Explain Like I'm CEO (Silver Layer)

**Pause. Answer out loud (2 minutes):**

Your VP asks: *"The raw data is in the cloud. Now what?"*

> "We built a cleaning station. Duplicates are removed, timestamps are converted to our timezone, customer records are linked to their orders, and anything suspicious gets flagged for review. The data that comes out of this stage is trustworthy. You can query it and the numbers will be right."

---
## 10. Build: Gold Layer

> **Gold = business-ready.** Pre-aggregated, star schema, optimized for analyst queries. On GCP, the Gold layer lives in **BigQuery**.

### Gold Layer Marts

| Mart | What It Answers | Grain |
|------|----------------|-------|
| `mart_daily_campaign` | How did each campaign perform today? | One row per campaign per day |
| `mart_hourly_volume` | When do calls peak? | One row per hour of day |
| `mart_conversion` | What's our call-to-order funnel? | One row per campaign per day |

In [ ]:
# ============================================================
# Lab: Load Silver into BigQuery, then build Gold marts
# ============================================================

load_to_bq = """
# Step 1: Load Silver Parquet from GCS into BigQuery
bq load --source_format=PARQUET --replace \\
    silver.calls \\
    'gs://de-training-YOUR-NAME/silver/calls/*.parquet'

bq load --source_format=PARQUET --replace \\
    silver.orders \\
    'gs://de-training-YOUR-NAME/silver/orders/*.parquet'

# Load reference tables
bq load --source_format=PARQUET --replace \\
    bronze.campaigns \\
    'gs://de-training-YOUR-NAME/bronze/campaigns/*.parquet'

# Verify
bq ls silver
bq show silver.calls
"""
print(load_to_bq)

# You should see: silver.calls table with ~500 rows

In [ ]:
# ============================================================
# Lab: Build Gold marts in BigQuery SQL
# Run these in BigQuery Console or bq CLI
# ============================================================

gold_marts = """
-- mart_daily_campaign
CREATE OR REPLACE TABLE gold.mart_daily_campaign AS
SELECT
    c.call_date,
    camp.campaign_name,
    c.dnis,
    COUNT(c.call_id) AS total_calls,
    COUNTIF(c.disposition = 'completed') AS conversions,
    AVG(c.duration) AS avg_duration_sec,
    SUM(CAST(c.amount AS FLOAT64)) AS total_revenue,
    ROUND(SAFE_DIVIDE(COUNTIF(c.disposition = 'completed'), COUNT(c.call_id)) * 100, 2) AS conversion_rate
FROM silver.calls c
LEFT JOIN bronze.campaigns camp ON c.dnis = camp.dnis
GROUP BY 1, 2, 3
ORDER BY call_date, total_calls DESC;

-- mart_hourly_volume
CREATE OR REPLACE TABLE gold.mart_hourly_volume AS
SELECT
    call_date,
    EXTRACT(HOUR FROM start_time_est) AS call_hour,
    COUNT(call_id) AS total_calls,
    AVG(duration) AS avg_duration_sec,
    COUNTIF(disposition = 'completed') AS conversions
FROM silver.calls
GROUP BY 1, 2
ORDER BY call_date, call_hour;

-- mart_conversion (funnel)
CREATE OR REPLACE TABLE gold.mart_conversion AS
SELECT
    c.call_date,
    camp.campaign_name,
    COUNT(c.call_id) AS total_calls,
    COUNTIF(c.duration > 30) AS qualified_calls,
    COUNTIF(c.disposition = 'completed') AS conversions,
    SUM(CAST(c.amount AS FLOAT64)) AS revenue,
    ROUND(SAFE_DIVIDE(COUNTIF(c.duration > 30), COUNT(c.call_id)) * 100, 2) AS qualification_rate,
    ROUND(SAFE_DIVIDE(COUNTIF(c.disposition = 'completed'), NULLIF(COUNTIF(c.duration > 30), 0)) * 100, 2) AS conversion_rate
FROM silver.calls c
LEFT JOIN bronze.campaigns camp ON c.dnis = camp.dnis
GROUP BY 1, 2;

-- Verify
SELECT * FROM gold.mart_daily_campaign ORDER BY call_date, total_calls DESC LIMIT 20;
SELECT * FROM gold.mart_hourly_volume ORDER BY call_date, call_hour LIMIT 24;
SELECT * FROM gold.mart_conversion ORDER BY revenue DESC LIMIT 10;
"""
print(gold_marts)

# EXPECTED OUTPUT:
# mart_daily_campaign: ~30 rows (10 campaigns x 3 days)
# mart_hourly_volume: ~36 rows (12 hours x 3 days)
# mart_conversion: ~30 rows
# Check in BigQuery Console -> Preview tab (free, no query cost)

#### AWS Path: Gold Layer on Redshift

> **If your cohort uses AWS, load Silver data into Redshift and build Gold marts there.**

In [ ]:
# ============================================================
# Lab (AWS): Load Silver into Redshift and build Gold marts
# ============================================================

redshift_gold = """
-- Run these in Redshift Query Editor v2
-- (Console: console.aws.amazon.com/sqlworkbench)

-- Step 1: Create schemas
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

-- Step 2: Load Silver Parquet from S3 into Redshift
-- (Make sure your Redshift cluster has the DE-Pipeline IAM role attached)
COPY silver.calls
FROM 's3://de-training-YOUR-NAME/silver/calls/'
IAM_ROLE 'arn:aws:iam::ACCOUNT_ID:role/DE-Pipeline-Redshift-Role'
FORMAT AS PARQUET;

COPY silver.orders
FROM 's3://de-training-YOUR-NAME/silver/orders/'
IAM_ROLE 'arn:aws:iam::ACCOUNT_ID:role/DE-Pipeline-Redshift-Role'
FORMAT AS PARQUET;

-- Load reference data
COPY bronze.campaigns
FROM 's3://de-training-YOUR-NAME/bronze/campaigns/'
IAM_ROLE 'arn:aws:iam::ACCOUNT_ID:role/DE-Pipeline-Redshift-Role'
FORMAT AS PARQUET;

-- Verify
SELECT COUNT(*) FROM silver.calls;   -- ~500 rows
SELECT COUNT(*) FROM silver.orders;   -- ~75 rows

-- Step 3: Build Gold marts (same logic as BigQuery, minor syntax diffs)

-- mart_daily_campaign
CREATE TABLE gold.mart_daily_campaign AS
SELECT
    c.call_date,
    camp.campaign_name,
    c.dnis,
    COUNT(c.call_id) AS total_calls,
    SUM(CASE WHEN c.disposition = 'completed' THEN 1 ELSE 0 END) AS conversions,
    AVG(c.duration) AS avg_duration_sec,
    SUM(c.amount::DECIMAL(10,2)) AS total_revenue,
    ROUND(SUM(CASE WHEN c.disposition = 'completed' THEN 1.0 ELSE 0 END) /
          NULLIF(COUNT(c.call_id), 0) * 100, 2) AS conversion_rate
FROM silver.calls c
LEFT JOIN bronze.campaigns camp ON c.dnis = camp.dnis
GROUP BY 1, 2, 3
ORDER BY call_date, total_calls DESC;

-- mart_hourly_volume
CREATE TABLE gold.mart_hourly_volume AS
SELECT
    call_date,
    EXTRACT(HOUR FROM start_time_est) AS call_hour,
    COUNT(call_id) AS total_calls,
    AVG(duration) AS avg_duration_sec,
    SUM(CASE WHEN disposition = 'completed' THEN 1 ELSE 0 END) AS conversions
FROM silver.calls
GROUP BY 1, 2
ORDER BY call_date, call_hour;

-- mart_conversion
CREATE TABLE gold.mart_conversion AS
SELECT
    c.call_date,
    camp.campaign_name,
    COUNT(c.call_id) AS total_calls,
    SUM(CASE WHEN c.duration > 30 THEN 1 ELSE 0 END) AS qualified_calls,
    SUM(CASE WHEN c.disposition = 'completed' THEN 1 ELSE 0 END) AS conversions,
    SUM(c.amount::DECIMAL(10,2)) AS revenue
FROM silver.calls c
LEFT JOIN bronze.campaigns camp ON c.dnis = camp.dnis
GROUP BY 1, 2;

-- Verify
SELECT * FROM gold.mart_daily_campaign ORDER BY call_date, total_calls DESC LIMIT 20;
"""
print(redshift_gold)
print()
print('Key Redshift vs BigQuery syntax differences:')
print('  BigQuery: COUNTIF(condition)      Redshift: SUM(CASE WHEN ... THEN 1 ELSE 0 END)')
print('  BigQuery: SAFE_DIVIDE(a, b)       Redshift: a / NULLIF(b, 0)')
print('  BigQuery: CAST(x AS FLOAT64)      Redshift: x::DECIMAL(10,2)')
print('  BigQuery: CREATE OR REPLACE TABLE Redshift: DROP TABLE IF EXISTS + CREATE TABLE')

In [ ]:
# ============================================================
# Lab (AWS): Submit Silver transform to EMR
# ============================================================

emr_silver = """
# Upload Silver transform script
aws s3 cp silver_transform.py s3://de-training-YOUR-NAME/code/

# In silver_transform.py, change:
# BUCKET = "s3://de-training-YOUR-NAME"   # <-- S3 path

# Submit to EMR
aws emr add-steps \\
    --cluster-id j-XXXXXXXXXXXXX \\
    --steps '[{
        "Type": "Spark",
        "Name": "Silver Transform",
        "ActionOnFailure": "CONTINUE",
        "Args": [
            "spark-submit",
            "--deploy-mode", "cluster",
            "--packages", "io.delta:delta-spark_2.12:3.1.0",
            "s3://de-training-YOUR-NAME/code/silver_transform.py"
        ]
    }]'

# Monitor
aws emr list-steps --cluster-id j-XXXXXXXXXXXXX \\
    --query 'Steps[*].[Name,Status.State]' --output table

# Verify Silver output
aws s3 ls s3://de-training-YOUR-NAME/silver/calls/ --recursive | head -10
"""
print(emr_silver)
print()
print('Complete AWS pipeline flow:')
print('  S3 (Bronze) --> EMR PySpark (Silver) --> Redshift COPY + SQL (Gold)')
print('  Same data, same logic, same results as GCP path.')

#### AWS Path: Verify the Full Pipeline

```
calls.json ----\
orders.csv -----+-> S3 (Bronze) -> EMR PySpark (Silver) -> Redshift (Gold)
payments.xml --/    s3://bucket/   s3://bucket/silver/     gold.mart_*
                    bronze/         Delta Lake tables       Ready for dashboards
```

**Verify in 3 consoles:**

1. **S3 Console** → `bronze/calls/date=2026-03-19/` has raw JSON + `_delta_log/`
2. **EMR Console** → Steps tab shows completed Bronze + Silver jobs
3. **Redshift Query Editor v2** → `gold` schema has marts
   - Run: `SELECT * FROM gold.mart_daily_campaign LIMIT 10;`

#### Console Walkthrough: Verify the Full GCP Pipeline

**Your complete pipeline on GCP:**

```
calls.json ----\                                             
orders.csv -----+-> GCS (Bronze) -> Dataproc PySpark (Silver) -> BigQuery (Gold)
payments.xml --/    gs://bucket/     gs://bucket/silver/         gold.mart_*
                    bronze/           Delta Lake tables           Ready for dashboards
```

**Verify in 3 consoles:**

1. **GCS Console** → `bronze/calls/date=2026-03-19/` has raw JSON + `_delta_log/`
2. **Dataproc Console** → Jobs tab shows completed Bronze + Silver jobs
3. **BigQuery Console** → `gold` dataset has `mart_daily_campaign`, `mart_hourly_volume`, `mart_conversion`
   - Click any mart → **Preview** (free) → see the data
   - Click **Details** → see row count

---
## 10a. See Your Pipeline's Output — Visualization

> **This is the payoff.** You built Bronze, Silver, Gold. Now see what the VP sees. These charts are what your entire pipeline exists for.

We'll query the Gold marts and visualize them directly in this notebook using matplotlib. In production, these charts would live in Looker Studio, Tableau, or Metabase — but the data powering them comes from YOUR pipeline.

In [ ]:
# ============================================================
# Setup: Install and import visualization libraries
# ============================================================

# WHY: matplotlib is Python's standard charting library.
# Every DE should know basic visualization to verify pipeline output.
# You're not building dashboards — that's the analyst's job.
# You're VERIFYING that your pipeline produces correct, useful data.

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
import csv
from collections import defaultdict, Counter
from datetime import datetime

# Set clean chart style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('Visualization libraries loaded.')
print('Now let\'s see what your pipeline produces.')

In [ ]:
# ============================================================
# Load the synthetic dataset (simulating Gold mart queries)
# In production, you'd query BigQuery/Redshift directly.
# Here we read from the data/ files and simulate the Gold output.
# ============================================================

# WHY: We read the raw files and apply the same transforms
# (dedup, join, aggregate) that the pipeline does, so the
# charts show what the Gold marts would contain.

# Load calls
calls = []
with open('data/calls.json') as f:
    for line in f:
        calls.append(json.loads(line))

# Load orders
orders = []
with open('data/orders.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        orders.append(row)

# Load campaigns
campaigns = {}
with open('data/campaigns.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        campaigns[row['dnis']] = row['campaign_name']

# Deduplicate calls (same as Silver layer)
seen_ids = set()
unique_calls = []
for call in calls:
    if call['call_id'] not in seen_ids:
        seen_ids.add(call['call_id'])
        unique_calls.append(call)

# Build order lookup
order_by_call = {}
for order in orders:
    order_by_call[order['call_id']] = order

print(f'Loaded: {len(unique_calls)} calls (deduped), {len(orders)} orders, {len(campaigns)} campaigns')
print('Simulating Gold mart aggregations...')

### Chart 1: Campaign Performance

> **What the VP asks:** "Which campaign is generating the most calls and converting the best?"

In [ ]:
# ============================================================
# Chart 1: Campaign Performance — Total Calls & Conversions
# This is what mart_daily_campaign powers in the dashboard.
# ============================================================

# Aggregate: count calls and conversions per campaign
# WHY: This mirrors the Gold mart SQL:
#   SELECT campaign_name, COUNT(*) as total_calls,
#          COUNTIF(disposition='completed') as conversions
#   FROM silver.calls JOIN campaigns USING(dnis)
#   GROUP BY campaign_name

campaign_stats = defaultdict(lambda: {'calls': 0, 'conversions': 0, 'revenue': 0.0})

for call in unique_calls:
    camp_name = campaigns.get(call['dnis'], 'Unknown')
    campaign_stats[camp_name]['calls'] += 1
    if call['disposition'] == 'completed':
        campaign_stats[camp_name]['conversions'] += 1
        # Check if there's an order for this call
        if call['call_id'] in order_by_call:
            try:
                campaign_stats[camp_name]['revenue'] += float(order_by_call[call['call_id']]['total'])
            except (ValueError, KeyError):
                pass

# Sort by total calls descending
sorted_campaigns = sorted(campaign_stats.items(), key=lambda x: x[1]['calls'], reverse=True)
names = [c[0] for c in sorted_campaigns]
total_calls = [c[1]['calls'] for c in sorted_campaigns]
conversions = [c[1]['conversions'] for c in sorted_campaigns]

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(names))
bars1 = ax.bar(x, total_calls, width=0.4, label='Total Calls', color='#4285F4', align='center')
bars2 = ax.bar([i + 0.4 for i in x], conversions, width=0.4, label='Conversions', color='#34A853', align='center')

ax.set_xlabel('Campaign')
ax.set_ylabel('Count')
ax.set_title('Campaign Performance: Total Calls vs Conversions', fontsize=14, fontweight='bold')
ax.set_xticks([i + 0.2 for i in x])
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
ax.legend()

# Add conversion rate labels on bars
for i, (tc, cv) in enumerate(zip(total_calls, conversions)):
    rate = cv / tc * 100 if tc > 0 else 0
    ax.text(i + 0.2, max(tc, cv) + 1, f'{rate:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# You should see: Bar chart with 10 campaigns.
# Each has a blue bar (total calls) and green bar (conversions).
# Conversion rate % shown above each pair.
# This is the FIRST thing the VP looks at every morning.

### Chart 2: Hourly Call Volume

> **What the VP asks:** "When should we staff more agents?"

In [ ]:
# ============================================================
# Chart 2: Hourly Call Volume — When Do Calls Peak?
# This is what mart_hourly_volume powers.
# ============================================================

# WHY: Staffing decisions depend on knowing peak hours.
# Too few agents at 2pm = dropped calls = lost revenue.
# Too many agents at 6am = wasted payroll.

# Parse hours from UTC timestamps, convert to EST (UTC - 5)
hourly_counts = defaultdict(lambda: defaultdict(int))

for call in unique_calls:
    # Parse UTC time and convert to EST
    utc_time = datetime.strptime(call['start_time'], '%Y-%m-%dT%H:%M:%SZ')
    est_hour = (utc_time.hour - 5) % 24  # Simple EST conversion
    date = call['date']
    hourly_counts[date][est_hour] += 1

# Plot hourly volume for each day
fig, ax = plt.subplots(figsize=(14, 5))

colors = ['#4285F4', '#EA4335', '#FBBC04']  # Google colors
dates = sorted(hourly_counts.keys())

for idx, date in enumerate(dates):
    hours = sorted(hourly_counts[date].keys())
    counts = [hourly_counts[date][h] for h in hours]
    ax.plot(hours, counts, marker='o', linewidth=2, color=colors[idx % 3],
            label=date, markersize=6)

ax.set_xlabel('Hour of Day (EST)')
ax.set_ylabel('Number of Calls')
ax.set_title('Hourly Call Volume by Day (EST)', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h}:00' for h in range(24)], rotation=45, fontsize=8)
ax.legend(title='Date')
ax.axvspan(10, 14, alpha=0.1, color='red', label='Peak hours')  # Highlight peak

plt.tight_layout()
plt.show()

# You should see: Line chart with 3 lines (one per day).
# Peak hours are typically 10am-2pm EST.
# The VP uses this to decide: 'Staff 3 extra agents 10am-2pm.'

### Chart 3: Call-to-Order Conversion Funnel

> **What the VP asks:** "How many calls actually turn into revenue?"

In [ ]:
# ============================================================
# Chart 3: Conversion Funnel — Calls → Qualified → Converted → Paid
# This is what mart_conversion powers.
# ============================================================

# WHY: The funnel shows where prospects drop off.
# If conversion rate is low, is it because calls are too short?
# Or because completed calls don't result in orders?

total = len(unique_calls)
qualified = sum(1 for c in unique_calls if c['duration'] > 30)  # >30 sec = qualified
completed = sum(1 for c in unique_calls if c['disposition'] == 'completed')
with_order = sum(1 for c in unique_calls if c['call_id'] in order_by_call)
paid = sum(1 for c in unique_calls 
           if c['call_id'] in order_by_call 
           and order_by_call[c['call_id']].get('payment_status') == 'completed')

stages = ['All Calls', 'Qualified\n(>30 sec)', 'Completed\n(disposition)', 'Has Order', 'Paid']
values = [total, qualified, completed, with_order, paid]
colors = ['#4285F4', '#5B9BD5', '#34A853', '#FBBC04', '#EA4335']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(stages, values, color=colors, width=0.6, edgecolor='white', linewidth=2)

# Add value labels and drop-off percentages
for i, (bar, val) in enumerate(zip(bars, values)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val}', ha='center', fontsize=13, fontweight='bold')
    if i > 0:
        drop = (1 - val/values[i-1]) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                f'-{drop:.0f}%', ha='center', fontsize=10, color='white', fontweight='bold')

ax.set_ylabel('Count')
ax.set_title('Call-to-Payment Conversion Funnel', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(values) * 1.15)

# Overall conversion rate
overall_rate = paid / total * 100 if total > 0 else 0
ax.text(0.98, 0.95, f'Overall: {paid}/{total} = {overall_rate:.1f}%',
        transform=ax.transAxes, ha='right', va='top', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

# You should see: Funnel chart narrowing from ~500 calls to ~50 paid orders.
# Drop-off percentages show where prospects are lost.
# The VP uses this to ask: 'Why are 25% of completed calls not resulting in orders?'

### Chart 4: Daily Revenue Trend

> **What the VP asks:** "Are we trending up or down?"

In [ ]:
# ============================================================
# Chart 4: Daily Revenue Trend
# ============================================================

# WHY: Revenue trend is the ultimate business metric.
# If it's flat or declining, something in the pipeline
# (campaigns, agents, products) needs attention.

daily_revenue = defaultdict(float)
daily_calls = defaultdict(int)
daily_conversions = defaultdict(int)

for call in unique_calls:
    date = call['date']
    daily_calls[date] += 1
    if call['disposition'] == 'completed':
        daily_conversions[date] += 1
        if call['call_id'] in order_by_call:
            try:
                daily_revenue[date] += float(order_by_call[call['call_id']]['total'])
            except (ValueError, KeyError):
                pass

dates = sorted(daily_revenue.keys())
revenues = [daily_revenue[d] for d in dates]
call_counts = [daily_calls[d] for d in dates]

fig, ax1 = plt.subplots(figsize=(12, 5))

# Revenue bars
bars = ax1.bar(dates, revenues, color='#34A853', alpha=0.7, label='Revenue ($)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Revenue ($)', color='#34A853')
ax1.tick_params(axis='y', labelcolor='#34A853')

# Add revenue labels on bars
for bar, rev in zip(bars, revenues):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'${rev:,.0f}', ha='center', fontsize=11, fontweight='bold', color='#34A853')

# Call count line on secondary axis
ax2 = ax1.twinx()
ax2.plot(dates, call_counts, color='#4285F4', marker='s', linewidth=2,
         markersize=8, label='Total Calls')
ax2.set_ylabel('Total Calls', color='#4285F4')
ax2.tick_params(axis='y', labelcolor='#4285F4')

ax1.set_title('Daily Revenue & Call Volume', fontsize=14, fontweight='bold')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

# Print summary (what the Gold mart query would return)
print('\nDaily Summary (from mart_daily_campaign):')
print(f'{"Date":<15} {"Calls":>8} {"Conversions":>13} {"Revenue":>10} {"Conv Rate":>10}')
print('-' * 58)
for d in dates:
    conv_rate = daily_conversions[d] / daily_calls[d] * 100 if daily_calls[d] > 0 else 0
    print(f'{d:<15} {daily_calls[d]:>8} {daily_conversions[d]:>13} {"$" + str(int(daily_revenue[d])):>10} {conv_rate:>9.1f}%')

# You should see: Bar chart (revenue) + line (call count) for 3 days.
# Plus a text summary table matching what BigQuery would return.

### Chart 5: Call Disposition Breakdown

> **What the VP asks:** "How many calls are actually completing vs dropping?"

In [ ]:
# ============================================================
# Chart 5: Call Disposition Breakdown
# ============================================================

# WHY: If 'dropped' calls are increasing, there's a system
# or staffing problem. If 'transferred' is high, the VA
# might not be handling calls well enough.

dispositions = Counter(c['disposition'] for c in unique_calls)

labels = list(dispositions.keys())
sizes = list(dispositions.values())
colors = ['#34A853', '#4285F4', '#EA4335', '#FBBC04', '#9AA0A6']
explode = [0.05 if l == 'completed' else 0 for l in labels]  # Highlight completed

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
wedges, texts, autotexts = ax1.pie(sizes, explode=explode, labels=labels,
    colors=colors[:len(labels)], autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 10})
ax1.set_title('Call Disposition Distribution', fontsize=13, fontweight='bold')

# Horizontal bar chart (same data, different view)
sorted_disp = sorted(dispositions.items(), key=lambda x: x[1], reverse=True)
ax2.barh([d[0] for d in sorted_disp], [d[1] for d in sorted_disp],
         color=colors[:len(sorted_disp)], edgecolor='white')
for i, (name, count) in enumerate(sorted_disp):
    ax2.text(count + 2, i, f'{count} ({count/len(unique_calls)*100:.1f}%)',
            va='center', fontsize=10)
ax2.set_xlabel('Number of Calls')
ax2.set_title('Calls by Disposition', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# You should see: Pie chart + bar chart showing the same data.
# 'completed' should be ~35%, 'transferred' ~25%, etc.
# If 'dropped' is high, that's a red flag for the VP.

### This Is Why You Built the Pipeline

Every chart above is powered by the data that flows through your pipeline:

```
calls.json  -->  Bronze (raw)  -->  Silver (clean)  -->  Gold (marts)  -->  Charts above
                  GCS               Dataproc              BigQuery          Looker Studio
                  Delta Lake        dedup, UTC->EST        SQL aggregation   or matplotlib
```

**Without your pipeline:**
- VP asks "revenue by campaign?" → developer writes SQL for 2 days → numbers might be wrong

**With your pipeline:**
- VP opens dashboard → sees these 5 charts → makes staffing/budget decisions in minutes

**In production, these charts live in Looker Studio** (see Section 3.12 for setup). The matplotlib versions here prove the concept. The pipeline is the same either way.

> **Data engineering is invisible when it works.** The VP never thinks about Bronze layers or Delta Lake MERGE. They just see charts that are accurate, fresh, and automatic. That's your job done right.

### Pipeline Checkpoint

**You just built a complete medallion pipeline on GCP:**

- Schema enforcement at Bronze (Delta Lake) ✓
- Deduplication (~10 duplicates removed) ✓
- UTC → EST timezone conversion ✓
- Null handling (phone numbers) ✓
- Nested JSON flattened ✓
- Dead letter queue for orphaned orders ✓
- Pre-aggregated Gold marts in BigQuery ✓

### Business Value Delivered

| Before (manual) | After (your pipeline) | Business Impact |
|---|---|---|
| VP asks "revenue by campaign?" → 2 days of manual SQL | `SELECT * FROM gold.mart_daily_campaign` → 2 seconds | Decisions happen same-day |
| Duplicate calls inflate numbers by ~2% | Deduplication removes them automatically | Accurate forecasting |
| Timezone bug: 7pm EST calls show as next day | UTC→EST conversion in Silver | Correct hourly staffing |
| Orphaned orders silently break revenue totals | Quarantined and flagged | Finance trusts the numbers |

> **What's missing:** This pipeline runs manually. In M09, you'll orchestrate it with Cloud Composer (Airflow on GCP) so it runs automatically every day.

#### Exercise: Explain Like I'm CEO (Gold Layer)

Your VP asks: *"Can I see the numbers now?"*

> "Yes. We have three pre-built reports: daily campaign performance, hourly call volume, and conversion funnel. They update automatically. Open the dashboard and the numbers are there."

**Notice the journey:** Bronze = storage. Silver = trust. Gold = decisions.

### Exercise: Break the System

| Scenario | What Breaks? | How Would You Fix It? |
|---|---|---|
| GCS bucket deleted | Everything gone | Enable versioning + cross-region replication |
| 10x more data tomorrow | Dataproc might OOM, costs spike | Auto-scaling on Dataproc, billing alerts |
| Source adds new field | Bronze accepts it, Silver MERGE might fail | `mergeSchema=true`, test in dev first |
| Pipeline fails at 3 AM | Gold marts stale, VP sees old numbers | Cloud Composer alerting (M09) |
| Silver transform runs twice | Without MERGE: duplicates. With MERGE: safe. | Always MERGE or OVERWRITE |
| Customer says "my order missing" | It's in quarantine | Investigate, reprocess, link manually |

---
## 11. Cost Optimization

Cloud pipelines can get expensive fast. Here's how to keep costs under control.

### Storage Costs

| Strategy | Service | Savings |
|----------|---------|---------|
| **Use Parquet over CSV/JSON** | S3/GCS | 60-80% less storage (columnar compression) |
| **S3 lifecycle policies** | S3 | Move old Bronze data to Glacier after 30 days |
| **Delta Lake VACUUM** | S3 | Delete old file versions after 7 days |
| **Partition pruning** | Athena, Spark | Scan only the dates you need → less data read |

### Compute Costs

| Strategy | Service | Savings |
|----------|---------|---------|
| **Spot/preemptible instances** | EMR/Dataproc task nodes | 70-90% off on-demand pricing |
| **Auto-terminate clusters** | EMR/Dataproc | Don't leave clusters running overnight |
| **Right-size warehouses** | Snowflake | Use XS for dev, M for ETL, L only for heavy analytics |
| **Auto-suspend** | Snowflake, Redshift Serverless | Stop paying when no queries running |
| **Reserved instances** | Redshift | Up to 75% off with 1-year commitment |
| **EMR Serverless** | EMR | Pay per vCPU-second, not per cluster-hour |

### Query Costs

| Strategy | Service | Savings |
|----------|---------|---------|
| **Partition pruning** | BigQuery, Athena | Only scan dates you need → less TB billed |
| **Clustering** | BigQuery | Data sorted → less scanned for filtered queries |
| **Column pruning** | BigQuery, Athena | `SELECT col1, col2` not `SELECT *` |
| **Materialized views** | Redshift, BigQuery, Snowflake | Pre-compute expensive aggregations |

### Cost Estimation for Our Pipeline

| Component | Estimated Monthly Cost | Notes |
|-----------|----------------------:|-------|
| S3 storage (Bronze+Silver+Gold, ~100GB) | ~$2.30 | Standard tier |
| EMR (Silver transform, 2x m5.xlarge, 1 hr/day) | ~$50 | With spot instances for task nodes |
| Redshift Serverless (Gold queries) | ~$20-50 | Depends on query volume |
| BigQuery (Gold queries, ~10TB/month scanned) | ~$50 | On-demand pricing |
| Snowflake (XS warehouse, ~2 hrs/day) | ~$60 | $2/credit × ~30 credits |
| **Total** | **~$180-210/month** | For a training/dev pipeline |

> **Production costs** scale with data volume and query frequency. A real VA analytics pipeline processing 100K+ calls/day might cost $500-2,000/month depending on architecture choices.


---
## Appendix A: AWS CLI Cheat Sheet

```bash
# Identity
aws sts get-caller-identity                    # Who am I?
aws iam list-roles                              # List IAM roles

# S3
aws s3 ls                                       # List buckets
aws s3 ls s3://bucket/prefix/ --recursive       # List files
aws s3 mb s3://bucket-name                      # Create bucket
aws s3 cp local_file s3://bucket/key            # Upload
aws s3 cp s3://bucket/key local_file            # Download
aws s3 sync local_dir s3://bucket/prefix/       # Sync directory
aws s3 rm s3://bucket/key                       # Delete file
aws s3 rm s3://bucket/prefix/ --recursive       # Delete prefix

# EMR
aws emr create-cluster ...                      # Create cluster
aws emr add-steps --cluster-id j-XXX ...        # Submit job
aws emr describe-cluster --cluster-id j-XXX     # Check status
aws emr list-steps --cluster-id j-XXX           # List job steps
aws emr terminate-clusters --cluster-ids j-XXX  # Shut down

# Glue
aws glue create-crawler ...                     # Create crawler
aws glue start-crawler --name NAME              # Run crawler
aws glue get-tables --database-name DB          # List tables

# Redshift (via psql or Redshift Query Editor)
# Connect: psql -h cluster.region.redshift.amazonaws.com -p 5439 -U admin -d dev
```


---
## Appendix B: Glossary

| Term | Definition |
|------|-----------|
| **ARN** | Amazon Resource Name — unique identifier for any AWS resource |
| **ACID** | Atomicity, Consistency, Isolation, Durability — transaction guarantees |
| **CDC** | Change Data Capture — tracking inserts/updates/deletes from a source system |
| **Columnar storage** | Data stored by column (Parquet, Redshift) — efficient for analytical queries |
| **Compute-storage separation** | Storage and processing are independent and scale separately |
| **Data lake** | Central repository for raw data in any format (S3/GCS) |
| **Data warehouse** | Structured, optimized store for analytical queries (Redshift/BQ/Snowflake) |
| **Dead letter queue** | Where failed/invalid records go for investigation |
| **Delta Lake** | Open-source storage layer adding ACID transactions to data lakes |
| **Distribution key** | Redshift: determines how rows are spread across nodes |
| **External table** | Query data in S3 without loading it into the warehouse |
| **Hidden partitioning** | Iceberg: partition by any expression without exposing partition columns |
| **IAM** | Identity and Access Management — who can do what to which resources |
| **Iceberg** | Apache open table format — engine-agnostic alternative to Delta Lake |
| **Lakehouse** | Architecture combining data lake (cheap storage) with warehouse (query performance) |
| **Medallion architecture** | Bronze → Silver → Gold data layers |
| **MERGE** | SQL operation that inserts new rows and updates existing ones in one statement |
| **Partition pruning** | Skipping irrelevant data partitions during queries (saves time and cost) |
| **SCD Type 2** | Slowly Changing Dimension — track history with effective/end dates |
| **Snowpark** | Snowflake's DataFrame API (like PySpark but runs on Snowflake compute) |
| **Snowpipe** | Snowflake's continuous auto-loading from S3/GCS |
| **Sort key** | Redshift: determines data order on disk for faster range queries |
| **Spectrum** | Redshift feature to query S3 data without loading |
| **Spot instance** | AWS compute at 70-90% discount (can be terminated with 2 min notice) |
| **Stage (Snowflake)** | Named location (internal or S3/GCS) for loading/unloading data |
| **Stream (Snowflake)** | CDC capture — tracks all changes on a table since last consumed |
| **Task (Snowflake)** | Scheduled SQL or Snowpark execution within Snowflake |
| **Time travel** | Query previous versions of a Delta Lake or Iceberg table |
| **Transaction log** | Delta Lake's `_delta_log/` — records every change for ACID guarantees |
| **Trust policy** | IAM: defines WHO can assume a role |
| **VARIANT** | Snowflake data type for semi-structured data (JSON, Avro, Parquet) |
| **Virtual warehouse** | Snowflake compute cluster — independent, auto-suspend, scalable |
| **VACUUM** | Delta Lake: delete old file versions to save storage |
| **Z-ORDER** | Delta Lake: co-locate related data in files for faster filtered queries |
